In [ ]:
import json
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, precision_recall_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.colors as mc
from pathlib import Path
from collections import defaultdict

# Apply consistent styling
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 18
})

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Where this notebook writes its rendered figures and CSV tables
RESULTS_DIR = Path("../results")
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR  = RESULTS_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

OVERALL_RESULTS_PATH      = FIGURES_DIR  # kept for naming compatibility with downstream cells
CALIBRATION_FIGURES_PATH  = FIGURES_DIR / "calibration_diagrams"
CALIBRATION_FIGURES_PATH.mkdir(parents=True, exist_ok=True)

# Per-record confidence scores are loaded from the PhysioNet release
CONFIDENCE_SCORES_DIR = Path("../data/confidence_scores")
PREDICTIONS_DIR       = Path("../data/predictions")
SOURCE_DIR            = Path("../data/source")

llm_ids = [
    "Qwen2.5-0.5B-Instruct",
    "Llama-3.2-1B-Instruct",
    "Qwen2.5-1.5B-Instruct",
    "Llama-3.2-3B-Instruct",
    "Qwen2.5-3B-Instruct",
]

LLM_DISPLAY_NAMES = {
    "Qwen2.5-0.5B-Instruct": "Qwen-0.5B",
    "Llama-3.2-1B-Instruct": "Llama-1B",
    "Qwen2.5-1.5B-Instruct": "Qwen-1.5B",
    "Llama-3.2-3B-Instruct": "Llama-3B",
    "Qwen2.5-3B-Instruct": "Qwen-3B",
}

# method2path keeps the same key structure as the original notebook so the
# rest of the cells (figure rendering, table assembly) do not need to change.
# The two changed fields are `subdir` (where scores live under
# data/confidence_scores/{model}/{subdir}/) and `score_filename_tmpl`.
method2path = {
    "PTRUE":    {"subdir": "ptrue",     "method_name": "P(True)",     "base_color": "#E74C3C", "shape": "d", "filling": ""},
    "PIK":      {"subdir": "pik",       "method_name": "P(IK)",       "base_color": "#F39C12", "shape": "o", "filling": ""},
    "SAPLMA-M": {"subdir": "saplma_m",  "method_name": "SAPLMA-M",    "base_color": "#3498DB", "shape": "s", "filling": ""},
    "SAPLMA-UM":{"subdir": "saplma_um", "method_name": "SAPLMA-UM",   "base_color": "#3498DB", "shape": "^", "filling": ""},
    "SAPLMA-F": {"subdir": "saplma_f",  "method_name": "SAPLMA-F",    "base_color": "#3498DB", "shape": "v", "filling": ""},
    "CCPS":     {"subdir": "ccps",      "method_name": "CCPS (Ours)", "base_color": "#27AE60", "shape": "o", "filling": ""},
}

dataset_types = ["contextual", "synthesis", "clinical_inference"]
dataset_display_names = {
    "contextual":         "CORAL-MCQA-Contextual",
    "synthesis":          "CORAL-MCQA-Synthesis",
    "clinical_inference": "CORAL-MCQA-Clinical-Inference",
}

print(f"LLMs            : {len(llm_ids)}")
print(f"Methods         : {list(method2path.keys())}")
print(f"Dataset types   : {dataset_types}")
print(f"Reading scores  : {CONFIDENCE_SCORES_DIR}")


## Setup: Load Data and Compute Metrics

The following cells load the results data, compute metrics, and create the necessary data structures for the COMBO section.

In [ ]:
# NOTE: The following code loads results and creates df_metrics, coverage data, and aggregated curves.
# This code should match what's in results.ipynb before the COMBO section.
# If these variables already exist from running results.ipynb, you can skip this cell.

# For now, we'll assume these variables need to be loaded/computed.
# You may need to run the corresponding cells from results.ipynb to create:
# - df_metrics
# - llm2method2dataset2results  
# - llm2dataset2sharedRIDacrossMethods
# - aggregated_curves_contextual, aggregated_ece_curves_contextual
# - aggregated_curves (synthesis), aggregated_ece_curves (synthesis)
# - aggregated_curves_clinical, aggregated_ece_curves_clinical
# - risk_coverage_data_contextual, risk_coverage_data (synthesis), risk_coverage_data_clinical
# - ece_coverage_data_contextual, ece_coverage_data (synthesis), ece_coverage_data_clinical
# - method_styles

# Method styles definition
method_styles = {
    'CCPS': {'color': method2path['CCPS']['base_color'], 'linestyle': '-', 'linewidth': 2.5, 'label': method2path['CCPS']['method_name']},
    'PIK': {'color': method2path['PIK']['base_color'], 'linestyle': '--', 'linewidth': 2, 'label': method2path['PIK']['method_name']},
    'PTRUE': {'color': method2path['PTRUE']['base_color'], 'linestyle': '-.', 'linewidth': 2, 'label': method2path['PTRUE']['method_name']},
    'SAPLMA-M': {'color': method2path['SAPLMA-M']['base_color'], 'linestyle': '-', 'linewidth': 2, 'label': method2path['SAPLMA-M']['method_name']},
    'SAPLMA-UM': {'color': method2path['SAPLMA-UM']['base_color'], 'linestyle': '--', 'linewidth': 2, 'label': method2path['SAPLMA-UM']['method_name']},
    'SAPLMA-F': {'color': method2path['SAPLMA-F']['base_color'], 'linestyle': '-.', 'linewidth': 2, 'label': method2path['SAPLMA-F']['method_name']}
}

print("Method styles configured:")
for method_name, style in method_styles.items():
    print(f"  {method_name}: {style['label']} - {style['color']}")

In [ ]:
# Weighted Cost Calculation Functions
def calculate_weighted_cost_at_optimal_coverage(results, confidence_scores, risk_labels_dir):
    """
    Calculate weighted cost at optimal coverage level.
    
    Similar to calculate_weighted_loss_curve but returns only the cost at optimal coverage.
    
    Args:
        results: List of dicts with 'ground_truth_correctness', 'predicted_answer_key', 'question_id'
        confidence_scores: List of confidence scores (same order as results)
        risk_labels_dir: Directory containing risk label JSON files
    
    Returns:
        cost: Cost at optimal coverage level (normalized per sample)
    """
    import numpy as np
    from pathlib import Path
    
    HUMAN_REVIEW_COST = 5
    
    def load_risk_labels(risk_labels_dir, question_id):
        """Load risk labels for a given question ID."""
        risk_file = Path(risk_labels_dir) / f"{question_id}.json"
        if not risk_file.exists():
            return None
        try:
            with open(risk_file, 'r') as f:
                return json.load(f)
        except:
            return None
    
    # Combine results and confidence scores
    data = list(zip(results, confidence_scores))
    
    # Sort by confidence score (lowest to highest)
    # Lower confidence = send to human first
    data_sorted = sorted(data, key=lambda x: x[1])
    
    total_samples = len(data_sorted)
    if total_samples == 0:
        return 0.0
    
    system_losses = []
    
    # Simulate different human workload levels (0% to 100% in 1% increments)
    for human_workload_pct in range(0, 101):
        num_human_reviews = int(total_samples * human_workload_pct / 100)
        
        # Human review cost
        human_cost = num_human_reviews * HUMAN_REVIEW_COST
        
        # AI cost (for retained samples)
        ai_cost = 0
        for i in range(num_human_reviews, total_samples):
            result = data_sorted[i][0]
            
            # If AI is correct, no cost
            if result['ground_truth_correctness'] == 1:
                ai_cost += 0
            else:
                # AI is incorrect - look up risk score for the predicted answer
                question_id = result.get('question_id', result.get('sidx', ''))
                predicted_key = result.get('predicted_answer_key', '')
                
                # Load risk labels
                risk_labels = load_risk_labels(risk_labels_dir, question_id)
                
                if risk_labels and predicted_key:
                    risk_key = f"{predicted_key}_Risk"
                    risk_score = risk_labels.get(risk_key, 5)  # Default to medium risk if not found
                    if risk_score is not None:
                        ai_cost += risk_score
                    else:
                        # If no risk labels found, default to medium risk
                        ai_cost += 5
                else:
                    # If no risk labels found, default to medium risk
                    ai_cost += 5
        
        # Total system loss
        total_loss = human_cost + ai_cost
        system_losses.append(total_loss)
    
    # Find optimal coverage (minimum loss)
    optimal_loss = min(system_losses)
    
    # Normalize by number of samples to get cost per sample
    cost_per_sample = optimal_loss / total_samples if total_samples > 0 else 0.0
    
    return cost_per_sample

def calculate_weighted_cost_at_fixed_threshold(results, confidence_scores, risk_labels_dir, threshold=0.5, verbose=False):
    """
    Calculate weighted cost at a fixed confidence threshold.
    Samples with confidence < threshold are sent for human review.
    
    Args:
        results: List of result dicts with 'ground_truth_correctness', 'question_id', 'predicted_answer_key'
        confidence_scores: List of confidence scores (same order as results)
        risk_labels_dir: Directory containing risk label JSON files
        threshold: Confidence threshold (samples below this go to human review)
        verbose: If True, log warnings for missing risk files
    
    Returns:
        cost_per_sample: Normalized cost per sample
    """
    import numpy as np
    from pathlib import Path
    
    HUMAN_REVIEW_COST = 5
    DEFAULT_AI_ERROR_COST = 5  # Default cost when risk labels are missing
    
    missing_risk_files = []  # Track missing risk files for safety check
    
    def load_risk_labels(risk_labels_dir, question_id):
        """Load risk labels for a given question ID with safety check."""
        risk_file = Path(risk_labels_dir) / f"{question_id}.json"
        if not risk_file.exists():
            missing_risk_files.append(str(risk_file))
            return None
        try:
            with open(risk_file, 'r') as f:
                return json.load(f)
        except (json.JSONDecodeError, IOError) as e:
            if verbose:
                print(f"Warning: Failed to load risk file {risk_file}: {e}")
            return None
    
    # Combine results and confidence scores
    data = list(zip(results, confidence_scores))
    
    total_samples = len(data)
    if total_samples == 0:
        return 0.0
    
    human_cost = 0
    ai_cost = 0
    ai_handled_count = 0
    ai_error_count = 0
    
    for result, conf_score in data:
        if conf_score < threshold:
            # Send to human review
            human_cost += HUMAN_REVIEW_COST
        else:
            # AI handles it
            ai_handled_count += 1
            if result['ground_truth_correctness'] == 1:
                ai_cost += 0
            else:
                # AI is incorrect - look up risk score
                ai_error_count += 1
                question_id = result.get('question_id', result.get('sidx', ''))
                predicted_key = result.get('predicted_answer_key', '')
                
                risk_labels = load_risk_labels(risk_labels_dir, question_id)
                
                if risk_labels and predicted_key:
                    risk_key = f"{predicted_key}_Risk"
                    risk_score = risk_labels.get(risk_key, DEFAULT_AI_ERROR_COST)
                    ai_cost += risk_score if risk_score is not None else DEFAULT_AI_ERROR_COST
                else:
                    ai_cost += DEFAULT_AI_ERROR_COST
    
    # Safety check: warn if significant number of risk files are missing
    if missing_risk_files and verbose:
        unique_missing = set(missing_risk_files)
        print(f"Warning: {len(unique_missing)} unique risk files not found. "
              f"Using default cost={DEFAULT_AI_ERROR_COST} for {len(missing_risk_files)} lookups.")
    
    total_cost = human_cost + ai_cost
    cost_per_sample = total_cost / total_samples if total_samples > 0 else 0.0
    
    return cost_per_sample

# Map dataset types to risk label directories
RISK_LABELS_DIR_MAP = {
    "contextual": Path("../generations/risk_labels_v2/gpt-5.1"),
    "synthesis": Path("../generations/risk_labels_mhr/gpt-5.1"),
    "clinical_inference": Path("../generations/risk_labels_l3/gpt-5.1")
}

In [ ]:
# Load every (LLM, method, tier) JSON of per-record confidence scores from
# data/confidence_scores/{model}/{method-subdir}/CORTEX_{tier}.json.

def load_method_results(llm_id, method_name, method_info, dataset_type):
    path = CONFIDENCE_SCORES_DIR / llm_id / method_info["subdir"] / f"CORTEX_{dataset_type}.json"
    if not path.exists():
        print(f"  ! missing {path}")
        return None
    with path.open() as f:
        return json.load(f)


llm2method2dataset2results = {}
for llm_id in llm_ids:
    llm2method2dataset2results[llm_id] = {}
    print(f"\nLoading scores for {llm_id}")
    for method_name, method_info in method2path.items():
        llm2method2dataset2results[llm_id][method_name] = {}
        for dataset_type in dataset_types:
            results = load_method_results(llm_id, method_name, method_info, dataset_type)
            if results is not None:
                llm2method2dataset2results[llm_id][method_name][dataset_type] = results
                print(f"  {method_name:9s} {dataset_type:20s}: {len(results)}")

# Backwards-compatible shortcut used by older cells
llm2method2results = {
    llm_id: {
        m: data["contextual"]
        for m, data in mtree.items() if "contextual" in data
    }
    for llm_id, mtree in llm2method2dataset2results.items()
}


In [ ]:
import sys as _sys; from pathlib import Path as _Path
_sys.path.insert(0, str(_Path("..").resolve()))
from utils.data_io import read_table

# ==============================================================================
# PATCH: Add question_id and predicted_answer_key to results
# ==============================================================================
# This cell patches the loaded results to include:
# 1. question_id (sidx from dataset CSV) - needed to load risk labels
# 2. predicted_answer_key (A/B/C/D) - needed to look up specific risk scores
#
# We need to:
# - Map record_id (row index) to sidx from dataset CSV files
# - Extract predicted answer from answered CSV files (from llm_output)
# - Add these fields to each result dict

print("\n" + "="*80)
print("🔧 PATCHING RESULTS: Adding question_id and predicted_answer_key")
print("="*80)

# Dataset CSV file paths
DATASET_CSV_PATHS = {
    "contextual": Path(SOURCE_DIR / "CORTEX_contextual.jsonl"),
    "synthesis": Path(SOURCE_DIR / "CORTEX_synthesis.jsonl"),
    "clinical_inference": Path(SOURCE_DIR / "CORTEX_clinical_inference.jsonl")
}

# Answered CSV file paths (for extracting predicted answers)
# Format: ../data/answered_{subdir}/{llm_id}/CORTEX_{dataset_type}_labeled.jsonl
ANSWERED_CSV_SUBDIRS = {
    "contextual": "predictions",
    "synthesis": "predictions",
    "clinical_inference": "predictions"
}

ANSWERED_CSV_SUFFIXES = {
    "contextual": "",
    "synthesis": "_mhr",
    "clinical_inference": "_l3"
}

def extract_predicted_answer_key(llm_output):
    """
    Extract predicted answer key (A, B, C, or D) from llm_output text.
    Takes the first character and converts to uppercase.
    """
    if pd.isna(llm_output) or not llm_output:
        return None
    llm_output_str = str(llm_output).strip().upper()
    if not llm_output_str:
        return None
    first_char = llm_output_str[0]
    if first_char in ['A', 'B', 'C', 'D']:
        return first_char
    return None

def load_dataset_mapping(dataset_type):
    """Load mapping from row index (record_id) to sidx (question_id)."""
    csv_path = DATASET_CSV_PATHS.get(dataset_type)
    if not csv_path or not csv_path.exists():
        print(f"  ⚠️  Dataset CSV not found: {csv_path}")
        return None
    
    try:
        df = read_table(csv_path)
        # Create mapping: row index -> sidx
        # record_id in test_labels.json is the row index as string
        mapping = {}
        for idx, row in df.iterrows():
            record_id = str(idx)
            sidx = str(row.get('sidx', ''))
            if sidx:
                mapping[record_id] = sidx
        print(f"  ✅ Loaded {len(mapping)} record_id->sidx mappings from {csv_path.name}")
        return mapping
    except Exception as e:
        print(f"  ❌ Error loading dataset CSV {csv_path}: {e}")
        return None

def load_predicted_answers(llm_id, dataset_type):
    """Load predicted answers from answered CSV file."""
    subdir = ANSWERED_CSV_SUBDIRS.get(dataset_type)
    suffix = ANSWERED_CSV_SUFFIXES.get(dataset_type, "")
    
    csv_path = Path(f"../data/predictions/{llm_id}/CORTEX_{dataset_type}_labeled.jsonl")
    
    if not csv_path.exists():
        print(f"  ⚠️  Answered CSV not found: {csv_path}")
        return None
    
    try:
        df = read_table(csv_path)
        # Create mapping: sidx -> predicted_answer_key
        mapping = {}
        for _, row in df.iterrows():
            sidx = str(row.get('sidx', ''))
            llm_output = row.get('llm_output', '')
            predicted_key = extract_predicted_answer_key(llm_output)
            if sidx and predicted_key:
                mapping[sidx] = predicted_key
        print(f"  ✅ Loaded {len(mapping)} sidx->predicted_answer mappings from {csv_path.name}")
        return mapping
    except Exception as e:
        print(f"  ❌ Error loading answered CSV {csv_path}: {e}")
        return None

# Load dataset mappings for all dataset types
print("\n📂 Loading dataset mappings...")
dataset_mappings = {}
for dataset_type in dataset_types:
    dataset_mappings[dataset_type] = load_dataset_mapping(dataset_type)

# Load predicted answers for all LLM-dataset combinations
print("\n📂 Loading predicted answers...")
predicted_answer_mappings = {}
for llm_id in llm_ids:
    predicted_answer_mappings[llm_id] = {}
    for dataset_type in dataset_types:
        predicted_answer_mappings[llm_id][dataset_type] = load_predicted_answers(llm_id, dataset_type)

# Patch the results
print("\n🔧 Patching results...")
patched_count = 0
missing_question_id_count = 0
missing_predicted_answer_count = 0

for llm_id in llm_ids:
    for method_name in method2path.keys():
        if method_name not in llm2method2dataset2results[llm_id]:
            continue
        
        for dataset_type in dataset_types:
            if dataset_type not in llm2method2dataset2results[llm_id][method_name]:
                continue
            
            results = llm2method2dataset2results[llm_id][method_name][dataset_type]
            dataset_mapping = dataset_mappings.get(dataset_type)
            predicted_mapping = predicted_answer_mappings.get(llm_id, {}).get(dataset_type)
            
            if not dataset_mapping:
                print(f"  ⚠️  Skipping {llm_id}-{method_name}-{dataset_type}: no dataset mapping")
                continue
            
            for result in results:
                record_id = str(result.get('record_id', ''))
                
                # Add question_id (sidx)
                if dataset_mapping and record_id in dataset_mapping:
                    result['question_id'] = dataset_mapping[record_id]
                    result['sidx'] = dataset_mapping[record_id]  # Also add as sidx for compatibility
                    patched_count += 1
                else:
                    missing_question_id_count += 1
                
                # Add predicted_answer_key
                if result.get('question_id') and predicted_mapping:
                    question_id = result['question_id']
                    if question_id in predicted_mapping:
                        result['predicted_answer_key'] = predicted_mapping[question_id]
                    else:
                        missing_predicted_answer_count += 1
                else:
                    missing_predicted_answer_count += 1

print(f"\n✅ Patching complete!")
print(f"   - Patched {patched_count} results with question_id")
print(f"   - Missing question_id: {missing_question_id_count}")
print(f"   - Missing predicted_answer_key: {missing_predicted_answer_count}")

# Verify patching worked
print("\n🔍 Verification sample (first 3 results):")
sample_count = 0
for llm_id in llm_ids:
    if sample_count >= 3:
        break
    for method_name in method2path.keys():
        if sample_count >= 3:
            break
        if method_name not in llm2method2dataset2results[llm_id]:
            continue
        for dataset_type in dataset_types:
            if sample_count >= 3:
                break
            if dataset_type not in llm2method2dataset2results[llm_id][method_name]:
                continue
            results = llm2method2dataset2results[llm_id][method_name][dataset_type]
            if results:
                sample = results[0]
                print(f"   {llm_id}-{method_name}-{dataset_type}:")
                print(f"      record_id: {sample.get('record_id')}")
                print(f"      question_id: {sample.get('question_id', 'MISSING')}")
                print(f"      predicted_answer_key: {sample.get('predicted_answer_key', 'MISSING')}")
                print(f"      ground_truth_correctness: {sample.get('ground_truth_correctness')}")
                sample_count += 1

print("\n" + "="*80)

In [ ]:
# Function to get shared record IDs across methods for a specific LLM and dataset type

def get_shared_record_ids(llm_id, method2dataset2results, dataset_type):
    """Get shared record IDs for a specific dataset type."""
    # Extract results for this dataset type
    method2results = {}
    for method_name in method2dataset2results:
        if dataset_type in method2dataset2results[method_name]:
            method2results[method_name] = method2dataset2results[method_name][dataset_type]
    
    available_methods = list(method2results.keys())
    print(f"\nProcessing {llm_id} ({dataset_display_names[dataset_type]}):")
    print(f"  Available methods: {available_methods}")

    if not available_methods:
        print(f"  No methods available for {llm_id}")
        return set()

    if len(available_methods) == 1:
        method_name = available_methods[0]
        record_ids = {r["record_id"] for r in method2results[method_name]}
        print(f"  Only one method available, using all {len(record_ids)} record IDs")
        return record_ids

    method2record_ids = {
        m: {r["record_id"] for r in method2results[m]}
        for m in available_methods
    }

    for m, ids in method2record_ids.items():
        print(f"    {m}: {len(ids)} unique record IDs")

    shared_ids = set.intersection(*method2record_ids.values())
    print(f"  Mutual record IDs: {len(shared_ids)}")

    print("  Records lost per method:")
    for m, ids in method2record_ids.items():
        lost = len(ids) - len(shared_ids)
        if lost:
            print(f"    {m}: {lost} records lost ({(lost / len(ids)) * 100:.1f}%)")
        else:
            print(f"    {m}: no records lost")

    return shared_ids

# === Main Execution ===
# Structure: llm2dataset2sharedRIDacrossMethods[llm_id][dataset_type] = shared_ids
llm2dataset2sharedRIDacrossMethods = {}

print(f"\n{'='*60}")
print("Finding mutual record IDs across methods for each LLM and dataset:")
print(f"{'='*60}")

for llm_id in llm_ids:
    llm2dataset2sharedRIDacrossMethods[llm_id] = {}
    for dataset_type in dataset_types:
        method2dataset2results = llm2method2dataset2results[llm_id]
        shared_ids = get_shared_record_ids(llm_id, method2dataset2results, dataset_type)
        llm2dataset2sharedRIDacrossMethods[llm_id][dataset_type] = shared_ids

print(f"\n{'='*60}")
print("Summary of mutual records per LLM and dataset:")
print(f"{'='*60}")

In [ ]:
# === Metric Calculation Functions ===

def calculate_ece(y_true, y_conf, n_bins=10):
    """Calculate Expected Calibration Error (ECE)."""
    if len(y_true) == 0:
        return 1.0  # Worst possible ECE
    
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0.0
    
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (y_conf > bin_lower) & (y_conf <= bin_upper)
        prop_in_bin = in_bin.mean()
        
        if prop_in_bin > 0:
            accuracy_in_bin = y_true[in_bin].mean()
            avg_confidence_in_bin = y_conf[in_bin].mean()
            ece += abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    
    return float(ece)

def calculate_all_metrics(y_true, y_conf, threshold=0.5):
    """
    Calculate all evaluation metrics with proper zero-division handling.
    
    Args:
        y_true: Binary ground truth labels (0 or 1)
        y_conf: Confidence scores between 0 and 1
        threshold: Threshold for binary classification
        
    Returns:
        Dictionary containing all metrics
    """
    if len(y_true) == 0:
        return {
            'n_samples': 0,
            'accuracy': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'sensitivity': 0.0,
            'specificity': 0.0,
            'ece': 1.0,
            'brier': 1.0,
            'auroc': 0.5,
            'aucpr': 0.0
        }
    
    # Convert confidence to binary predictions
    y_pred = (y_conf >= threshold).astype(int)
    
    # Basic metrics
    metrics = {
        'n_samples': len(y_true),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'ece': calculate_ece(y_true, y_conf),
        'brier': float(brier_score_loss(y_true, y_conf))
    }
    
    # Handle edge cases for classification metrics
    if len(np.unique(y_true)) < 2:
        # All samples have the same label
        if np.all(y_true == y_pred):
            metrics.update({
                'precision': 1.0,
                'recall': 1.0,
                'f1': 1.0,
                'sensitivity': 1.0,
                'specificity': 1.0
            })
        else:
            metrics.update({
                'precision': 0.0,
                'recall': 0.0,
                'f1': 0.0,
                'sensitivity': 0.0,
                'specificity': 0.0
            })
    else:
        # Calculate with proper zero division handling
        metrics['precision'] = float(precision_score(y_true, y_pred, zero_division=0.0))
        metrics['recall'] = float(recall_score(y_true, y_pred, zero_division=0.0))
        metrics['f1'] = float(f1_score(y_true, y_pred, zero_division=0.0))
        
        # Calculate sensitivity and specificity
        try:
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
            metrics['sensitivity'] = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
            metrics['specificity'] = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
        except ValueError:
            metrics['sensitivity'] = 0.0
            metrics['specificity'] = 0.0
    
    # AUC metrics (handle edge cases)
    if len(np.unique(y_true)) < 2:
        metrics['auroc'] = 0.5  # Random performance
        metrics['aucpr'] = float(np.mean(y_true))  # Baseline AUCPR
    else:
        try:
            metrics['auroc'] = float(roc_auc_score(y_true, y_conf))
            precision, recall, _ = precision_recall_curve(y_true, y_conf)
            metrics['aucpr'] = float(auc(recall, precision))
        except Exception:
            metrics['auroc'] = 0.5
            metrics['aucpr'] = float(np.mean(y_true))
    
    return metrics

def calculate_metrics_for_llm_method(llm_id, method_name, results, shared_record_ids, dataset_type):
    """Calculate metrics for a specific LLM and method by averaging per-dataset metrics."""
    
    # Filter results to only include shared record IDs (or use all if shared_record_ids is empty)
    if shared_record_ids:
        filtered_results = [r for r in results if r["record_id"] in shared_record_ids]
    else:
        # If no shared IDs, use all available results from this method
        filtered_results = results
    
    if not filtered_results:
        print(f"Warning: No records found for {llm_id}-{method_name}")
        return None, []
    
    # Group results by dataset
    dataset_results = {}
    for result in filtered_results:
        dataset = result.get("dataset", "unknown")
        if dataset not in dataset_results:
            dataset_results[dataset] = []
        dataset_results[dataset].append(result)
    
    # Calculate metrics for each dataset
    dataset_metrics = {}
    total_samples = 0
    
    for dataset, dataset_data in dataset_results.items():
        if not dataset_data:
            continue
            
        y_true = np.array([r["ground_truth_correctness"] for r in dataset_data])
        y_conf = np.array([r["confidence_score"] for r in dataset_data])
        
        # Check for NaN/inf values
        if np.isnan(y_conf).any() or np.isinf(y_conf).any():
            bad_indices = np.where(np.isnan(y_conf) | np.isinf(y_conf))[0]
            print(f"❌ NaN/Inf detected in {llm_id}-{method_name}-{dataset}")
            y_true = np.delete(y_true, bad_indices)
            y_conf = np.delete(y_conf, bad_indices)
        
        dataset_metric = calculate_all_metrics(y_true, y_conf)
        
        # Calculate weighted cost at optimal coverage for this dataset
        # Get risk labels directory based on the dataset_type parameter
        risk_labels_dir = RISK_LABELS_DIR_MAP.get(dataset_type, RISK_LABELS_DIR_MAP["contextual"])
        
        # Extract confidence scores for cost calculation
        confidence_scores = y_conf.tolist()
        
        # Calculate cost
        try:
            cost = calculate_weighted_cost_at_fixed_threshold(
                dataset_data, 
                confidence_scores, 
                risk_labels_dir
            )
        except Exception as e:
            print(f"Warning: Could not calculate cost for {llm_id}-{method_name}-{dataset}: {e}")
            cost = 0.0
        
        # Add cost to dataset metric
        dataset_metric['cost'] = cost
        
        dataset_metrics[dataset] = dataset_metric
        total_samples += len(dataset_data)
    
    if not dataset_metrics:
        return None, []
    
    # Average metrics across datasets (unweighted average)
    metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'sensitivity', 'specificity', 'ece', 'brier', 'auroc', 'aucpr', 'cost']
    
    averaged_metrics = {}
    for key in metric_keys:
        values = [dataset_metrics[dataset][key] for dataset in dataset_metrics.keys()]
        averaged_metrics[key] = np.mean(values)
    
    # Add metadata
    averaged_metrics.update({
        'llm_id': llm_id,
        'method_name': method_name,
        'n_samples': total_samples,  # Total samples (for reference)
        'shared_record_ids_count': len(shared_record_ids) if shared_record_ids else 0,
        'used_all_records': not bool(shared_record_ids),  # Flag indicating if all records were used
        'filtered_results_count': len(filtered_results),
        'n_datasets': len(dataset_metrics),
        'datasets_included': list(dataset_metrics.keys())
    })
    
    print(f"  {llm_id}-{method_name}: {len(filtered_results)} records across {len(dataset_metrics)} datasets, Avg ECE: {averaged_metrics['ece']:.4f}, Avg Brier: {averaged_metrics['brier']:.4f}")
    
    return averaged_metrics, filtered_results

def process_all_llm_methods(save_individual=False):
    """Process all LLM-method combinations for both datasets and save individual results."""
    print("\n" + "="*60)
    print("🚀 Calculating dataset-balanced metrics for all LLMs and methods")
    print("="*60)
    
    all_metrics = []
    
    for dataset_type in dataset_types:
        print(f"\n{'='*60}")
        print(f"Processing {dataset_display_names[dataset_type]}:")
        print(f"{'='*60}")
        
        for llm_id in llm_ids:
            print(f"\nProcessing {llm_id} ({dataset_display_names[dataset_type]}):")
            shared_record_ids = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
            
            if not shared_record_ids:
                print(f"  ⚠️  No shared record IDs for {llm_id}, using all available records from each method...")
            else:
                print(f"  📊 Using {len(shared_record_ids)} shared record IDs")
            
            for method_name in method2path.keys():
                if method_name in llm2method2dataset2results[llm_id]:
                    if dataset_type in llm2method2dataset2results[llm_id][method_name]:
                        results = llm2method2dataset2results[llm_id][method_name][dataset_type]
                        
                        # Use shared IDs if available, otherwise use all IDs from this method
                        record_ids_to_use = shared_record_ids if shared_record_ids else None
                        
                        metrics, filtered_results = calculate_metrics_for_llm_method(
                            llm_id, method_name, results, record_ids_to_use, dataset_type
                        )
                        
                        if metrics:
                            # Add dataset type to metrics
                            metrics['dataset_type'] = dataset_type
                            metrics['dataset_display_name'] = dataset_display_names[dataset_type]
                            all_metrics.append(metrics)
    
    print(f"\n✅ Successfully processed {len(all_metrics)} LLM-method-dataset combinations")
    return all_metrics

def create_aggregated_results(all_metrics, save_aggregated=False):
    """Create DataFrame and save aggregated results for both datasets."""
    if not all_metrics:
        print("❌ No metrics calculated - no valid data found!")
        return None
    
    print(f"\n" + "="*60)
    print("✅ Creating aggregated results DataFrame")
    print("="*60)
    
    # Create DataFrame
    df_metrics = pd.DataFrame(all_metrics)
    print(f"✅ Created main DataFrame with {len(df_metrics)} rows")
    
    # Reorder and rename columns as requested
    df_metrics = df_metrics.rename(columns={
        'llm_id': 'llm',
        'method_name': 'method',
        'n_samples': 'samples'
    })
    print(f"✅ Renamed columns successfully")
    
    # Convert LLM IDs to display names
    df_metrics['llm'] = df_metrics['llm'].map(lambda x: LLM_DISPLAY_NAMES.get(x, x))
    
    # Calculate corrects and incorrects from filtered_results_count and accuracy
    df_metrics['#corrects'] = (df_metrics['samples'] * df_metrics['accuracy']).round().astype(int)
    df_metrics['#incorrects'] = df_metrics['samples'] - df_metrics['#corrects']
    
    # Calculate percentages
    df_metrics['%corrects'] = (df_metrics['#corrects'] / df_metrics['samples'] * 100).round(2)
    df_metrics['%incorrects'] = (df_metrics['#incorrects'] / df_metrics['samples'] * 100).round(2)
    print(f"✅ Added #corrects, #incorrects, %corrects, and %incorrects columns")
    
    # Ensure dataset_type and dataset_display_name are included
    if 'dataset_type' not in df_metrics.columns:
        df_metrics['dataset_type'] = 'unknown'
    if 'dataset_display_name' not in df_metrics.columns:
        df_metrics['dataset_display_name'] = 'Unknown'
    
    # Define final column order
    final_columns = ['dataset_display_name', 'llm', 'method', 'dataset_type', 'samples', '#corrects', '%corrects', '#incorrects', '%incorrects',
                    'ece', 'brier', 'accuracy', 'precision', 'recall', 'f1', 
                    'sensitivity', 'specificity', 'aucpr', 'auroc', 'cost']
    
    # Select the columns we need (only those that exist)
    available_columns = [col for col in final_columns if col in df_metrics.columns]
    df_metrics = df_metrics[available_columns]
    print(f"✅ Selected required columns")
    
    # Create hierarchical index with dataset_display_name, llm, and method
    df_metrics_hierarchical = df_metrics.set_index(['dataset_display_name', 'llm', 'method'])
    print(f"✅ Created hierarchical index with Dataset Display Name > LLM > Method")
    
    return df_metrics_hierarchical

# === Main Execution ===
print("🚀 Starting main processing...")
all_metrics = process_all_llm_methods(save_individual=False)

print("📊 Creating aggregated results...")
df_metrics = create_aggregated_results(all_metrics, save_aggregated=False)

print(f"\n" + "="*60)
print("🎉 Processing complete!")
print("="*60)
print(f"✅ df_metrics created with {len(df_metrics)} rows")

In [ ]:
# === Coverage Curve Calculation Functions ===

def calculate_auroc_coverage_curve(results, confidence_scores, method_name="Method"):
    """
    Calculate AUROC at different coverage levels (human workload percentages).
    
    Args:
        results: List of result dicts with 'ground_truth_correctness'
        confidence_scores: List of confidence scores (same order as results)
        method_name: Name of the method (for logging)
    
    Returns:
        human_workloads: List of human workload percentages (0-99)
        system_auroc_scores: List of system AUROC scores at each workload level
    """
    # Combine results and confidence scores
    data = list(zip(results, confidence_scores))
    
    # Sort by confidence score (lowest to highest)
    # Lower confidence = send to human first
    data_sorted = sorted(data, key=lambda x: x[1])
    
    # Extract ground truth correctness and confidence scores
    gt_correctness = np.array([item[0]['ground_truth_correctness'] for item in data_sorted])
    conf_scores = np.array([item[1] for item in data_sorted])
    
    total_samples = len(data_sorted)
    if total_samples == 0:
        return [], []
    
    human_workloads = []
    system_auroc_scores = []
    
    # Simulate different human workload levels (0% to 99% in 1% increments)
    # Exclude 100% since at 100% there are no AI-handled cases (all go to humans)
    for human_workload_pct in range(0, 100):
        num_human_reviews = int(total_samples * human_workload_pct / 100)
        
        # Retained subset (highest confidence samples that AI answers)
        if num_human_reviews < total_samples:
            retained_gt = gt_correctness[num_human_reviews:]
            retained_conf = conf_scores[num_human_reviews:]
            
            # Calculate AUROC on the retained subset
            if len(retained_gt) > 0 and len(np.unique(retained_gt)) > 1:
                auroc = roc_auc_score(retained_gt, retained_conf)
            else:
                # If all samples have same label, AUROC is undefined (use 0.5)
                auroc = 0.5
        else:
            # All samples sent to human, no AI-handled cases
            # This shouldn't happen with range(0, 100), but keep as safety check
            auroc = 0.5  # Default when no AI cases
        
        human_workloads.append(human_workload_pct)
        system_auroc_scores.append(auroc)
    
    return human_workloads, system_auroc_scores


def calculate_ece_coverage_curve(results, confidence_scores, method_name="Method"):
    """
    Calculate ECE at different coverage levels (human workload percentages).
    
    Args:
        results: List of result dicts with 'ground_truth_correctness'
        confidence_scores: List of confidence scores (same order as results)
        method_name: Name of the method (for logging)
    
    Returns:
        human_workloads: List of human workload percentages (0-99)
        ece_values: List of ECE values at each workload level
    """
    # Combine results and confidence scores
    data = list(zip(results, confidence_scores))
    
    # Sort by confidence score (lowest to highest)
    data_sorted = sorted(data, key=lambda x: x[1])
    
    # Extract ground truth correctness and confidence scores
    gt_correctness = np.array([item[0]['ground_truth_correctness'] for item in data_sorted])
    conf_scores = np.array([item[1] for item in data_sorted])
    
    total_samples = len(data_sorted)
    if total_samples == 0:
        return [], []
    
    human_workloads = []
    ece_values = []
    
    # Simulate different human workload levels (0% to 99% in 1% increments)
    # Exclude 100% since at 100% there are no AI-handled cases (all go to humans)
    for human_workload_pct in range(0, 100):
        num_human_reviews = int(total_samples * human_workload_pct / 100)
        
        # Retained subset (highest confidence samples that AI answers)
        if num_human_reviews < total_samples:
            retained_gt = gt_correctness[num_human_reviews:]
            retained_conf = conf_scores[num_human_reviews:]
            
            # Calculate ECE on the retained subset
            if len(retained_gt) > 0:
                ece = calculate_ece(retained_gt, retained_conf, n_bins=10)
            else:
                ece = 1.0  # Worst possible ECE
        else:
            # All samples sent to human, no AI-handled cases
            # This shouldn't happen with range(0, 100), but keep as safety check
            ece = 1.0  # Worst possible ECE
        
        human_workloads.append(human_workload_pct)
        ece_values.append(ece)
    
    return human_workloads, ece_values

# === Create Coverage Data for Each Dataset Type ===

def create_coverage_data_for_dataset(dataset_type):
    """Create coverage data for a specific dataset type."""
    risk_coverage_data = {}
    ece_coverage_data = {}
    
    print(f"\nCalculating AUROC-Coverage and ECE-Coverage data for {dataset_display_names[dataset_type]}...")
    
    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        risk_coverage_data[llm_display] = {}
        ece_coverage_data[llm_display] = {}
        
        shared_record_ids = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
        
        for method_name in method2path.keys():
            if method_name not in llm2method2dataset2results[llm_id]:
                continue
            if dataset_type not in llm2method2dataset2results[llm_id][method_name]:
                continue
            
            results = llm2method2dataset2results[llm_id][method_name][dataset_type]
            
            # Filter to shared record IDs if available
            if shared_record_ids:
                filtered_results = [r for r in results if r["record_id"] in shared_record_ids]
            else:
                filtered_results = results
            
            if not filtered_results:
                continue
            
            confidence_scores = [r["confidence_score"] for r in filtered_results]
            
            # Calculate AUROC coverage curve
            workloads, auroc_scores = calculate_auroc_coverage_curve(
                filtered_results, confidence_scores, method_name
            )
            
            # Calculate ECE coverage curve
            ece_workloads, ece_vals = calculate_ece_coverage_curve(
                filtered_results, confidence_scores, method_name
            )
            
            risk_coverage_data[llm_display][method_name] = {
                'human_workloads': workloads,
                'system_auroc_scores': auroc_scores
            }
            
            ece_coverage_data[llm_display][method_name] = {
                'human_workloads': ece_workloads,
                'ece_values': ece_vals
            }
    
    return risk_coverage_data, ece_coverage_data

def create_aggregated_curves_for_dataset(risk_coverage_data, ece_coverage_data):
    """Create aggregated curves across all LLMs for a dataset."""
    aggregated_curves = {}
    aggregated_ece_curves = {}
    
    for method_name in ['CCPS', 'PIK', 'PTRUE', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F']:
        all_workloads = []
        all_auroc_scores = []
        all_ece_values = []
        
        for llm_display in risk_coverage_data:
            if method_name in risk_coverage_data[llm_display]:
                data = risk_coverage_data[llm_display][method_name]
                all_workloads.append(data['human_workloads'])
                all_auroc_scores.append(data['system_auroc_scores'])
            
            if method_name in ece_coverage_data[llm_display]:
                ece_data = ece_coverage_data[llm_display][method_name]
                all_ece_values.append(ece_data['ece_values'])
        
        if all_workloads and all_auroc_scores:
            # Use first workload as reference (should be same for all)
            workloads = all_workloads[0]
            mean_auroc_scores = np.mean(all_auroc_scores, axis=0)
            std_auroc_scores = np.std(all_auroc_scores, axis=0)
            
            aggregated_curves[method_name] = {
                'workloads': workloads,
                'mean_auroc_scores': mean_auroc_scores,
                'std_auroc_scores': std_auroc_scores
            }
        
        if all_ece_values:
            workloads = ece_coverage_data[list(ece_coverage_data.keys())[0]][method_name]['human_workloads']
            mean_ece = np.mean(all_ece_values, axis=0)
            std_ece = np.std(all_ece_values, axis=0)
            
            aggregated_ece_curves[method_name] = {
                'workloads': workloads,
                'mean_ece': mean_ece,
                'std_ece': std_ece
            }
    
    return aggregated_curves, aggregated_ece_curves

# === Create Coverage Data for All Datasets ===

# Contextual dataset
risk_coverage_data_contextual, ece_coverage_data_contextual = create_coverage_data_for_dataset("contextual")
aggregated_curves_contextual, aggregated_ece_curves_contextual = create_aggregated_curves_for_dataset(
    risk_coverage_data_contextual, ece_coverage_data_contextual
)

# Synthesis dataset (note: variable name is 'risk_coverage_data' not 'risk_coverage_data_synthesis')
risk_coverage_data, ece_coverage_data = create_coverage_data_for_dataset("synthesis")
aggregated_curves, aggregated_ece_curves = create_aggregated_curves_for_dataset(
    risk_coverage_data, ece_coverage_data
)

# Clinical inference dataset
risk_coverage_data_clinical, ece_coverage_data_clinical = create_coverage_data_for_dataset("clinical_inference")
aggregated_curves_clinical, aggregated_ece_curves_clinical = create_aggregated_curves_for_dataset(
    risk_coverage_data_clinical, ece_coverage_data_clinical
)

print("\n✅ Coverage data and aggregated curves created for all datasets!")

### COMBO

In [ ]:
METHODS_TO_SHOW = ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']

In [ ]:
# ==============================================================================
# ROW 1: DIAGNOSTIC VALIDITY - Calibration-Discrimination Analysis
# ==============================================================================
# Uses: combined_ece_auroc_calibration_all.png logic
# Message: CCPS is the only method that is both discriminative AND calibrated

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Polygon, Ellipse
from matplotlib.lines import Line2D

def create_row1_diagnostic_validity(
    df_metrics, 
    llm2method2dataset2results,
    llm2dataset2sharedRIDacrossMethods,
    method2path, 
    llm_ids,
    dataset_types,
    dataset_display_names,
    save_path,
    font_size_adjust=0
):
    """
    Row 1: Diagnostic Validity - Creates separate figures for each dataset
    Each figure: Left: ECE-AUROC Scatter showing methods in performance zones
                 Right: Calibration Reliability Diagram with histogram
    """
    print("🎨 Creating Row 1: Diagnostic Validity (separate figures for each dataset)...")
    
    df_filtered = df_metrics.reset_index()
    
    # Define datasets to process (including merged)
    dataset_configs = [
        {"dataset_type": "contextual", "display_name": dataset_display_names.get("contextual", "CORAL-MCQA-Contextual")},
        {"dataset_type": "synthesis", "display_name": dataset_display_names.get("synthesis", "CORAL-MCQA-Synthesis")},
        {"dataset_type": "clinical_inference", "display_name": dataset_display_names.get("clinical_inference", "CORAL-MCQA-Clinical-Inference")},
        {"dataset_type": None, "display_name": "All Datasets (Merged)"}  # None = merged
    ]
    
    # Helper function to create a single figure for a dataset
    def create_single_figure(dataset_type, dataset_display_name, save_path_single):
        """Create a single figure for a specific dataset."""
        # Filter data for this dataset
        if dataset_type is None:
            # Merged: use all data
            row_df = df_filtered.copy()
            row_dataset_types = dataset_types
        else:
            # Single dataset: filter by dataset_type
            row_df = df_filtered[df_filtered['dataset_type'] == dataset_type].copy()
            row_dataset_types = [dataset_type]
        
        # Create figure with GridSpec
        fig = plt.figure(figsize=(20, 8))
        gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[3, 1], 
                              hspace=0.05, wspace=0.12)
        
        ax_scatter = fig.add_subplot(gs[:, 0])
        ax_calib = fig.add_subplot(gs[0, 1])
        ax_hist = fig.add_subplot(gs[1, 1], sharex=ax_calib)
        
        # =========================================================================
        # LEFT: ECE-AUROC Scatter with Performance Zones
        # =========================================================================
        
        def add_background_regions(ax, alpha=0.08):
            # Perfect zone (top-right)
            perfect_zone = Polygon(
                [(0.7, 0.7), (1.0, 0.7), (1.0, 1.0), (0.7, 1.0)],
                facecolor='#C8E6C9', alpha=0.20, edgecolor='none', zorder=0
            )
            ax.add_patch(perfect_zone)
            ax.text(0.85, 0.97, 'Ideal\n(Calibrated + Discriminative)', 
                    ha='center', va='top', fontsize=14, alpha=0.6, 
                    style='italic', weight='bold')
            
            # Good calibration, poor discrimination (top-left)
            good_cal_zone = Polygon(
                [(0.0, 0.7), (0.7, 0.7), (0.7, 1.0), (0.0, 1.0)],
                facecolor='white', alpha=0.10, edgecolor='none', zorder=0
            )
            ax.add_patch(good_cal_zone)
            ax.text(0.55, 0.97, 'High Discrimination\n(Poorly Calibrated)', 
                    ha='center', va='top', fontsize=14, alpha=0.5,
                    style='italic', weight='bold')
            
            # Good discrimination, poor calibration (bottom-right)
            good_disc_zone = Polygon(
                [(0.7, 0.0), (1.0, 0.0), (1.0, 0.7), (0.7, 0.7)],
                facecolor='white', alpha=0.10, edgecolor='none', zorder=0
            )
            ax.add_patch(good_disc_zone)
            ax.text(0.85, 0.45, 'Well-Calibrated\n(Low Discrimination)', 
                    ha='center', va='center', fontsize=14, alpha=0.5,
                    style='italic', weight='bold')
            
            # Poor both (bottom-left)
            poor_zone = Polygon(
                [(0.0, 0.0), (0.7, 0.0), (0.7, 0.7), (0.0, 0.7)],
                facecolor='#FFCDD2', alpha=0.15, edgecolor='none', zorder=0
            )
            ax.add_patch(poor_zone)
            ax.text(0.55, 0.45, 'Suboptimal', 
                    ha='center', va='center', fontsize=14, alpha=0.5,
                    style='italic', weight='bold')
        
        add_background_regions(ax_scatter, alpha=0.06)
        
        # Prepare scatter data
        method_data = {}
        for method in METHODS_TO_SHOW:
            if method not in row_df['method'].unique():
                continue
            method_df = row_df[row_df['method'] == method]
            if len(method_df) == 0:
                continue
            calibration_scores = 1 - method_df['ece'].values
            auroc_scores = method_df['auroc'].values
            
            method_data[method] = {
                'calibration': calibration_scores,
                'auroc': auroc_scores,
                'mean_cal': np.mean(calibration_scores),
                'mean_auroc': np.mean(auroc_scores),
                'std_cal': np.std(calibration_scores),
                'std_auroc': np.std(auroc_scores)
            }
        
        # Plot each method
        for method, data in method_data.items():
            method_info = method2path.get(method, {})
            color = method_info.get("base_color", "gray")
            shape = method_info.get("shape", "o")
            display_name = method_info.get("method_name", method)
            
            # Individual points
            ax_scatter.scatter(data['calibration'], data['auroc'],
                              marker=shape, s=80, color=color, alpha=0.3,
                              edgecolors='white', linewidths=0.5, zorder=3)
            
            # Mean marker
            ax_scatter.scatter([data['mean_cal']], [data['mean_auroc']],
                              marker=shape, s=800, color=color, alpha=0.9,
                              edgecolors='white', linewidths=3,
                              label=display_name, zorder=10)
            
            # Confidence ellipse
            if data['std_cal'] > 0 and data['std_auroc'] > 0:
                ellipse = Ellipse((data['mean_cal'], data['mean_auroc']),
                                width=2*data['std_cal'], height=2*data['std_auroc'],
                                facecolor=color, alpha=0.15, edgecolor=color,
                                linewidth=1.5, linestyle='--', zorder=2)
                ax_scatter.add_patch(ellipse)
        
        ax_scatter.set_xlabel('Calibration Quality (1 - ECE) →', fontsize=14 + font_size_adjust, fontweight='bold')
        ax_scatter.set_ylabel('Discrimination (AUROC) →', fontsize=14 + font_size_adjust, fontweight='bold')
        ax_scatter.set_title(f'(A) Calibration vs. Discrimination Trade-off - {dataset_display_name}', 
                            fontsize=16 + font_size_adjust, fontweight='bold', pad=15)
        ax_scatter.set_xlim(0.38, 1.02)
        ax_scatter.set_ylim(0.38, 1.02)
        ax_scatter.grid(True, alpha=0.4, linestyle='-', linewidth=0.8, color='gray')
        ax_scatter.plot([0, 1], [0, 1], 'k:', linewidth=1.5, alpha=0.3, zorder=1)
        ax_scatter.tick_params(labelsize=14 + font_size_adjust)
        
        # =========================================================================
        # RIGHT: Calibration Reliability Diagram with Histogram
        # =========================================================================
        
        n_bins = 10
        MIN_BIN_COUNT = 5  # Minimum bin population threshold for visualization
        methods_calibration = {}
        
        for method_name in method2path.keys():
            all_confidences = []
            all_correctness = []
            
            for llm_id in llm_ids:
                for dt in row_dataset_types:
                    if method_name not in llm2method2dataset2results.get(llm_id, {}):
                        continue
                    if dt not in llm2method2dataset2results[llm_id].get(method_name, {}):
                        continue
                    
                    results = llm2method2dataset2results[llm_id][method_name][dt]
                    shared_ids = llm2dataset2sharedRIDacrossMethods.get(llm_id, {}).get(dt, set())
                    
                    if shared_ids:
                        filtered = [r for r in results if r["record_id"] in shared_ids]
                    else:
                        filtered = results
                    
                    if filtered:
                        all_confidences.extend([r["confidence_score"] for r in filtered])
                        all_correctness.extend([r["ground_truth_correctness"] for r in filtered])
            
            if not all_confidences:
                continue
            
            confidences = np.array(all_confidences)
            correctness = np.array(all_correctness)
            
            bins = np.linspace(0, 1, n_bins + 1)
            bin_indices = np.digitize(confidences, bins) - 1
            bin_indices = np.clip(bin_indices, 0, n_bins - 1)
            
            bin_conf, bin_acc, bin_count = [], [], []
            for b in range(n_bins):
                mask = (bin_indices == b)
                if mask.sum() > 0:
                    bin_conf.append(confidences[mask].mean())
                    bin_acc.append(correctness[mask].mean())
                    bin_count.append(mask.sum())
                else:
                    bin_conf.append(None)
                    bin_acc.append(None)
                    bin_count.append(0)
            
            methods_calibration[method_name] = {
                'bin_conf': bin_conf,
                'bin_acc': bin_acc,
                'bin_count': bin_count,
                'confidences': confidences
            }
        
        # Perfect calibration line
        ax_calib.plot([0, 1], [0, 1], 'k--', linewidth=2.5, alpha=0.5, 
                     label='Perfect Calibration', zorder=1)
        
        # Plot each method
        for method_name, data in methods_calibration.items():
            method_info = method2path.get(method_name, {})
            color = method_info.get("base_color", "gray")
            shape = method_info.get("shape", "o")
            display_name = method_info.get("method_name", method_name)
            
            # Suppress extremely low-count bins for visual stability only
            valid_indices = [
                i for i, conf in enumerate(data['bin_conf'])
                if conf is not None and data['bin_count'][i] >= MIN_BIN_COUNT
            ]
            if not valid_indices:
                continue
            
            x_vals = [data['bin_conf'][i] for i in valid_indices]
            y_vals = [data['bin_acc'][i] for i in valid_indices]
            
            ax_calib.plot(x_vals, y_vals, marker=shape, linestyle='-',
                         linewidth=4, markersize=16, color=color,
                         label=display_name,
                         markeredgewidth=2, markeredgecolor='white', zorder=5)
            
            # Histogram
            ax_hist.hist(data['confidences'], bins=20, alpha=0.5, 
                        color=color, edgecolor='white', linewidth=0.5)
        
        ax_calib.set_ylabel('Empirical Accuracy →', fontsize=14 + font_size_adjust, fontweight='bold')
        ax_calib.set_xlim(-0.02, 1.02)
        ax_calib.set_ylim(-0.02, 1.02)
        ax_calib.grid(True, alpha=0.4, linestyle='-', linewidth=0.8, color='gray')
        ax_calib.set_title(f'(B) Calibration Reliability Diagram - {dataset_display_name}', 
                          fontsize=16 + font_size_adjust, fontweight='bold', pad=15)
        ax_calib.tick_params(labelbottom=False, labelsize=14 + font_size_adjust)
        
        ax_hist.set_xlabel('Predicted Confidence →', fontsize=14 + font_size_adjust, fontweight='bold')
        ax_hist.set_ylabel('Count', fontsize=12 + font_size_adjust, fontweight='bold')
        ax_hist.set_xlim(-0.02, 1.02)
        ax_hist.grid(True, axis='y', alpha=0.4, linestyle='-', linewidth=0.8, color='gray')
        ax_hist.tick_params(labelsize=14 + font_size_adjust)
        
        # Legend
        handles_calib, labels_calib = ax_calib.get_legend_handles_labels()
        legend_handles = []
        legend_labels = []
        seen_labels = set()
        
        for handle, label in zip(handles_calib, labels_calib):
            if label != 'Perfect Calibration' and label not in seen_labels:
                legend_handles.append(handle)
                legend_labels.append(label)
                seen_labels.add(label)
        
        if legend_handles and legend_labels:
            fig.legend(legend_handles, legend_labels, loc='upper center', 
                       bbox_to_anchor=(0.5, 0.995), ncol=len(legend_labels), 
                       fontsize=13 + font_size_adjust, 
                       frameon=True, fancybox=True, shadow=True)
        
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.savefig(save_path_single, dpi=300, bbox_inches='tight')
        plt.savefig(str(save_path_single).replace('.png', '.pdf'), dpi=300, bbox_inches='tight')
        print(f"✅ Saved figure for {dataset_display_name} to: {save_path_single}")
        plt.show()
        return fig
    
    # Create separate figures for each dataset
    figures = []
    base_path = Path(save_path)
    
    for config in dataset_configs:
        dataset_type = config["dataset_type"]
        dataset_display_name = config["display_name"]
        
        # Create filename based on dataset
        if dataset_type is None:
            filename_suffix = "merged"
        else:
            filename_suffix = dataset_type
        
        save_path_single = base_path.parent / f"{base_path.stem}_{filename_suffix}{base_path.suffix}"
        fig = create_single_figure(dataset_type, dataset_display_name, save_path_single)
        figures.append(fig)
    
    # Return the last figure (for compatibility)
    return figures[-1] if figures else None

# Generate Row 1
row1_save_path = OVERALL_RESULTS_PATH / "master_figure_row1_diagnostic_validity.png"
fig_row1 = create_row1_diagnostic_validity(
    df_metrics=df_metrics,
    llm2method2dataset2results=llm2method2dataset2results,
    llm2dataset2sharedRIDacrossMethods=llm2dataset2sharedRIDacrossMethods,
    method2path=method2path,
    llm_ids=llm_ids,
    dataset_types=dataset_types,
    dataset_display_names=dataset_display_names,
    save_path=row1_save_path
)

In [ ]:
# ==============================================================================
# ROW 1: DIAGNOSTIC VALIDITY - Combined Figure (All Datasets Stacked)
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Ellipse, FancyBboxPatch
from pathlib import Path

def create_row1_diagnostic_validity(
    df_metrics, 
    llm2method2dataset2results,
    llm2dataset2sharedRIDacrossMethods,
    method2path, 
    llm_ids,
    dataset_types,
    dataset_display_names,
    save_path,
    font_addition=0
):

    print("🎨 Creating combined diagnostic validity figure...")

    df_filtered = df_metrics.reset_index()

    dataset_configs = [
        {"dataset_type": "contextual", "display_name": "Contextual"},
        {"dataset_type": "synthesis", "display_name": "Synthesis"},
        {"dataset_type": "clinical_inference", "display_name": "Inference"},
        {"dataset_type": None, "display_name": "Merged"}
    ]

    n_rows = len(dataset_configs)

    fig = plt.figure(figsize=(20, 7 * n_rows))

    gs = fig.add_gridspec(
        n_rows,
        2,
        width_ratios=[1, 1],
        hspace=0.15,
        wspace=0.12
    )

    legend_handles = []
    legend_labels = []

    for row_idx, config in enumerate(dataset_configs):

        dataset_type = config["dataset_type"]
        dataset_display_name = config["display_name"]

        if dataset_type is None:
            row_df = df_filtered.copy()
            row_dataset_types = dataset_types
        else:
            row_df = df_filtered[df_filtered['dataset_type'] == dataset_type].copy()
            row_dataset_types = [dataset_type]

        ax_scatter = fig.add_subplot(gs[row_idx, 0])
        ax_calib = fig.add_subplot(gs[row_idx, 1])

        # =====================================================================
        # LEFT: Scatter
        # =====================================================================

        def add_background_regions(ax):

            perfect_zone = Polygon(
                [(0.7,0.7),(1,0.7),(1,1),(0.7,1)],
                facecolor="#C8E6C9",
                alpha=0.20,
                edgecolor="none"
            )
            ax.add_patch(perfect_zone)

            ax.text(
                0.85,0.97,
                "Ideal\n(Calibrated + Discriminative)",
                ha="center",
                va="top",
                fontsize=14 + font_addition,
                alpha=0.6,
                style="italic",
                weight="bold"
            )

            poor_zone = Polygon(
                [(0,0),(0.7,0),(0.7,0.7),(0,0.7)],
                facecolor="#FFCDD2",
                alpha=0.15,
                edgecolor="none"
            )
            ax.add_patch(poor_zone)

        add_background_regions(ax_scatter)

        method_data = {}

        for method in METHODS_TO_SHOW:

            if method not in row_df['method'].unique():
                continue

            method_df = row_df[row_df['method'] == method]

            calibration_scores = 1 - method_df["ece"].values
            auroc_scores = method_df["auroc"].values

            method_data[method] = {
                "calibration": calibration_scores,
                "auroc": auroc_scores,
                "mean_cal": np.mean(calibration_scores),
                "mean_auroc": np.mean(auroc_scores),
                "std_cal": np.std(calibration_scores),
                "std_auroc": np.std(auroc_scores)
            }

        for method, data in method_data.items():

            method_info = method2path.get(method, {})

            color = method_info.get("base_color","gray")
            shape = method_info.get("shape","o")
            display_name = method_info.get("method_name",method)

            ax_scatter.scatter(
                data["calibration"],
                data["auroc"],
                marker=shape,
                s=80,
                color=color,
                alpha=0.3,
                edgecolors="white",
                linewidths=0.5
            )

            m = ax_scatter.scatter(
                [data["mean_cal"]],
                [data["mean_auroc"]],
                marker=shape,
                s=800,
                color=color,
                alpha=0.9,
                edgecolors="white",
                linewidths=3,
                label=display_name
            )

            if row_idx == 0:
                from matplotlib.lines import Line2D
                legend_handles.append(
                    Line2D(
                        [0],
                        [0],
                        marker=shape,
                        color='none',
                        markerfacecolor=color,
                        markeredgecolor='white',
                        markersize=16,
                        markeredgewidth=2,
                        linestyle='None'
                    )
                )
                legend_labels.append(display_name)

            if data["std_cal"]>0 and data["std_auroc"]>0:

                ellipse = Ellipse(
                    (data["mean_cal"],data["mean_auroc"]),
                    width=2*data["std_cal"],
                    height=2*data["std_auroc"],
                    facecolor=color,
                    alpha=0.15,
                    edgecolor=color,
                    linewidth=1.5,
                    linestyle="--"
                )

                ax_scatter.add_patch(ellipse)

        ax_scatter.set_xlim(0.38,1.02)
        ax_scatter.set_ylim(0.38,1.02)

        ax_scatter.grid(True,alpha=0.4)

        ax_scatter.plot([0,1],[0,1],"k:",alpha=0.3)

        # Add column title only for first row
        if row_idx == 0:
            ax_scatter.set_title(
                "Calibration vs Discrimination",
                fontsize=18 + font_addition,
                weight="bold",
                pad=20
            )
        
        # Add black border for "All Datasets" row
        if dataset_type is None:
            for spine in ax_scatter.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(3)
        
        # Add sticky note with dataset name in top-left corner
        sticky_note = FancyBboxPatch(
            (0.04, 0.90),
            0.18,
            0.05,
            boxstyle="round,pad=0.01",
            transform=ax_scatter.transAxes,
            facecolor="#E8E8E8",
            edgecolor="#A0A0A0",
            linewidth=2,
            alpha=0.9,
            zorder=1000
        )
        ax_scatter.add_patch(sticky_note)
        
        ax_scatter.text(
            0.13, 0.925,
            dataset_display_name,
            transform=ax_scatter.transAxes,
            fontsize=13 + font_addition,
            weight="bold",
            ha="center",
            va="center",
            zorder=1001
        )

        # Only show x-label and y-label on bottom row
        if row_idx == n_rows - 1:
            ax_scatter.set_xlabel("Calibration Quality (1 − ECE)", fontsize=14 + font_addition, fontweight='bold')
            ax_scatter.set_ylabel("Discrimination (AUROC)", fontsize=14 + font_addition, fontweight='bold')
        else:
            ax_scatter.set_xlabel("")
            ax_scatter.set_ylabel("")
        
        # Apply font size to tick labels
        ax_scatter.tick_params(labelsize=13 + font_addition)
        for label in ax_scatter.get_xticklabels():
            label.set_fontsize(13 + font_addition)
        for label in ax_scatter.get_yticklabels():
            label.set_fontsize(13 + font_addition)

        # =====================================================================
        # RIGHT: Calibration curve
        # =====================================================================

        n_bins = 10
        methods_calibration = {}

        for method_name in METHODS_TO_SHOW:

            all_conf = []
            all_corr = []

            for llm_id in llm_ids:
                for dt in row_dataset_types:

                    if method_name not in llm2method2dataset2results.get(llm_id,{}):
                        continue

                    if dt not in llm2method2dataset2results[llm_id].get(method_name,{}):
                        continue

                    results = llm2method2dataset2results[llm_id][method_name][dt]

                    shared_ids = llm2dataset2sharedRIDacrossMethods.get(llm_id,{}).get(dt,set())

                    if shared_ids:
                        filtered=[r for r in results if r["record_id"] in shared_ids]
                    else:
                        filtered=results

                    if filtered:
                        all_conf.extend([r["confidence_score"] for r in filtered])
                        all_corr.extend([r["ground_truth_correctness"] for r in filtered])

            if not all_conf:
                continue

            confidences=np.array(all_conf)
            correctness=np.array(all_corr)

            bins=np.linspace(0,1,n_bins+1)

            bin_idx=np.digitize(confidences,bins)-1
            bin_idx=np.clip(bin_idx,0,n_bins-1)

            bin_conf=[]
            bin_acc=[]
            bin_count=[]

            for b in range(n_bins):

                mask=(bin_idx==b)

                if mask.sum()>0:
                    bin_conf.append(confidences[mask].mean())
                    bin_acc.append(correctness[mask].mean())
                    bin_count.append(mask.sum())

            methods_calibration[method_name]={
                "conf":np.array(bin_conf),
                "acc":np.array(bin_acc),
                "count":np.array(bin_count)
            }

        ax_calib.plot([0,1],[0,1],"k--",linewidth=2.5,alpha=0.5)

        for method_name,data in methods_calibration.items():

            method_info = method2path.get(method_name,{})

            color=method_info.get("base_color","gray")
            shape=method_info.get("shape","o")

            x_vals=data["conf"]
            y_vals=data["acc"]
            counts=data["count"]

            lw=1+8*(counts/counts.max())

            for i in range(len(x_vals)-1):

                ax_calib.plot(
                    x_vals[i:i+2],
                    y_vals[i:i+2],
                    color=color,
                    linewidth=lw[i],
                    alpha=0.9,
                    solid_capstyle="round"
                )

            ax_calib.scatter(
                x_vals,
                y_vals,
                s=60,
                color=color,
                edgecolor="white",
                linewidth=1.5,
                marker=shape
            )

        ax_calib.set_xlim(-0.02,1.02)
        ax_calib.set_ylim(-0.02,1.02)

        ax_calib.grid(True,alpha=0.4)

        # Add column title only for first row
        if row_idx == 0:
            ax_calib.set_title(
                "Density-Weighted Calibration",
                fontsize=18 + font_addition,
                weight="bold",
                pad=20
            )
        
        # Add black border for "All Datasets" row
        if dataset_type is None:
            for spine in ax_calib.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(3)
        
        # Add sticky note with dataset name in bottom-right corner
        sticky_note = FancyBboxPatch(
            (0.78, 0.05),
            0.18,
            0.05,
            boxstyle="round,pad=0.01",
            transform=ax_calib.transAxes,
            facecolor="#E8E8E8",
            edgecolor="#A0A0A0",
            linewidth=2,
            alpha=0.9,
            zorder=1000
        )
        ax_calib.add_patch(sticky_note)
        
        ax_calib.text(
            0.87, 0.075,
            dataset_display_name,
            transform=ax_calib.transAxes,
            fontsize=13 + font_addition,
            weight="bold",
            ha="center",
            va="center",
            zorder=1001
        )

        # Only show x-label and y-label on bottom row
        if row_idx == n_rows - 1:
            ax_calib.set_xlabel("Predicted Confidence", fontsize=14 + font_addition, fontweight='bold')
            ax_calib.set_ylabel("Empirical Accuracy", fontsize=14 + font_addition, fontweight='bold')
        else:
            ax_calib.set_xlabel("")
            ax_calib.set_ylabel("")
        
        # Apply font size to tick labels
        ax_calib.tick_params(labelsize=13 + font_addition)
        for label in ax_calib.get_xticklabels():
            label.set_fontsize(13 + font_addition)
        for label in ax_calib.get_yticklabels():
            label.set_fontsize(13 + font_addition)

    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5,0.925),
        ncol=len(legend_labels),
        fontsize=13 + font_addition,
        frameon=True,
        fancybox=True,
        shadow=True
    )

    plt.tight_layout(rect=[0,0,1,0.97])

    plt.savefig(save_path,dpi=300,bbox_inches="tight")
    plt.savefig(str(save_path).replace(".png",".pdf"),dpi=300,bbox_inches="tight")
    plt.savefig(str(save_path).replace(".png",".svg"),bbox_inches="tight")

    print(f"✅ Saved combined figure to {save_path}")

    plt.show()

    return fig


# Example usage at the end of the script:
row1_save_path = OVERALL_RESULTS_PATH / "master_figure_row1_diagnostic_validity.png"

fig_row1 = create_row1_diagnostic_validity(
    df_metrics=df_metrics,
    llm2method2dataset2results=llm2method2dataset2results,
    llm2dataset2sharedRIDacrossMethods=llm2dataset2sharedRIDacrossMethods,
    method2path=method2path,
    llm_ids=llm_ids,
    dataset_types=dataset_types,
    dataset_display_names=dataset_display_names,
    save_path=row1_save_path,
    font_addition=4
)


In [ ]:
# ==============================================================================
# ROW 2: CLINICAL UTILITY - Triage Performance & Cost of Reliability
# ==============================================================================
# Uses: auroc_ece_coverage curves (ALL DATASETS) + cost_of_reliability_all_datasets.png
# Message: CCPS enables safe automation while baselines collapse on hard tasks

methods = METHODS_TO_SHOW

def create_row2_clinical_utility(
    aggregated_curves_contextual,
    aggregated_ece_curves_contextual,
    aggregated_curves_synthesis,
    aggregated_ece_curves_synthesis,
    aggregated_curves_clinical,
    aggregated_ece_curves_clinical,
    risk_coverage_data_list,
    method2path,
    method_styles,
    save_path,
    font_size_adjust=0
):
    """
    Row 2: Clinical Utility - Creates separate figures for each dataset
    Each figure: Left: AUROC @ Coverage, Middle: ECE @ Coverage, Right: Cost of Reliability
    """
    print("🎨 Creating Row 2: Clinical Utility (separate figures for each dataset)...")
    
    # Map dataset types to their aggregated curves and risk coverage data
    dataset_configs = [
        {
            "dataset_type": "contextual",
            "display_name": "CORAL-MCQA-Contextual",
            "aggregated_auroc": aggregated_curves_contextual,
            "aggregated_ece": aggregated_ece_curves_contextual,
            "risk_coverage_data": risk_coverage_data_list[0] if len(risk_coverage_data_list) > 0 else None
        },
        {
            "dataset_type": "synthesis",
            "display_name": "CORAL-MCQA-Synthesis",
            "aggregated_auroc": aggregated_curves_synthesis,
            "aggregated_ece": aggregated_ece_curves_synthesis,
            "risk_coverage_data": risk_coverage_data_list[1] if len(risk_coverage_data_list) > 1 else None
        },
        {
            "dataset_type": "clinical_inference",
            "display_name": "CORAL-MCQA-Clinical-Inference",
            "aggregated_auroc": aggregated_curves_clinical,
            "aggregated_ece": aggregated_ece_curves_clinical,
            "risk_coverage_data": risk_coverage_data_list[2] if len(risk_coverage_data_list) > 2 else None
        },
        {
            "dataset_type": None,
            "display_name": "All Datasets (Merged)",
            "aggregated_auroc": None,  # Will combine all
            "aggregated_ece": None,  # Will combine all
            "risk_coverage_data": None  # Will combine all
        }
    ]
    
    # Helper function to create a single figure for a dataset
    def create_single_figure(config, save_path_single):
        """Create a single figure for a specific dataset."""
        dataset_type = config["dataset_type"]
        dataset_display_name = config["display_name"]
        
        # Determine which aggregated curves to use
        if dataset_type is None:
            # Merged: combine all datasets
            all_aggregated_auroc = [aggregated_curves_contextual, aggregated_curves_synthesis, aggregated_curves_clinical]
            all_aggregated_ece = [aggregated_ece_curves_contextual, aggregated_ece_curves_synthesis, aggregated_ece_curves_clinical]
            
            # Combine AUROC curves across all datasets
            combined_auroc_curves = {}
            for method_name in ['CCPS', 'PIK', 'PTRUE', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F']:
                all_auroc_scores = []
                workloads = None
                
                for agg_curves in all_aggregated_auroc:
                    if method_name in agg_curves:
                        all_auroc_scores.append(agg_curves[method_name]['mean_auroc_scores'])
                        if workloads is None:
                            workloads = agg_curves[method_name]['workloads']
                
                if all_auroc_scores and workloads is not None:
                    combined_auroc_curves[method_name] = {
                        'workloads': workloads,
                        'mean_auroc_scores': np.mean(all_auroc_scores, axis=0),
                        'std_auroc_scores': np.std(all_auroc_scores, axis=0)
                    }
            
            # Combine ECE curves across all datasets
            combined_ece_curves = {}
            for method_name in ['CCPS', 'PIK', 'PTRUE', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F']:
                all_ece_scores = []
                workloads = None
                
                for agg_curves in all_aggregated_ece:
                    if method_name in agg_curves:
                        all_ece_scores.append(agg_curves[method_name]['mean_ece'])
                        if workloads is None:
                            workloads = agg_curves[method_name]['workloads']
                
                if all_ece_scores and workloads is not None:
                    combined_ece_curves[method_name] = {
                        'workloads': workloads,
                        'mean_ece': np.mean(all_ece_scores, axis=0),
                        'std_ece': np.std(all_ece_scores, axis=0)
                    }
            
            # For merged, use all risk coverage data
            risk_data_for_cost = risk_coverage_data_list
        else:
            # Single dataset: use its specific curves
            combined_auroc_curves = config["aggregated_auroc"]
            combined_ece_curves = config["aggregated_ece"]
            # For single dataset, use only its risk coverage data
            risk_data_for_cost = [config["risk_coverage_data"]] if config["risk_coverage_data"] is not None else []
        
        # Create figure
        fig = plt.figure(figsize=(20, 5.5))
        gs = fig.add_gridspec(1, 3, width_ratios=[1.2, 1, 1], wspace=0.22)
        
        ax_cost = fig.add_subplot(gs[0])
        ax_auroc = fig.add_subplot(gs[1])
        ax_ece = fig.add_subplot(gs[2])
        
        # =========================================================================
        # LEFT: Cost of Reliability
        # =========================================================================
        
        def get_workload_for_target(risk_data, target_value, method_name):
            workloads = risk_data[method_name]['human_workloads']
            scores = risk_data[method_name]['system_auroc_scores']
            valid_indices = [i for i, score in enumerate(scores) if score >= target_value]
            if not valid_indices:
                return 100.0
            return workloads[valid_indices[0]]
        
        targets = [0.65, 0.70, 0.75, 0.80]
        
        # Collect data points from the relevant dataset(s)
        plot_data = []
        for method in METHODS_TO_SHOW:
            for target in targets:
                efforts = []
                # Iterate through each dataset in risk_data_for_cost
                for risk_data in risk_data_for_cost:
                    if risk_data is None:
                        continue
                    # Iterate through each LLM in this dataset
                    for llm, llm_data in risk_data.items():
                        if method in llm_data:
                            effort = get_workload_for_target(llm_data, target, method)
                            efforts.append(effort)
                
                if efforts:
                    plot_data.append({
                        'Method': method,
                        'Target': f"≥ {target}",
                        'Effort': np.mean(efforts),
                        'std': np.std(efforts)
                    })
        
        df_cost = pd.DataFrame(plot_data)
        
        method_colors_bar = {m: method2path[m]['base_color'] for m in methods}
        method_labels_bar = {m: method2path[m]['method_name'] for m in methods}
        
        x = np.arange(len(targets))
        width = 0.25
        
        for i, method in enumerate(methods):
            method_df = df_cost[df_cost['Method'] == method]
            if len(method_df) > 0:
                bars = ax_cost.bar(x + i*width - width, method_df['Effort'], width, 
                                  label=method_labels_bar[method], 
                                  color=method_colors_bar[method],
                                  edgecolor='black', linewidth=0.5)
                
                # Add error bars
                for j, (bar, (_, row)) in enumerate(zip(bars, method_df.iterrows())):
                    height = bar.get_height()
                    std = row['std']
                    x_center = bar.get_x() + bar.get_width() / 2
                    
                    ax_cost.plot([x_center, x_center], 
                                [max(0, height - std), min(100, height + std)],
                                color='black', linewidth=1.5)
        
        ax_cost.set_xlabel('Target AUROC Threshold', fontsize=13 + font_size_adjust, fontweight='bold')
        ax_cost.set_ylabel('← Human Review Required (%)', fontsize=13 + font_size_adjust, fontweight='bold', labelpad=5)
        ax_cost.set_title(f'(C) Cost of Reliability - {dataset_display_name}', 
                         fontsize=14 + font_size_adjust, fontweight='bold', pad=10)
        ax_cost.set_xticks(x)
        ax_cost.set_xticklabels([f"≥{t}" for t in targets])
        ax_cost.set_ylim(0, 115)
        ax_cost.tick_params(labelsize=14 + font_size_adjust)
        # Remove individual legend (will add shared one)
        if ax_cost.get_legend():
            ax_cost.get_legend().remove()
        ax_cost.grid(axis='y', alpha=0.3)
        
        # =========================================================================
        # MIDDLE: AUROC @ Coverage
        # =========================================================================
        
        for method_name in METHODS_TO_SHOW:
            if method_name in combined_auroc_curves:
                data = combined_auroc_curves[method_name]
                style = method_styles[method_name]
                
                ax_auroc.plot(data['workloads'], data['mean_auroc_scores'], 
                             color=style['color'], linestyle=style['linestyle'], 
                             linewidth=style['linewidth'] * 1.8, label=style['label'], alpha=0.9)
                
                ax_auroc.fill_between(data['workloads'], 
                                      data['mean_auroc_scores'] - data['std_auroc_scores'],
                                      data['mean_auroc_scores'] + data['std_auroc_scores'],
                                      color=style['color'], alpha=0.2)
        
        # Add reference lines
        ax_auroc.axhline(y=0.50, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, label='Random (0.50)')
        
        ax_auroc.set_xlabel('Human Review Rate (%) →', fontsize=13 + font_size_adjust, fontweight='bold')
        ax_auroc.set_ylabel('AUROC of AI-Handled Cases →', fontsize=13 + font_size_adjust, fontweight='bold')
        ax_auroc.set_title(f'(D) Discrimination Under Triage - {dataset_display_name}', 
                          fontsize=14 + font_size_adjust, fontweight='bold', pad=10)
        ax_auroc.grid(True, alpha=0.3)
        ax_auroc.set_xlim(0, 100)
        ax_auroc.set_ylim(0.45, 1.0)
        ax_auroc.tick_params(labelsize=14 + font_size_adjust)
        # Remove individual legend (will add shared one)
        if ax_auroc.get_legend():
            ax_auroc.get_legend().remove()
        
        # =========================================================================
        # RIGHT: ECE @ Coverage
        # =========================================================================
        
        for method_name in METHODS_TO_SHOW:
            if method_name in combined_ece_curves:
                data = combined_ece_curves[method_name]
                style = method_styles[method_name]
                
                ax_ece.plot(data['workloads'], 1 - data['mean_ece'], 
                           color=style['color'], linestyle=style['linestyle'], 
                           linewidth=style['linewidth'] * 1.8, label=style['label'], alpha=0.9)
                
                ax_ece.fill_between(data['workloads'], 
                                    1 - (data['mean_ece'] + data['std_ece']),
                                    1 - (data['mean_ece'] - data['std_ece']),
                                    color=style['color'], alpha=0.2)
        
        ax_ece.set_xlabel('Human Review Rate (%) →', fontsize=13 + font_size_adjust, fontweight='bold')
        ax_ece.set_ylabel('Calibration Quality →', fontsize=13 + font_size_adjust, fontweight='bold')
        ax_ece.set_title(f'(E) Calibration Under Triage - {dataset_display_name}', 
                        fontsize=14 + font_size_adjust, fontweight='bold', pad=10)
        ax_ece.grid(True, alpha=0.3)
        ax_ece.set_xlim(0, 100)
        ax_ece.set_ylim(0.5, 1.0)
        ax_ece.tick_params(labelsize=14 + font_size_adjust)
        # Remove individual legend (will add shared one)
        if ax_ece.get_legend():
            ax_ece.get_legend().remove()
        
        # =========================================================================
        # SHARED LEGEND AT TOP CENTER
        # =========================================================================
        
        # Create legend handles manually
        
        legend_handles = [
            plt.Rectangle((0, 0), 1, 1, color=method2path[m]['base_color'], label=method2path[m]['method_name'])
            for m in METHODS_TO_SHOW
        ]
        legend_labels = [method2path[m]['method_name'] for m in METHODS_TO_SHOW]
        
        fig.legend(legend_handles, legend_labels, 
                   loc='upper center', 
                   bbox_to_anchor=(0.5, 1.05), 
                   ncol=len(METHODS_TO_SHOW), 
                   fontsize=13 + font_size_adjust, 
                   frameon=True, 
                   fancybox=True, 
                   shadow=True,
                   columnspacing=1.5,
                   handletextpad=0.8)
        
        plt.tight_layout(rect=[0, 0, 1, 0.92])
        plt.savefig(save_path_single, dpi=300, bbox_inches='tight')
        plt.savefig(str(save_path_single).replace('.png', '.pdf'), dpi=300, bbox_inches='tight')
        print(f"✅ Saved figure for {dataset_display_name} to: {save_path_single}")
        plt.show()
        return fig
    
    # Create separate figures for each dataset
    figures = []
    base_path = Path(save_path)
    
    for config in dataset_configs:
        dataset_type = config["dataset_type"]
        dataset_display_name = config["display_name"]
        
        # Create filename based on dataset
        if dataset_type is None:
            filename_suffix = "merged"
        else:
            filename_suffix = dataset_type
        
        save_path_single = base_path.parent / f"{base_path.stem}_{filename_suffix}{base_path.suffix}"
        fig = create_single_figure(config, save_path_single)
        figures.append(fig)
    
    # Return the last figure (for compatibility)
    return figures[-1] if figures else None

# Prepare data for Row 2
risk_data_list = []
if 'risk_coverage_data_contextual' in dir():
    risk_data_list.append(risk_coverage_data_contextual)
if 'risk_coverage_data' in dir():
    risk_data_list.append(risk_coverage_data)
if 'risk_coverage_data_clinical' in dir():
    risk_data_list.append(risk_coverage_data_clinical)

# Generate Row 2
row2_save_path = OVERALL_RESULTS_PATH / "master_figure_row2_clinical_utility.png"
fig_row2 = create_row2_clinical_utility(
    aggregated_curves_contextual=aggregated_curves_contextual,
    aggregated_ece_curves_contextual=aggregated_ece_curves_contextual,
    aggregated_curves_synthesis=aggregated_curves,
    aggregated_ece_curves_synthesis=aggregated_ece_curves,
    aggregated_curves_clinical=aggregated_curves_clinical,
    aggregated_ece_curves_clinical=aggregated_ece_curves_clinical,
    risk_coverage_data_list=risk_data_list,
    method2path=method2path,
    method_styles=method_styles,
    save_path=row2_save_path
)

In [ ]:
# ==============================================================================
# ROW 2: CLINICAL UTILITY - Cost of Reliability (Bar Plots Only)
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import matplotlib.colors as mc
import colorsys

def soften_color(color, saturation_scale=0.6, value_scale=1.05):
    """
    Reduce saturation to make colors less intense while keeping the same hue.
    """
    r, g, b = mc.to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    s *= saturation_scale
    l = min(1, l * value_scale)
    return colorsys.hls_to_rgb(h, l, s)

def create_row2_cost_of_reliability_only(
    risk_coverage_data_contextual,
    risk_coverage_data_synthesis,
    risk_coverage_data_clinical,
    method2path,
    save_path,
    font_size_adjust=0
):

    print("🎨 Creating Row 2: Cost of Reliability (Bar Plots Only)...")
    
    dataset_configs = [
        {"dataset_type": "contextual","display_name": "Contextual","risk_data": risk_coverage_data_contextual},
        {"dataset_type": "synthesis","display_name": "Synthesis","risk_data": risk_coverage_data_synthesis},
        {"dataset_type": "clinical_inference","display_name": "Inference","risk_data": risk_coverage_data_clinical},
        {"dataset_type": "merged","display_name": "Merged","risk_data": None}
    ]
    
    fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharey=True)
    
    def get_workload_for_target(risk_data, target_value, method_name):
        workloads = risk_data[method_name]['human_workloads']
        scores = risk_data[method_name]['system_auroc_scores']
        valid_indices = [i for i, score in enumerate(scores) if score >= target_value]
        if not valid_indices:
            return 100.0
        return workloads[valid_indices[0]]
    
    targets = [0.65, 0.70, 0.75, 0.80]
    methods = METHODS_TO_SHOW
    
    method_colors = {
        m: soften_color(method2path[m]['base_color'], saturation_scale=0.8, value_scale=1.05)
        for m in methods
    }
    method_labels = {m: method2path[m]['method_name'] for m in methods}

    width = 0.18
    x = np.arange(len(targets))

    for idx, config in enumerate(dataset_configs):

        ax = axes[idx]
        dataset_type = config["dataset_type"]
        dataset_display_name = config["display_name"]

        plot_data = []

        if dataset_type == "merged":

            all_risk_data = [
                risk_coverage_data_contextual,
                risk_coverage_data_synthesis,
                risk_coverage_data_clinical
            ]
            
            for method in methods:
                for target in targets:

                    efforts = []

                    for risk_data in all_risk_data:
                        for llm, llm_data in risk_data.items():
                            if method in llm_data:
                                effort = get_workload_for_target(llm_data, target, method)
                                efforts.append(effort)

                    if efforts:
                        plot_data.append({
                            "Method": method,
                            "Target": f"≥ {target}",
                            "Effort": np.mean(efforts),
                            "std": np.std(efforts)
                        })

        else:

            risk_data = config["risk_data"]

            for method in methods:
                for target in targets:

                    efforts = []

                    for llm, llm_data in risk_data.items():
                        if method in llm_data:
                            effort = get_workload_for_target(llm_data, target, method)
                            efforts.append(effort)

                    if efforts:
                        plot_data.append({
                            "Method": method,
                            "Target": f"≥ {target}",
                            "Effort": np.mean(efforts),
                            "std": np.std(efforts)
                        })

        df_cost = pd.DataFrame(plot_data)

        for i, method in enumerate(methods):

            method_df = df_cost[df_cost["Method"] == method]

            if len(method_df) == 0:
                continue

            offset = (i - (len(methods)-1)/2) * width

            ax.bar(
                x + offset,
                method_df["Effort"],
                width,
                yerr=method_df["std"],
                capsize=4,
                label=method_labels[method],
                color=method_colors[method],
                edgecolor="black",
                linewidth=0.4,
                error_kw={"elinewidth":1.2}
            )

        ax.set_title(
            dataset_display_name,
            fontsize=16 + font_size_adjust,
            fontweight="bold",
            pad=15
        )

        ax.set_xticks(x)
        ax.set_xticklabels(
            [f"≥ {t}" for t in targets],
            fontsize=14 + font_size_adjust
        )

        ax.set_ylim(0,115)
        ax.tick_params(axis="y", labelsize=14 + font_size_adjust)

        ax.set_xlim(-0.5, len(targets)-0.5)

        ax.grid(axis="y", alpha=0.3)

        if dataset_type == "merged":
            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(3)

        if idx == 0:
            ax.set_ylabel(
                "← Human Review Required (%)",
                fontsize=16 + font_size_adjust,
                fontweight="bold"
            )

    fig.text(
        0.5,
        0.02,
        "Target AUROC Threshold",
        ha="center",
        fontsize=16 + font_size_adjust,
        fontweight="bold"
    )

    legend_handles = [
        plt.Rectangle((0,0),1,1,color=method_colors[m],label=method_labels[m])
        for m in methods
    ]

    legend_labels = [method_labels[m] for m in methods]

    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5,1.05),
        ncol=len(methods),
        fontsize=13 + font_size_adjust,
        frameon=True,
        fancybox=True,
        shadow=True,
        columnspacing=1.5,
        handletextpad=0.8
    )

    plt.tight_layout(rect=[0,0.05,1,0.95])

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.savefig(str(save_path).replace(".png",".pdf"), dpi=300, bbox_inches="tight")
    plt.savefig(str(save_path).replace(".png",".svg"), bbox_inches="tight")

    print(f"✅ Saved Cost of Reliability figure to: {save_path}")

    plt.show()

    return fig


row2_cost_save_path = OVERALL_RESULTS_PATH / "master_figure_row2_cost_of_reliability.png"

fig_row2_cost = create_row2_cost_of_reliability_only(
    risk_coverage_data_contextual=risk_coverage_data_contextual,
    risk_coverage_data_synthesis=risk_coverage_data,
    risk_coverage_data_clinical=risk_coverage_data_clinical,
    method2path=method2path,
    save_path=row2_cost_save_path,
    font_size_adjust=4
)

In [ ]:
# ==============================================================================
# TABLE: Cost of Reliability - Human Review Required (%) by Dataset and Target AUROC
# ==============================================================================

import numpy as np
import pandas as pd
from scipy import stats

print("=" * 120)
print("TABLE 2a: Cost of Reliability - Mean ± Standard Deviation")
print("Note: Results averaged across 5 LLMs (n=5 per method/dataset/target)")
print("=" * 120)

def get_workload_for_target(risk_data, target_value, method_name):
    """Get the minimum workload needed to achieve target AUROC."""
    workloads = risk_data[method_name]['human_workloads']
    scores = risk_data[method_name]['system_auroc_scores']
    valid_indices = [i for i, score in enumerate(scores) if score >= target_value]
    if not valid_indices:
        return 100.0
    return workloads[valid_indices[0]]

# Dataset configurations
dataset_configs = [
    {
        "dataset_type": "contextual",
        "display_name": "Contextual",
        "risk_data": risk_coverage_data_contextual
    },
    {
        "dataset_type": "synthesis",
        "display_name": "Synthesis",
        "risk_data": risk_coverage_data
    },
    {
        "dataset_type": "clinical_inference",
        "display_name": "Inference",
        "risk_data": risk_coverage_data_clinical
    },
    {
        "dataset_type": "merged",
        "display_name": "Merged",
        "risk_data": None  # Will combine all three
    }
]

targets = [0.65, 0.70, 0.75, 0.80, 0.85]
methods = METHODS_TO_SHOW
method_labels = {m: method2path[m]['method_name'] for m in methods}

# Collect all data for mean/std table
table_rows_mean = []
# Collect all data for median/IQR table
table_rows_median = []
# Collect all data for mean (range) table
table_rows_range = []

for config in dataset_configs:
    dataset_type = config["dataset_type"]
    dataset_display_name = config["display_name"]
    
    # Process each target threshold
    for target_idx, target in enumerate(targets):
        row_data_mean = {}
        row_data_median = {}
        row_data_range = {}
        
        if dataset_type == "merged":
            # Merged: combine all three datasets
            all_risk_data = [
                risk_coverage_data_contextual,
                risk_coverage_data,
                risk_coverage_data_clinical
            ]
            
            for method in methods:
                efforts = []
                for risk_data in all_risk_data:
                    for llm, llm_data in risk_data.items():
                        if method in llm_data:
                            effort = get_workload_for_target(llm_data, target, method)
                            efforts.append(effort)
                
                if efforts:
                    # Mean ± Std
                    mean_effort = np.mean(efforts)
                    std_effort = np.std(efforts)
                    row_data_mean[method_labels[method]] = f"{mean_effort:.0f} ± {std_effort:.0f}"
                    
                    # Median [Q1-Q3]
                    median_effort = np.median(efforts)
                    q1 = np.percentile(efforts, 25)
                    q3 = np.percentile(efforts, 75)
                    row_data_median[method_labels[method]] = f"{median_effort:.0f} [{q1:.0f}-{q3:.0f}]"
                    
                    # Mean (95% CI) - clamped to valid percentage range [0, 100]
                    n = len(efforts)
                    se = std_effort / np.sqrt(n) if n > 1 else 0
                    # Use t-distribution for CI (more accurate for small samples)
                    if n > 1:
                        t_value = stats.t.ppf(0.975, df=n-1)  # 95% CI, two-tailed
                    else:
                        t_value = 1.96  # Fallback for n=1
                    ci_lower = max(0.0, mean_effort - t_value * se)  # Clamp to 0
                    ci_upper = min(100.0, mean_effort + t_value * se)  # Clamp to 100
                    row_data_range[method_labels[method]] = f"{mean_effort:.0f} ({ci_lower:.0f} to {ci_upper:.0f})"
                else:
                    row_data_mean[method_labels[method]] = "N/A"
                    row_data_median[method_labels[method]] = "N/A"
                    row_data_range[method_labels[method]] = "N/A"
        else:
            # Single dataset
            risk_data = config["risk_data"]
            
            for method in methods:
                efforts = []
                for llm, llm_data in risk_data.items():
                    if method in llm_data:
                        effort = get_workload_for_target(llm_data, target, method)
                        efforts.append(effort)
                
                if efforts:
                    # Mean ± Std
                    mean_effort = np.mean(efforts)
                    std_effort = np.std(efforts)
                    row_data_mean[method_labels[method]] = f"{mean_effort:.0f} ± {std_effort:.0f}"
                    
                    # Median [Q1-Q3]
                    median_effort = np.median(efforts)
                    q1 = np.percentile(efforts, 25)
                    q3 = np.percentile(efforts, 75)
                    row_data_median[method_labels[method]] = f"{median_effort:.0f} [{q1:.0f}-{q3:.0f}]"
                    
                    # Mean (95% CI) - clamped to valid percentage range [0, 100]
                    n = len(efforts)
                    se = std_effort / np.sqrt(n) if n > 1 else 0
                    # Use t-distribution for CI (more accurate for small samples)
                    if n > 1:
                        t_value = stats.t.ppf(0.975, df=n-1)  # 95% CI, two-tailed
                    else:
                        t_value = 1.96  # Fallback for n=1
                    ci_lower = max(0.0, mean_effort - t_value * se)  # Clamp to 0
                    ci_upper = min(100.0, mean_effort + t_value * se)  # Clamp to 100
                    row_data_range[method_labels[method]] = f"{mean_effort:.0f} ({ci_lower:.0f} to {ci_upper:.0f})"
                else:
                    row_data_mean[method_labels[method]] = "N/A"
                    row_data_median[method_labels[method]] = "N/A"
                    row_data_range[method_labels[method]] = "N/A"
        
        # Create row label
        if target_idx == 0:
            row_label = dataset_display_name
        else:
            row_label = ""
        
        # Reorder columns: Dataset, Target AUROC, then methods (for mean table)
        ordered_row_mean = {'Dataset': row_label, 'Target AUROC': f"≥ {target}"}
        for method in methods:
            ordered_row_mean[method_labels[method]] = row_data_mean[method_labels[method]]
        table_rows_mean.append(ordered_row_mean)
        
        # Reorder columns: Dataset, Target AUROC, then methods (for median table)
        ordered_row_median = {'Dataset': row_label, 'Target AUROC': f"≥ {target}"}
        for method in methods:
            ordered_row_median[method_labels[method]] = row_data_median[method_labels[method]]
        table_rows_median.append(ordered_row_median)
        
        # Reorder columns: Dataset, Target AUROC, then methods (for range table)
        ordered_row_range = {'Dataset': row_label, 'Target AUROC': f"≥ {target}"}
        for method in methods:
            ordered_row_range[method_labels[method]] = row_data_range[method_labels[method]]
        table_rows_range.append(ordered_row_range)

# Create DataFrames
df_table_mean = pd.DataFrame(table_rows_mean)
df_table_mean = df_table_mean.set_index(['Dataset', 'Target AUROC'])

df_table_median = pd.DataFrame(table_rows_median)
df_table_median = df_table_median.set_index(['Dataset', 'Target AUROC'])

df_table_range = pd.DataFrame(table_rows_range)
df_table_range = df_table_range.set_index(['Dataset', 'Target AUROC'])

# Print mean table
print("\n")
print(df_table_mean.to_string())
print("\n")

# Save mean table to CSV
csv_path_mean = OVERALL_RESULTS_PATH / "table_cost_of_reliability_mean_std.csv"
df_table_mean.to_csv(csv_path_mean)
print(f"✅ Saved mean/std table to {csv_path_mean}")

print("\n\n")
print("=" * 120)
print("TABLE 2b: Cost of Reliability - Median [Q1-Q3]")
print("Note: Results across 5 LLMs (n=5 per method/dataset/target). Recommended for skewed data.")
print("=" * 120)

# Print median table
print("\n")
print(df_table_median.to_string())
print("\n")

# Save median table to CSV
csv_path_median = OVERALL_RESULTS_PATH / "table_cost_of_reliability_median_iqr.csv"
df_table_median.to_csv(csv_path_median)
print(f"✅ Saved median/IQR table to {csv_path_median}")

print("\n\n")
print("=" * 120)
print("TABLE 2c: Cost of Reliability - Mean (95% CI)")
print("Note: Results across 5 LLMs (n=5 per method/dataset/target). CI clamped to [0, 100].")
print("=" * 120)

# Print range table
print("\n")
print(df_table_range.to_string())
print("\n")

# Save range table to CSV
csv_path_range = OVERALL_RESULTS_PATH / "table_cost_of_reliability_mean_95ci.csv"
df_table_range.to_csv(csv_path_range)
print(f"✅ Saved mean/95% CI table to {csv_path_range}")

print("\n✅ Tables generated successfully")


In [ ]:
# ==============================================================================
# ROW 3: TECHNICAL ROBUSTNESS - Cross-Model Reliability (ALL DATASETS)
# ==============================================================================
# Uses: option1_robustness_boxplots logic with 4 bins
# Message: CCPS is consistently reliable across different LLM families

from matplotlib.ticker import NullLocator

def create_row3_technical_robustness(
    risk_coverage_data_contextual,
    ece_coverage_data_contextual,
    risk_coverage_data_synthesis,
    ece_coverage_data_synthesis,
    risk_coverage_data_clinical,
    ece_coverage_data_clinical,
    method2path,
    save_path,
    font_size_adjust=0
):
    """
    Row 3: Technical Robustness - Creates separate figures for each dataset
    Each figure: Boxplots showing AUROC and ECE distributions across LLMs at different workload bins
    """
    print("🎨 Creating Row 3: Technical Robustness (separate figures for each dataset)...")
    
    dataset_configs = [
        {
            "dataset_type": "contextual",
            "display_name": "CORAL-MCQA-Contextual",
            "risk_coverage_data": risk_coverage_data_contextual,
            "ece_coverage_data": ece_coverage_data_contextual
        },
        {
            "dataset_type": "synthesis",
            "display_name": "CORAL-MCQA-Synthesis",
            "risk_coverage_data": risk_coverage_data_synthesis,
            "ece_coverage_data": ece_coverage_data_synthesis
        },
        {
            "dataset_type": "clinical_inference",
            "display_name": "CORAL-MCQA-Clinical-Inference",
            "risk_coverage_data": risk_coverage_data_clinical,
            "ece_coverage_data": ece_coverage_data_clinical
        },
        {
            "dataset_type": None,
            "display_name": "All Datasets (Merged)",
            "risk_coverage_data": None,
            "ece_coverage_data": None
        }
    ]
    
    def create_single_figure(config, save_path_single):
        dataset_type = config["dataset_type"]
        dataset_display_name_full = config["display_name"]
        dataset_display_name = dataset_display_name_full.split("-")[-1] if "-" in dataset_display_name_full else dataset_display_name_full
        
        workload_bins = [
            (0, 25, "0-25%", "AI Reviews Most"),
            (25, 50, "25-50%", "AI Reviews Many"),
            (50, 75, "50-75%", "Humans Review Many"),
            (75, 100, "75-100%", "Humans Review Most")
        ]
        bin_labels_numeric = [label for _, _, label, _ in workload_bins]
        bin_labels_text = [text for _, _, _, text in workload_bins]
        bin_labels_original = [f"{text}\n({num})" for _, _, num, text in workload_bins]
        
        methods = METHODS_TO_SHOW
        method_colors = {m: method2path[m]['base_color'] for m in methods}
        method_labels = {m: method2path[m]['method_name'] for m in methods}
        
        auroc_data = {b: {m: [] for m in methods} for b in bin_labels_original}
        ece_data_bins = {b: {m: [] for m in methods} for b in bin_labels_original}
        
        if dataset_type is None:
            all_datasets = [
                (risk_coverage_data_contextual, ece_coverage_data_contextual),
                (risk_coverage_data_synthesis, ece_coverage_data_synthesis),
                (risk_coverage_data_clinical, ece_coverage_data_clinical)
            ]
        else:
            all_datasets = [(config["risk_coverage_data"], config["ece_coverage_data"])]
        
        for risk_data, ece_data in all_datasets:
            if risk_data is None or ece_data is None:
                continue
            for llm_display in risk_data:
                for method in methods:
                    if method not in risk_data[llm_display] or method not in ece_data[llm_display]:
                        continue
                    
                    auroc_vals = risk_data[llm_display][method]['system_auroc_scores']
                    ece_vals = ece_data[llm_display][method]['ece_values']
                    workloads = risk_data[llm_display][method]['human_workloads']
                    
                    for start, end, num_label, text_label in workload_bins:
                        label = f"{text_label}\n({num_label})"
                        idx = [i for i, w in enumerate(workloads) if start <= w < end]
                        if idx:
                            auroc_data[label][method].append(np.mean([auroc_vals[i] for i in idx]))
                            ece_data_bins[label][method].append(np.mean([ece_vals[i] for i in idx]))
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        label_mapping = {orig: num for orig, num in zip(bin_labels_original, bin_labels_numeric)}
        
        df_auroc = pd.DataFrame([
            {'Workload Bin': label_mapping[b], 'Method': m, 'AUROC': v}
            for b in bin_labels_original for m in methods for v in auroc_data[b][m]
        ])
        
        if len(df_auroc) > 0:
            sns.boxplot(
                data=df_auroc,
                x='Workload Bin',
                y='AUROC',
                hue='Method',
                palette=method_colors,
                order=bin_labels_numeric,
                hue_order=methods,
                linewidth=1.5,
                ax=ax1
            )
        
        # ---- HARD FIX FOR EXTRA VERTICAL LINES ----
        ax1.grid(False)
        ax1.grid(False, axis='x')
        ax1.xaxis.set_minor_locator(NullLocator())
        ax1.tick_params(axis='x', which='minor', bottom=False)
        ax1.xaxis.grid(False)
        
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        
        ax1.set_xticks([i - 0.5 for i in range(len(bin_labels_numeric) + 1)])
        ax1.set_xticklabels(['0', '25', '50', '75', '100'])
        ax1.set_xlim(-0.5, len(bin_labels_numeric) - 0.5)
        
        ax1.set_axisbelow(True)
        ax1.yaxis.grid(True, linestyle='--', linewidth=0.8, alpha=0.6, color='gray')
        
        for i in range(len(bin_labels_numeric) + 1):
            ax1.axvline(x=i - 0.5, color='gray', linestyle='-', linewidth=1.0, alpha=0.3, zorder=0)
        
        # Top annotations without ticks (prevents vertical lines)
        ax1_twin = ax1.twiny()
        ax1_twin.set_xlim(ax1.get_xlim())
        
        ax1_twin.set_xticks([])        # no ticks at all
        ax1_twin.set_xticklabels([])   # no labels
        ax1_twin.tick_params(top=False, bottom=False)
        
        ax1_twin.spines['top'].set_visible(False)
        ax1_twin.spines['right'].set_visible(False)
        ax1_twin.spines['bottom'].set_visible(False)
        
        # Manually place text labels at bin centers
        for i, label in enumerate(bin_labels_text):
            ax1.text(
                i,
                1.02,
                label,
                transform=ax1.get_xaxis_transform(),
                ha='center',
                va='bottom',
                fontsize=11 + font_size_adjust
            )
        
        ax1.set_title(f'(F) AUROC Robustness across Models - {dataset_display_name}',
                      fontweight='bold', fontsize=14 + font_size_adjust, y=1.1)
        ax1.set_xlabel('Human Review Burden (%)', fontsize=12 + font_size_adjust, fontweight='bold')
        ax1.set_ylabel('AUROC', fontsize=12 + font_size_adjust, fontweight='bold')
        ax1.set_ylim(0.0, 1.0)
        ax1.set_yticks(np.arange(0.0, 1.01, 0.1))
        ax1.tick_params(labelsize=13 + font_size_adjust)
        
        if ax1.get_legend():
            ax1.get_legend().remove()
        
        df_ece = pd.DataFrame([
            {'Workload Bin': label_mapping[b], 'Method': m, 'ECE': v}
            for b in bin_labels_original for m in methods for v in ece_data_bins[b][m]
        ])
        
        if len(df_ece) > 0:
            df_ece['Calibration Quality'] = 1 - df_ece['ECE']
            sns.boxplot(
                data=df_ece,
                x='Workload Bin',
                y='Calibration Quality',
                hue='Method',
                palette=method_colors,
                order=bin_labels_numeric,
                hue_order=methods,
                linewidth=1.5,
                ax=ax2
            )
        
        # ---- SAME HARD FIX ON RIGHT AXIS ----
        ax2.grid(False)
        ax2.grid(False, axis='x')
        ax2.xaxis.set_minor_locator(NullLocator())
        ax2.tick_params(axis='x', which='minor', bottom=False)
        ax2.xaxis.grid(False)
        
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        
        ax2.set_xticks([i - 0.5 for i in range(len(bin_labels_numeric) + 1)])
        ax2.set_xticklabels(['0', '25', '50', '75', '100'])
        ax2.set_xlim(-0.5, len(bin_labels_numeric) - 0.5)
        
        ax2.set_axisbelow(True)
        ax2.yaxis.grid(True, linestyle='--', linewidth=0.8, alpha=0.6, color='gray')
        
        for i in range(len(bin_labels_numeric) + 1):
            ax2.axvline(x=i - 0.5, color='gray', linestyle='-', linewidth=1.0, alpha=0.3, zorder=0)
        
        # Top annotations without ticks (prevents vertical lines)
        ax2_twin = ax2.twiny()
        ax2_twin.set_xlim(ax2.get_xlim())
        
        ax2_twin.set_xticks([])        # no ticks at all
        ax2_twin.set_xticklabels([])   # no labels
        ax2_twin.tick_params(top=False, bottom=False)
        
        ax2_twin.spines['top'].set_visible(False)
        ax2_twin.spines['right'].set_visible(False)
        ax2_twin.spines['bottom'].set_visible(False)
        
        # Manually place text labels at bin centers
        for i, label in enumerate(bin_labels_text):
            ax2.text(
                i,
                1.02,
                label,
                transform=ax2.get_xaxis_transform(),
                ha='center',
                va='bottom',
                fontsize=11 + font_size_adjust
            )
        
        ax2.set_title(f'(G) Calibration Quality Robustness across Models - {dataset_display_name}',
                      fontweight='bold', fontsize=14 + font_size_adjust, y=1.1)
        ax2.set_xlabel('Human Review Burden (%)', fontsize=12 + font_size_adjust, fontweight='bold')
        ax2.set_ylabel('Calibration Quality', fontsize=12 + font_size_adjust, fontweight='bold')
        ax2.set_ylim(0.0, 1.0)
        ax2.set_yticks(np.arange(0.0, 1.01, 0.1))
        ax2.tick_params(labelsize=13 + font_size_adjust)
        
        if ax2.get_legend():
            ax2.get_legend().remove()
        
        legend_handles = [
            plt.Rectangle((0, 0), 1, 1, color=method_colors[m], label=method_labels[m])
            for m in METHODS_TO_SHOW
        ]
        
        fig.legend(
            legend_handles,
            [method_labels[m] for m in METHODS_TO_SHOW],
            loc='upper center',
            bbox_to_anchor=(0.5, 1),
            ncol=6,
            fontsize=13 + font_size_adjust,
            frameon=True,
            fancybox=True,
            shadow=True
        )
        
        plt.tight_layout(rect=[0, 0.15, 1, 0.92])
        plt.savefig(save_path_single, dpi=300, bbox_inches='tight')
        plt.savefig(str(save_path_single).replace('.png', '.pdf'), bbox_inches='tight')
        plt.show()
        
        return fig
    
    figures = []
    base_path = Path(save_path)
    
    for config in dataset_configs:
        suffix = "merged" if config["dataset_type"] is None else config["dataset_type"]
        save_path_single = base_path.parent / f"{base_path.stem}_{suffix}{base_path.suffix}"
        figures.append(create_single_figure(config, save_path_single))
    
    return figures[-1] if figures else None


# Generate Row 3 (ALL datasets combined)
row3_save_path = OVERALL_RESULTS_PATH / "master_figure_row3_technical_robustness.png"
fig_row3 = create_row3_technical_robustness(
    risk_coverage_data_contextual=risk_coverage_data_contextual,
    ece_coverage_data_contextual=ece_coverage_data_contextual,
    risk_coverage_data_synthesis=risk_coverage_data,
    ece_coverage_data_synthesis=ece_coverage_data,
    risk_coverage_data_clinical=risk_coverage_data_clinical,
    ece_coverage_data_clinical=ece_coverage_data_clinical,
    method2path=method2path,
    save_path=row3_save_path
)
# this needs a caption expanding what happens when you expand from left to right

In [ ]:
# ==============================================================================
# ROW 3: TECHNICAL ROBUSTNESS - Combined Figure (All Datasets Stacked)
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import NullLocator

def create_row3_technical_robustness_combined(
    risk_coverage_data_contextual,
    ece_coverage_data_contextual,
    risk_coverage_data_synthesis,
    ece_coverage_data_synthesis,
    risk_coverage_data_clinical,
    ece_coverage_data_clinical,
    method2path,
    save_path,
    font_addition=0
):
    """
    Row 3: Technical Robustness - Combined figure with all datasets stacked
    """
    print("🎨 Creating combined technical robustness figure...")
    
    dataset_configs = [
        {
            "dataset_type": "contextual",
            "display_name": "Contextual",
            "risk_coverage_data": risk_coverage_data_contextual,
            "ece_coverage_data": ece_coverage_data_contextual
        },
        {
            "dataset_type": "synthesis",
            "display_name": "Synthesis",
            "risk_coverage_data": risk_coverage_data_synthesis,
            "ece_coverage_data": ece_coverage_data_synthesis
        },
        {
            "dataset_type": "clinical_inference",
            "display_name": "Inference",
            "risk_coverage_data": risk_coverage_data_clinical,
            "ece_coverage_data": ece_coverage_data_clinical
        },
        {
            "dataset_type": None,
            "display_name": "Merged",
            "risk_coverage_data": None,
            "ece_coverage_data": None
        }
    ]
    
    n_rows = len(dataset_configs)
    
    fig = plt.figure(figsize=(20, 7 * n_rows))
    
    gs = fig.add_gridspec(
        n_rows,
        2,
        width_ratios=[1, 1],
        hspace=0.25,
        wspace=0.15
    )
    
    workload_bins = [
        (0, 25, "0-25%", "AI", "Reviews Most"),
        (25, 50, "25-50%", "AI", "Reviews Many"),
        (50, 75, "50-75%", "Human", "Reviews Many"),
        (75, 100, "75-100%", "Human", "Reviews Most")
    ]
    bin_labels_numeric = [label for _, _, label, _, _ in workload_bins]
    bin_labels_text = [f"{line1}\n{line2}" for _, _, _, line1, line2 in workload_bins]
    bin_labels_original = [f"{line1}\n{line2}\n({num})" for _, _, num, line1, line2 in workload_bins]
    
    methods = METHODS_TO_SHOW
    method_colors = {m: method2path[m]['base_color'] for m in methods}
    method_labels_dict = {m: method2path[m]['method_name'] for m in methods}
    
    legend_handles = []
    legend_labels = []
    
    for row_idx, config in enumerate(dataset_configs):
        dataset_type = config["dataset_type"]
        dataset_display_name = config["display_name"]
        
        ax_auroc = fig.add_subplot(gs[row_idx, 0])
        ax_calib = fig.add_subplot(gs[row_idx, 1])
        
        # Collect data for this dataset
        auroc_data = {b: {m: [] for m in methods} for b in bin_labels_original}
        ece_data_bins = {b: {m: [] for m in methods} for b in bin_labels_original}
        
        if dataset_type is None:
            # Merged: combine all datasets
            all_datasets = [
                (risk_coverage_data_contextual, ece_coverage_data_contextual),
                (risk_coverage_data_synthesis, ece_coverage_data_synthesis),
                (risk_coverage_data_clinical, ece_coverage_data_clinical)
            ]
        else:
            all_datasets = [(config["risk_coverage_data"], config["ece_coverage_data"])]
        
        for risk_data, ece_data in all_datasets:
            if risk_data is None or ece_data is None:
                continue
            for llm_display in risk_data:
                for method in methods:
                    if method not in risk_data[llm_display] or method not in ece_data[llm_display]:
                        continue
                    
                    auroc_vals = risk_data[llm_display][method]['system_auroc_scores']
                    ece_vals = ece_data[llm_display][method]['ece_values']
                    workloads = risk_data[llm_display][method]['human_workloads']
                    
                    for start, end, num_label, line1, line2 in workload_bins:
                        label = f"{line1}\n{line2}\n({num_label})"
                        idx = [i for i, w in enumerate(workloads) if start <= w < end]
                        if idx:
                            auroc_data[label][method].append(np.mean([auroc_vals[i] for i in idx]))
                            ece_data_bins[label][method].append(np.mean([ece_vals[i] for i in idx]))
        
        # =====================================================================
        # LEFT: AUROC Boxplots
        # =====================================================================
        
        label_mapping = {orig: num for orig, num in zip(bin_labels_original, bin_labels_numeric)}
        
        df_auroc = pd.DataFrame([
            {'Workload Bin': label_mapping[b], 'Method': m, 'AUROC': v}
            for b in bin_labels_original for m in methods for v in auroc_data[b][m]
        ])
        
        if len(df_auroc) > 0:
            sns.boxplot(
                data=df_auroc,
                x='Workload Bin',
                y='AUROC',
                hue='Method',
                palette=method_colors,
                order=bin_labels_numeric,
                hue_order=methods,
                linewidth=1.5,
                ax=ax_auroc
            )
        
        ax_auroc.grid(False)
        ax_auroc.grid(False, axis='x')
        ax_auroc.xaxis.set_minor_locator(NullLocator())
        ax_auroc.tick_params(axis='x', which='minor', bottom=False)
        ax_auroc.xaxis.grid(False)
        
        # Show all spines for Merged row, hide top/right for others
        if dataset_type is None:
            ax_auroc.spines['top'].set_visible(True)
            ax_auroc.spines['right'].set_visible(True)
        else:
            ax_auroc.spines['top'].set_visible(False)
            ax_auroc.spines['right'].set_visible(False)
        
        ax_auroc.set_xticks([i - 0.5 for i in range(len(bin_labels_numeric) + 1)])
        ax_auroc.set_xticklabels(['0', '25', '50', '75', '100'], fontsize=13 + font_addition)
        ax_auroc.set_xlim(-0.5, len(bin_labels_numeric) - 0.5)
        
        ax_auroc.set_axisbelow(True)
        ax_auroc.yaxis.grid(True, linestyle='--', linewidth=0.8, alpha=0.6, color='gray')
        
        for i in range(len(bin_labels_numeric) + 1):
            ax_auroc.axvline(x=i - 0.5, color='gray', linestyle='-', linewidth=1.0, alpha=0.3, zorder=0)
        
        # Top annotations
        ax_auroc_twin = ax_auroc.twiny()
        ax_auroc_twin.set_xlim(ax_auroc.get_xlim())
        ax_auroc_twin.set_xticks([])
        ax_auroc_twin.set_xticklabels([])
        ax_auroc_twin.tick_params(top=False, bottom=False)
        ax_auroc_twin.spines['top'].set_visible(False)
        ax_auroc_twin.spines['right'].set_visible(False)
        ax_auroc_twin.spines['bottom'].set_visible(False)
        
        # Make text bold for Merged row
        text_weight = 'bold' if dataset_type is None else 'normal'
        
        for i, label in enumerate(bin_labels_text):
            ax_auroc.text(
                i,
                1.02,
                label,
                transform=ax_auroc.get_xaxis_transform(),
                ha='center',
                va='bottom',
                fontsize=11 + font_addition,
                weight=text_weight
            )
        
        # Add column title only for first row
        if row_idx == 0:
            ax_auroc.set_title(
                'AUROC Robustness across Models',
                fontweight='bold',
                fontsize=18 + font_addition,
                y=1.12,
                pad=20
            )
        
        # Add black border for Merged row only (set after making spines visible)
        if dataset_type is None:
            for spine in ax_auroc.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(3)
                spine.set_visible(True)
        
        # Add sticky note with dataset name in bottom-right corner
        sticky_note = FancyBboxPatch(
            (0.78, 0.05),
            0.18,
            0.05,
            boxstyle="round,pad=0.01",
            transform=ax_auroc.transAxes,
            facecolor="#E8E8E8",
            edgecolor="#A0A0A0",
            linewidth=2,
            alpha=0.9,
            zorder=1000
        )
        ax_auroc.add_patch(sticky_note)
        
        ax_auroc.text(
            0.87, 0.075,
            dataset_display_name,
            transform=ax_auroc.transAxes,
            fontsize=13 + font_addition,
            weight="bold",
            ha="center",
            va="center",
            zorder=1001
        )
        
        # Only show x-label on bottom row
        if row_idx == n_rows - 1:
            ax_auroc.set_xlabel('Human Review Burden (%)', fontsize=14 + font_addition, fontweight='bold')
        else:
            ax_auroc.set_xlabel('')
        
        ax_auroc.set_ylabel('AUROC', fontsize=14 + font_addition, fontweight='bold')
        ax_auroc.set_ylim(0.0, 1.0)
        ax_auroc.set_yticks(np.arange(0.0, 1.01, 0.1))
        ax_auroc.tick_params(labelsize=13 + font_addition)
        
        # Apply font size to y-tick labels explicitly
        for label in ax_auroc.get_yticklabels():
            label.set_fontsize(13 + font_addition)
        
        if ax_auroc.get_legend():
            ax_auroc.get_legend().remove()
        
        # Collect legend handles from first row
        if row_idx == 0:
            legend_handles = [
                plt.Rectangle((0, 0), 1, 1, color=method_colors[m], label=method_labels_dict[m])
                for m in METHODS_TO_SHOW
            ]
            legend_labels = [method_labels_dict[m] for m in METHODS_TO_SHOW]
        
        # =====================================================================
        # RIGHT: Calibration Quality Boxplots
        # =====================================================================
        
        df_ece = pd.DataFrame([
            {'Workload Bin': label_mapping[b], 'Method': m, 'ECE': v}
            for b in bin_labels_original for m in methods for v in ece_data_bins[b][m]
        ])
        
        if len(df_ece) > 0:
            df_ece['Calibration Quality'] = 1 - df_ece['ECE']
            sns.boxplot(
                data=df_ece,
                x='Workload Bin',
                y='Calibration Quality',
                hue='Method',
                palette=method_colors,
                order=bin_labels_numeric,
                hue_order=methods,
                linewidth=1.5,
                ax=ax_calib
            )
        
        ax_calib.grid(False)
        ax_calib.grid(False, axis='x')
        ax_calib.xaxis.set_minor_locator(NullLocator())
        ax_calib.tick_params(axis='x', which='minor', bottom=False)
        ax_calib.xaxis.grid(False)
        
        # Show all spines for Merged row, hide top/right for others
        if dataset_type is None:
            ax_calib.spines['top'].set_visible(True)
            ax_calib.spines['right'].set_visible(True)
        else:
            ax_calib.spines['top'].set_visible(False)
            ax_calib.spines['right'].set_visible(False)
        
        ax_calib.set_xticks([i - 0.5 for i in range(len(bin_labels_numeric) + 1)])
        ax_calib.set_xticklabels(['0', '25', '50', '75', '100'], fontsize=13 + font_addition)
        ax_calib.set_xlim(-0.5, len(bin_labels_numeric) - 0.5)
        
        ax_calib.set_axisbelow(True)
        ax_calib.yaxis.grid(True, linestyle='--', linewidth=0.8, alpha=0.6, color='gray')
        
        for i in range(len(bin_labels_numeric) + 1):
            ax_calib.axvline(x=i - 0.5, color='gray', linestyle='-', linewidth=1.0, alpha=0.3, zorder=0)
        
        # Top annotations
        ax_calib_twin = ax_calib.twiny()
        ax_calib_twin.set_xlim(ax_calib.get_xlim())
        ax_calib_twin.set_xticks([])
        ax_calib_twin.set_xticklabels([])
        ax_calib_twin.tick_params(top=False, bottom=False)
        ax_calib_twin.spines['top'].set_visible(False)
        ax_calib_twin.spines['right'].set_visible(False)
        ax_calib_twin.spines['bottom'].set_visible(False)
        
        # Make text bold for Merged row
        for i, label in enumerate(bin_labels_text):
            ax_calib.text(
                i,
                1.02,
                label,
                transform=ax_calib.get_xaxis_transform(),
                ha='center',
                va='bottom',
                fontsize=11 + font_addition,
                weight=text_weight
            )
        
        # Add column title only for first row
        if row_idx == 0:
            ax_calib.set_title(
                'Calibration Quality Robustness across Models',
                fontweight='bold',
                fontsize=18 + font_addition,
                y=1.12,
                pad=20
            )
        
        # Add black border for Merged row only (set after making spines visible)
        if dataset_type is None:
            for spine in ax_calib.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(3)
                spine.set_visible(True)
        
        # Add sticky note with dataset name in bottom-right corner
        sticky_note = FancyBboxPatch(
            (0.78, 0.05),
            0.18,
            0.05,
            boxstyle="round,pad=0.01",
            transform=ax_calib.transAxes,
            facecolor="#E8E8E8",
            edgecolor="#A0A0A0",
            linewidth=2,
            alpha=0.9,
            zorder=1000
        )
        ax_calib.add_patch(sticky_note)
        
        ax_calib.text(
            0.87, 0.075,
            dataset_display_name,
            transform=ax_calib.transAxes,
            fontsize=13 + font_addition,
            weight="bold",
            ha="center",
            va="center",
            zorder=1001
        )
        
        # Only show x-label on bottom row
        if row_idx == n_rows - 1:
            ax_calib.set_xlabel('Human Review Burden (%)', fontsize=14 + font_addition, fontweight='bold')
        else:
            ax_calib.set_xlabel('')
        
        ax_calib.set_ylabel('Calibration Quality', fontsize=14 + font_addition, fontweight='bold')
        ax_calib.set_ylim(0.0, 1.0)
        ax_calib.set_yticks(np.arange(0.0, 1.01, 0.1))
        ax_calib.tick_params(labelsize=13 + font_addition)
        
        # Apply font size to y-tick labels explicitly
        for label in ax_calib.get_yticklabels():
            label.set_fontsize(13 + font_addition)
        
        if ax_calib.get_legend():
            ax_calib.get_legend().remove()
    
    # Add single legend at the top center (lower position)
    fig.legend(
        legend_handles,
        legend_labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.948),
        ncol=len(legend_labels),
        fontsize=13 + font_addition,
        frameon=True,
        fancybox=True,
        shadow=True
    )
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(str(save_path).replace('.png', '.pdf'), bbox_inches='tight')
    
    print(f"✅ Saved combined figure to {save_path}")
    
    plt.show()
    
    return fig


# Generate Row 3 combined figure
row3_save_path = OVERALL_RESULTS_PATH / "master_figure_row3_technical_robustness.png"
fig_row3 = create_row3_technical_robustness_combined(
    risk_coverage_data_contextual=risk_coverage_data_contextual,
    ece_coverage_data_contextual=ece_coverage_data_contextual,
    risk_coverage_data_synthesis=risk_coverage_data,
    ece_coverage_data_synthesis=ece_coverage_data,
    risk_coverage_data_clinical=risk_coverage_data_clinical,
    ece_coverage_data_clinical=ece_coverage_data_clinical,
    method2path=method2path,
    save_path=row3_save_path,
    font_addition=4
)


In [ ]:
# ==============================================================================
# DR. GHASSEMI'S THRESHOLD-BASED ANALYSIS (ACCURACY/ERROR-BASED)
# ==============================================================================
# Calculate optimal confidence thresholds where AI Error Rate <= (1/alpha) * Human Error Rate
# and measure % of cases that can be safely automated at those thresholds.

print("\n" + "="*80)
print("🎯 CALCULATING OPTIMAL THRESHOLDS (Dr. Ghassemi's Analysis - Error Rate Based)")
print("="*80)

def calculate_optimal_threshold_and_automation(
    results, 
    confidence_scores, 
    human_accuracy=0.90,  # The Benchmark we agreed on
    alpha=2.0,            # 2x better
    thresholds=np.arange(0.0, 1.0, 0.01) # Finer grain search
):
    """
    Find optimal confidence threshold where AI Error Rate <= (1/alpha) * Human Error Rate.
    
    Target Logic:
      - Human Accuracy: 90% (Error: 10%)
      - Target AI Error: 10% / 2 = 5%
      - Target AI Accuracy: 95%
    
    Args:
        results: List of result dicts with 'ground_truth_correctness'
        confidence_scores: List of confidence scores (same order as results)
        human_accuracy: The human benchmark (0.90)
        alpha: How much safer the AI needs to be (2.0)
        thresholds: Array of confidence thresholds to test
    
    Returns:
        Dictionary with:
            - tau: Optimal threshold (None if criterion not met)
            - pct_automated: % of cases with confidence >= tau
            - accuracy_at_tau: Accuracy at this threshold
            - target_accuracy: The target we aimed for
            - meets_criterion: Whether criterion was met
    """
    
    # 1. Determine Target Accuracy
    human_error_rate = 1.0 - human_accuracy
    target_error_rate = human_error_rate / alpha
    target_accuracy = 1.0 - target_error_rate
    
    # Combine results and confidence scores
    data = list(zip(results, confidence_scores))
    total_samples = len(data)
    
    if total_samples == 0:
        return {
            'tau': None, 'pct_automated': 0.0, 'accuracy_at_tau': 0.0,
            'target_accuracy': target_accuracy, 'meets_criterion': False
        }

    # Search for optimal threshold (start from LOW tau = high automation)
    # We want the LOWEST threshold that still satisfies the safety constraint
    optimal_tau = None
    optimal_pct_automated = 0.0
    optimal_accuracy = 0.0
    
    # Sort data by confidence for easier processing if needed, 
    # but simple loop is fine for clarity
    
    for tau in sorted(thresholds): 
        # Filter to samples with confidence >= tau (AI handles these)
        high_conf_subset = [(r, c) for r, c in data if c >= tau]
        
        # Guard rail: Need statistically significant sample size (e.g., >10 samples)
        if len(high_conf_subset) < 10:
            continue
            
        # Calculate Accuracy of this subset
        high_conf_gt = np.array([r['ground_truth_correctness'] for r, c in high_conf_subset])
        subset_accuracy = np.mean(high_conf_gt)
        
        # Check if we meet the Strict Safety Standard
        if subset_accuracy >= target_accuracy:
            # We found a valid threshold! 
            # Since we are iterating from low->high, the first one we find 
            # maximizes automation (coverage).
            optimal_tau = tau
            optimal_pct_automated = (len(high_conf_subset) / total_samples) * 100
            optimal_accuracy = subset_accuracy
            break # Stop, we found the best coverage that is safe
            
    meets_criterion = optimal_tau is not None
    
    return {
        'tau': optimal_tau,
        'pct_automated': optimal_pct_automated,
        'accuracy_at_tau': optimal_accuracy,
        'target_accuracy': target_accuracy,
        'meets_criterion': meets_criterion
    }

# Calculate thresholds for all LLM-method-dataset combinations
threshold_results = {}

for dataset_type in dataset_types:
    print(f"\n{'='*60}")
    print(f"Processing {dataset_display_names[dataset_type]}:")
    print(f"{'='*60}")
    
    threshold_results[dataset_type] = {}
    
    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        threshold_results[dataset_type][llm_display] = {}
        
        # Get shared IDs (ensure fair comparison)
        shared_record_ids = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
        
        for method_name in METHODS_TO_SHOW:
            if method_name not in method2path:
                continue
            if method_name not in llm2method2dataset2results.get(llm_id, {}): continue
            if dataset_type not in llm2method2dataset2results[llm_id].get(method_name, {}): continue
            
            results = llm2method2dataset2results[llm_id][method_name][dataset_type]
            
            # Filter to shared record IDs
            if shared_record_ids:
                filtered_results = [r for r in results if r["record_id"] in shared_record_ids]
            else:
                filtered_results = results
            
            if not filtered_results: continue
            
            confidence_scores = [r["confidence_score"] for r in filtered_results]
            
            # RUN THE NEW CALCULATION
            result = calculate_optimal_threshold_and_automation(
                filtered_results,
                confidence_scores,
                human_accuracy=0.90,  # 90% Benchmark
                alpha=2.0             # 2x Better
            )
            
            threshold_results[dataset_type][llm_display][method_name] = result
            
            # Print results
            if result['meets_criterion']:
                print(f"  {method_name}: ✅ τ={result['tau']:.2f} | "
                      f"Auto={result['pct_automated']:.1f}% | "
                      f"Acc={result['accuracy_at_tau']:.3f} (Target >{result['target_accuracy']:.3f})")
            else:
                print(f"  {method_name}: ❌ Cannot meet {result['target_accuracy']:.3f} accuracy")

print("\n✅ Threshold analysis complete!")

In [ ]:
# ==============================================================================
# DR. GHASSEMI'S FIGURE: % CASES AUTOMATED AT SAFE THRESHOLD
# ==============================================================================

def create_ghassemi_threshold_figure(
    threshold_results,
    method2path,
    dataset_type,
    dataset_display_name,
    save_path,
    font_size_adjust=0
):
    """
    Create Dr. Ghassemi's threshold-based automation figure.
    Bar chart showing % of cases that can be safely automated for each method
    while maintaining 2x lower error rate than humans.
    """
    print(f"\n🎨 Creating Dr. Ghassemi's figure for {dataset_display_name}...")
    
    # Extract data for this dataset
    dataset_data = threshold_results.get(dataset_type, {})
    if not dataset_data:
        print(f"❌ No data found for {dataset_type}")
        return None
    
    methods = ['PTRUE', 'PIK', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F', 'CCPS']
    method_automation = {m: [] for m in methods}
    method_taus = {m: [] for m in methods}
    method_accuracies = {m: [] for m in methods}
    
    # Aggregate data across LLMs
    for llm_display, llm_data in dataset_data.items():
        for method in methods:
            if method in llm_data:
                result = llm_data[method]
                if result['meets_criterion']:
                    method_automation[method].append(result['pct_automated'])
                    method_taus[method].append(result['tau'])
                    method_accuracies[method].append(result['accuracy_at_tau'])
                else:
                    # If it fails to meet safety standard, automation is 0
                    method_automation[method].append(0.0)
                    method_taus[method].append(np.nan)

    # Prepare plotting data
    plot_data = []
    for method in methods:
        vals = method_automation[method]
        if vals:
            plot_data.append({
                'Method': method,
                'Mean_Automation': np.mean(vals),
                'Std_Automation': np.std(vals),
                'Mean_Tau': np.nanmean(method_taus[method]),
                'Mean_Accuracy': np.nanmean(method_accuracies[method]),
                'N_LLMs': len(vals)
            })

    df_plot = pd.DataFrame(plot_data)
    df_plot = df_plot.sort_values('Mean_Automation') # Sort worst to best
    
    # Plot setup
    fig, ax = plt.subplots(figsize=(10, 7))
    x_pos = np.arange(len(df_plot))
    colors = [method2path[m]['base_color'] for m in df_plot['Method']]
    labels = [method2path[m]['method_name'] for m in df_plot['Method']]
    
    # Bars
    bars = ax.bar(x_pos, df_plot['Mean_Automation'], 
                  color=colors, edgecolor='black', linewidth=2, alpha=0.9)
    
    # Error bars
    for i, (idx, row) in enumerate(df_plot.iterrows()):
        if row['Mean_Automation'] > 0:
            ax.errorbar(i, row['Mean_Automation'], yerr=row['Std_Automation'],
                        fmt='none', ecolor='black', capsize=8, capthick=2, linewidth=2)
    
    # Annotations
    for i, (idx, row) in enumerate(df_plot.iterrows()):
        if row['Mean_Automation'] > 0:
            # Percentage on top
            ax.text(i, row['Mean_Automation'] + row['Std_Automation'] + 2,
                    f"{row['Mean_Automation']:.1f}%",
                    ha='center', va='bottom', fontsize=13, fontweight='bold')
            
            # Threshold inside bar
            label_y = row['Mean_Automation'] / 2 if row['Mean_Automation'] > 15 else -5
            label_color = 'white' if row['Mean_Automation'] > 15 else 'black'
            ax.text(i, label_y,
                    f"τ = {row['Mean_Tau']:.2f}\nAcc = {row['Mean_Accuracy']:.3f}",
                    ha='center', va='center', fontsize=10, color=label_color, fontweight='bold')
        else:
            ax.text(i, 2, "Unsafe", ha='center', va='bottom', fontsize=11, 
                    fontweight='bold', color='darkred', style='italic')

    # Formatting
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=14, fontweight='bold')
    ax.set_ylabel('% Cases Safely Automated →', fontsize=14 + font_size_adjust, fontweight='bold')
    ax.set_title(f'Safe Automation Capacity\n(Target: 2x Lower Error than Human)',
                 fontsize=16 + font_size_adjust, fontweight='bold', pad=20)
    
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Subtitle with specific math
    fig.text(0.5, 0.94, 
             f'Human Acc=90% (Err=10%) → Target AI Acc=95% (Err=5%)',
             ha='center', fontsize=11, style='italic', color='gray')

    plt.tight_layout(rect=[0, 0.04, 1, 0.92])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(str(save_path).replace('.png', '.pdf'), bbox_inches='tight')
    plt.show()
    return fig

In [ ]:
# ==============================================================================
# GENERATE DR. GHASSEMI'S FIGURES
# ==============================================================================

print("\n" + "="*80)
print("🎨 GENERATING DR. GHASSEMI'S THRESHOLD-BASED FIGURES")
print("="*80)

# Create output directory
GHASSEMI_FIGURES_PATH = OVERALL_RESULTS_PATH / "ghassemi_threshold_analysis"
os.makedirs(GHASSEMI_FIGURES_PATH, exist_ok=True)

# Generate figure for each dataset
ghassemi_figures = {}

for dataset_type in dataset_types:
    dataset_display_name = dataset_display_names[dataset_type]
    save_path = GHASSEMI_FIGURES_PATH / f"threshold_automation_{dataset_type}.png"
    
    fig = create_ghassemi_threshold_figure(
        threshold_results=threshold_results,
        method2path=method2path,
        dataset_type=dataset_type,
        dataset_display_name=dataset_display_name,
        save_path=save_path
    )
    
    ghassemi_figures[dataset_type] = fig

# Also create a merged version across all datasets
print("\n" + "="*60)
print("Creating merged figure across all datasets...")
print("="*60)

# Aggregate threshold results across all datasets
merged_threshold_results = {'merged': {}}

for dataset_type in dataset_types:
    for llm_display, llm_data in threshold_results.get(dataset_type, {}).items():
        # Create unique key for each LLM-dataset combination
        merged_key = f"{llm_display}-{dataset_type}"
        merged_threshold_results['merged'][merged_key] = llm_data

save_path_merged = GHASSEMI_FIGURES_PATH / "threshold_automation_merged.png"
fig_merged = create_ghassemi_threshold_figure(
    threshold_results=merged_threshold_results,
    method2path=method2path,
    dataset_type='merged',
    dataset_display_name='All Datasets (Merged)',
    save_path=save_path_merged
)

ghassemi_figures['merged'] = fig_merged

print("\n" + "="*80)
print("🎉 DR. GHASSEMI'S ANALYSIS COMPLETE!")
print("="*80)
print(f"\nGenerated figures:")
print(f"  📊 {GHASSEMI_FIGURES_PATH / 'threshold_automation_contextual.png'}")
print(f"  📊 {GHASSEMI_FIGURES_PATH / 'threshold_automation_synthesis.png'}")
print(f"  📊 {GHASSEMI_FIGURES_PATH / 'threshold_automation_clinical_inference.png'}")
print(f"  📊 {GHASSEMI_FIGURES_PATH / 'threshold_automation_merged.png'}")

In [ ]:
# ==============================================================================
# SUMMARY TABLE: Threshold Analysis Results (COST-BASED)
# ==============================================================================

print("\n" + "="*80)
print("📊 THRESHOLD ANALYSIS SUMMARY TABLE (COST-BASED)")
print("="*80)

summary_data = []

for dataset_type in dataset_types:
    dataset_display_name = dataset_display_names[dataset_type]
    
    for method in ['PTRUE', 'PIK', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F', 'CCPS']:
        method_display = method2path[method]['method_name']
        
        automations = []
        taus = []
        accuracies = []
        
        # Aggregate
        for llm_display, llm_data in threshold_results.get(dataset_type, {}).items():
            if method in llm_data:
                result = llm_data[method]
                if result['meets_criterion']:
                    automations.append(result['pct_automated'])
                    taus.append(result['tau'])
                    accuracies.append(result['accuracy_at_tau'])
                else:
                    automations.append(0.0) # Count as 0% automation if unsafe
        
        summary_data.append({
            'Dataset': dataset_display_name,
            'Method': method_display,
            'Mean τ': f"{np.nanmean(taus):.3f}" if taus else "N/A",
            'Mean % Automated': f"{np.mean(automations):.1f}%",
            'Mean Accuracy': f"{np.nanmean(accuracies):.3f}" if accuracies else "N/A",
            'Target Accuracy': f"{0.95:.3f}" # Hardcoded based on our alpha=2 logic
        })

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))
df_summary.to_csv(GHASSEMI_FIGURES_PATH / "threshold_analysis_summary.csv", index=False)

In [ ]:
# ==============================================================================
# FINANCIAL SENSITIVITY & SAFETY ANALYSIS (CONFUSION MATRIX)
# ==============================================================================
import pandas as pd
import numpy as np

print("\n" + "="*80)
print("💰 FINANCIAL & SAFETY ANALYSIS: 3 REALISTIC CLINICAL SETTINGS")
print("="*80)

# --- 1. DEFINING REALISTIC SCENARIOS ---
# Assumption: Human Oncologist takes 2.0 minutes per case
HUMAN_MINUTES_PER_CASE = 2.0  

# Pricing Options (Source: CMS 2026 Pricing - CPT 99205)
PRICE_PRIVATE  = 236.81  # Non-Facility Price
PRICE_HOSPITAL = 160.32  # Facility Price

scenarios = [
    {
        "Setting": "Private Specialist Group",
        "Type": "Private (Non-Facility)",
        "Wage_Basis": PRICE_PRIVATE,
        "Annual_Volume": 10000, 
        "Description": "Single specialized oncology department"
    },
    {
        "Setting": "Academic Medical Center",
        "Type": "Hospital (Facility)",
        "Wage_Basis": PRICE_HOSPITAL,
        "Annual_Volume": 50000,
        "Description": "Large university hospital system"
    },
    {
        "Setting": "Integrated Health Network",
        "Type": "Hospital (Facility)",
        "Wage_Basis": PRICE_HOSPITAL,
        "Annual_Volume": 1000000,
        "Description": "National IDN (e.g., VA, Kaiser) screening"
    }
]

# --- 2. CALCULATING IMPACT PER METHOD & DATASET ---

sensitivity_data = []

# Helper to process a specific dataset and scenario
def process_scenario_with_safety(dataset_name, method_name, scenario, automation_rate, accuracy_at_tau):
    # 1. Financials
    human_cost_per_case = (scenario["Wage_Basis"] / 60.0) * HUMAN_MINUTES_PER_CASE
    cases_automated = int(scenario["Annual_Volume"] * (automation_rate / 100.0))
    dollars_saved = cases_automated * human_cost_per_case

    # 2. Confusion Matrix of Automation (Safety Check)
    # accuracy_at_tau tells us how many of the automated cases were actually correct
    if cases_automated > 0:
        auto_correct = int(cases_automated * accuracy_at_tau) # "Safe Automations"
        auto_errors  = cases_automated - auto_correct         # "Unsafe Errors" (The Risk)
    else:
        auto_correct = 0
        auto_errors = 0

    auto_total = auto_correct + auto_errors

    return {
        "Scenario": scenario["Setting"],
        "Annual Volume": f"{scenario['Annual_Volume']:,}",
        "Dataset": dataset_name,
        "Method": method_name,
        "Mean % Auto": f"{automation_rate:.1f}%",
        "Est. Annual Savings": f"${dollars_saved:,.0f}",
        "Auto_Correct (Safe)": f"{auto_correct:,}",
        "Auto_Error (Risk)": f"{auto_errors:,}", 
        "Auto_Total": f"{auto_total:,}",
        "Error Rate": f"{(1-accuracy_at_tau)*100:.2f}%" if cases_automated > 0 else "N/A",
        # Raw values for sorting
        "_savings_raw": dollars_saved,
        "_m_order": 0
    }

# Loop through Datasets
all_stats = {'PTRUE': [], 'PIK': [], 'SAPLMA-M': [], 'SAPLMA-UM': [], 'SAPLMA-F': [], 'CCPS': []}

for dataset_type in dataset_types:
    dataset_disp = dataset_display_names[dataset_type]
    
    for method in ['PTRUE', 'PIK', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F', 'CCPS']:
        method_disp = method2path[method]['method_name']
        
        # Collect stats for all LLMs that met criterion
        rates = []
        accs = []
        
        llm_data = threshold_results.get(dataset_type, {})
        for res in llm_data.values():
            if method in res:
                if res[method]['meets_criterion']:
                    rates.append(res[method]['pct_automated'])
                    accs.append(res[method]['accuracy_at_tau'])
                else:
                    rates.append(0.0) 
                    accs.append(0.0) # Or NaN, but 0 volume implies 0 accuracy impact
        
        avg_rate = np.mean(rates) if rates else 0.0
        # For accuracy, we only average the valid ones (where automation > 0)
        valid_accs = [a for r, a in zip(rates, accs) if r > 0]
        avg_acc = np.mean(valid_accs) if valid_accs else 0.0
        
        # Store for merged calculation
        all_stats[method].append((avg_rate, avg_acc))
        
        # Calculate for each Scenario
        for scen in scenarios:
            row = process_scenario_with_safety(dataset_disp, method_disp, scen, avg_rate, avg_acc)
            sensitivity_data.append(row)

# --- 3. ADD "MERGED RESULTS" (AVERAGE ACROSS DATASETS) ---
for method in ['PTRUE', 'PIK', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F', 'CCPS']:
    method_disp = method2path[method]['method_name']
    
    # Average across the datasets
    stats = all_stats[method] # List of (rate, acc) tuples
    if stats:
        merged_rate = np.mean([s[0] for s in stats])
        # Weighted average of accuracy based on automation rate? Simple mean is safer for broad summary.
        valid_accs = [s[1] for s in stats if s[0] > 0]
        merged_acc = np.mean(valid_accs) if valid_accs else 0.0
    else:
        merged_rate = 0.0
        merged_acc = 0.0
    
    for scen in scenarios:
        row = process_scenario_with_safety("COMBINED (Avg All Data)", method_disp, scen, merged_rate, merged_acc)
        sensitivity_data.append(row)

# --- 4. OUTPUT & FORMATTING ---
df_sens = pd.DataFrame(sensitivity_data)

# Sorting
method_order = {
    method2path['PTRUE']['method_name']: 0, 
    method2path['PIK']['method_name']: 1, 
    method2path['CCPS']['method_name']: 2
}
df_sens['_m_order'] = df_sens['Method'].map(method_order)
scen_order = {s['Setting']: i for i, s in enumerate(scenarios)}
df_sens['_s_order'] = df_sens['Scenario'].map(scen_order)

df_sens = df_sens.sort_values(['_s_order', 'Dataset', '_m_order'])

# Clean columns: Only include 'Auto_Total' (remove 'Auto_Correct+Error_Sum')
cols = ['Scenario', 'Dataset', 'Method', 'Mean % Auto', 'Est. Annual Savings', 
        'Auto_Correct (Safe)', 'Auto_Error (Risk)', 'Auto_Total', 'Error Rate']
df_sens = df_sens[cols]

print(df_sens.to_string(index=False))

# Save
sensitivity_csv_path = GHASSEMI_FIGURES_PATH / "financial_safety_analysis.csv"
df_sens.to_csv(sensitivity_csv_path, index=False)
print(f"\n✅ Safety & Financial analysis saved to: {sensitivity_csv_path}")

In [ ]:
# ==============================================================================
# THREE-PANEL IMPACT DASHBOARD (REAL DATA + MULTI-SCENARIO FINANCIALS)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("\n" + "="*80)
print("🎯 CREATING IMPACT DASHBOARD (REAL METRICS + MULTI-SITE FINANCIALS)")
print("="*80)

dashboard_configs = [
    {"type": "contextual", "name": "CORAL-MCQA-Contextual"},
    {"type": "synthesis", "name": "CORAL-MCQA-Synthesis"},
    {"type": "clinical_inference", "name": "CORAL-MCQA-Clinical-Inference"},
    {"type": None, "name": "All Datasets (Merged)"}
]

def create_impact_dashboard(dataset_type, dataset_display_name, scenarios_list, save_path):
    """
    Panel A: Actual Test Set Yield (Real N)
    Panel B: Actual Test Set Safety (Real N)
    Panel C: Projected Savings across 3 Different Clinical Sites (Grouped Bar)
    """
    
    # -------------------------------------------------------------------------
    # 1. DATA PREP (REAL TEST SET METRICS)
    # -------------------------------------------------------------------------
    
    # Calculate Real N (Average per LLM)
    real_n_samples = 0
    datasets_to_use = dataset_types if dataset_type is None else [dataset_type]
    
    for dt in datasets_to_use:
        for llm in llm_ids:
            if 'CCPS' in llm2method2dataset2results.get(llm, {}):
                if dt in llm2method2dataset2results[llm]['CCPS']:
                    shared = llm2dataset2sharedRIDacrossMethods[llm].get(dt, set())
                    count = len(shared) if shared else len(llm2method2dataset2results[llm]['CCPS'][dt])
                    real_n_samples += count
    
    real_n_samples = int(real_n_samples / len(llm_ids)) 
    
    print(f"\nProcessing: {dataset_display_name}")
    print(f"ACTUAL Test Set Size (N): {real_n_samples:,}")
    
    # Setup Methods & Colors
    dashboard_methods = METHODS_TO_SHOW
    color_auto = {m: method2path[m]['base_color'] for m in dashboard_methods}
    color_human = '#E0E0E0' 
    color_safe = '#2E7D32'  
    color_risk = '#C62828'  
    
    # Calculate Aggregate Metrics per Method
    method_metrics = {}
    
    for method in dashboard_methods:
        automations, accuracies = [], []
        
        for dt in datasets_to_use:
            llm_data = threshold_results.get(dt, {})
            for llm_results in llm_data.values():
                if method in llm_results:
                    res = llm_results[method]
                    if res['meets_criterion']:
                        automations.append(res['pct_automated'])
                        accuracies.append(res['accuracy_at_tau'])
                    else:
                        automations.append(0.0)
        
        avg_pct_auto = np.mean(automations) if automations else 0.0
        avg_acc = np.mean([a for a in accuracies if a > 0]) if accuracies else 0.0
        
        # Real Test Set Counts (For Panels A & B)
        real_auto_count = int(real_n_samples * (avg_pct_auto / 100.0))
        real_correct = int(real_auto_count * avg_acc)
        real_error = real_auto_count - real_correct
        
        method_metrics[method] = {
            'pct_auto': avg_pct_auto,
            'real_auto': real_auto_count,
            'real_correct': real_correct,
            'real_error': real_error,
            'error_rate': (1 - avg_acc)*100 if real_auto_count > 0 else 0.0,
        }

    # -------------------------------------------------------------------------
    # 2. PLOTTING
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=(22, 7)) # Slightly wider for the grouped bars
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.4], wspace=0.25)
    
    ax_auto = fig.add_subplot(gs[0])
    ax_safety = fig.add_subplot(gs[1])
    ax_finance = fig.add_subplot(gs[2])
    
    x_pos = np.arange(len(dashboard_methods))
    bar_width = 0.6
    font_size_metrics = 14
    
    # --- PANEL A: AUTOMATION YIELD (%) ---
    # Shows metrics on the REAL test set N
    
    for i, method in enumerate(dashboard_methods):
        d = method_metrics[method]
        human_pct = 100.0 - d['pct_auto']
        human_count = int(real_n_samples * (human_pct / 100.0))
        
        # Stacked Bar
        ax_auto.bar(i, d['pct_auto'], bar_width, color=color_auto[method], edgecolor='black', alpha=0.9)
        ax_auto.bar(i, human_pct, bar_width, bottom=d['pct_auto'], color=color_human, edgecolor='black', hatch='///', alpha=0.6)
        
        # Labels in colored regions (AI automated)
        if d['pct_auto'] > 5:
            ax_auto.text(i, d['pct_auto']/2, f"{d['pct_auto']:.1f}%\n({d['real_auto']:,})", 
                         ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='white')
        
        # Labels in shaded regions (Human review)
        if human_pct > 5:
            ax_auto.text(i, d['pct_auto'] + human_pct/2, f"{human_pct:.1f}%\n({human_count:,})", 
                         ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='black')
        
        ax_auto.text(i, 102, f"{real_n_samples:,}", ha='center', va='bottom', fontsize=12, fontweight='bold', color='black')
    
    ax_auto.set_title('(H) Clinical Yield (% of Test Set Automated)', fontsize=16, fontweight='bold', pad=15)
    ax_auto.set_ylabel('% of Cases', fontsize=14, fontweight='bold')
    ax_auto.set_ylim(0, 115)
    ax_auto.set_xticks(x_pos)
    ax_auto.set_xticklabels([method2path[m]['method_name'] for m in dashboard_methods], fontsize=13, fontweight='bold')
    
    leg_human = mpatches.Patch(facecolor=color_human, edgecolor='black', hatch='///', label='Human Review')
    ax_auto.legend(handles=[leg_human], loc='upper center', fontsize=11, frameon=True, fancybox=True)
    
    # --- PANEL B: SAFETY (REAL COUNTS) ---
    # Shows Correct vs Error on REAL test set
    
    max_real_cases = max([d['real_auto'] for d in method_metrics.values()]) if method_metrics else 100
    
    for i, method in enumerate(dashboard_methods):
        d = method_metrics[method]
        
        ax_safety.bar(i, d['real_correct'], bar_width, color=color_auto[method], edgecolor='black', alpha=0.9)
        ax_safety.bar(i, d['real_error'], bar_width, bottom=d['real_correct'], color='#000000', edgecolor='black', alpha=0.9)
        
        # Calculate percentages relative to total automated cases
        total_auto = d['real_auto']
        if total_auto > 0:
            correct_pct = (d['real_correct'] / total_auto) * 100.0
            error_pct = (d['real_error'] / total_auto) * 100.0
        else:
            correct_pct = 0.0
            error_pct = 0.0
        
        # Method Color Label (Correct) - inside the bar
        if d['real_correct'] > 0:
            ax_safety.text(i, d['real_correct']/2, f"{correct_pct:.1f}%\n({d['real_correct']:,})", 
                           ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='white')
        
        # Black Label (Error) - above the bar in black
        if d['real_error'] > 0:
            y_txt = d['real_correct'] + d['real_error'] + (max_real_cases * 0.02)
            ax_safety.text(i, y_txt, f"{error_pct:.1f}%\n({d['real_error']:,})", 
                           ha='center', va='bottom', fontweight='bold', fontsize=font_size_metrics, color='#000000')
        elif d['pct_auto'] == 0:
             ax_safety.text(i, 5, "No Auto", ha='center', va='bottom', style='italic', color='gray')

    ax_safety.set_title('(I) Safety Profile (Actual Test Set Errors)', fontsize=16, fontweight='bold', pad=15)
    ax_safety.set_ylabel('Automated Cases (Count)', fontsize=14, fontweight='bold')
    ax_safety.set_ylim(0, max_real_cases * 1.15)
    ax_safety.set_xticks(x_pos)
    ax_safety.set_xticklabels([method2path[m]['method_name'] for m in dashboard_methods], fontsize=13, fontweight='bold')
    
    # Legend showing only Error (black)
    leg_b = [mpatches.Patch(facecolor='#000000', label='Error')]
    ax_safety.legend(handles=leg_b, loc='upper left', frameon=True)

    # --- PANEL C: FINANCIAL (MULTI-SCENARIO GROUPED BAR) ---
    # X-Axis: Sites (Small, Medium, Large)
    # Groups: Methods
    
    # sites = [s['Setting'].split(' ')[0] + '\n' + s['Description'] for s in scenarios_list]
    sites = ["Small", "Medium", "Large"]
    site_indices = np.arange(len(sites))
    width = 0.25  # Width of individual bars
    
    # We use a LOG scale because the values range from $10k to $3M
    ax_finance.set_yscale('log')
    
    # Calculate all savings values first to determine max for dynamic ylim
    all_savings_values = []
    all_method_savings = {}  # Store savings per method for later use
    
    for i, method in enumerate(dashboard_methods):
        d = method_metrics[method]
        pct_auto = d['pct_auto']
        savings_values = []
        
        for scen in scenarios_list:
            # Calculate savings for this specific scenario
            # Cost per case = (Wage / 60) * 2 mins
            cost_per_case = (scen["Wage_Basis"] / 60.0) * 2.0
            savings = scen["Annual_Volume"] * (pct_auto / 100.0) * cost_per_case
            savings_values.append(savings)
            if savings > 0:
                all_savings_values.append(savings)
        
        all_method_savings[method] = savings_values
    
    # Calculate dynamic y-axis limits based on max savings
    max_savings = max(all_savings_values) if all_savings_values else 1_000_000
    min_savings = min(all_savings_values) if all_savings_values else 1000
    
    # Set dynamic ylim: lower bound at least 1000, upper bound 1.5x max (similar to Panel B's 1.15)
    y_min = max(1000, min_savings * 0.5)  # At least 1000, or half of min if larger
    y_max = max_savings * 1.5  # 1.5x the maximum bar value
    
    # Now create bars and labels
    for i, method in enumerate(dashboard_methods):
        savings_values = all_method_savings[method]
        
        # Position bars: Center is 'site_indices'. Offset by 'i'
        # i=0 (PTrue) -> left, i=1 (PIK) -> center, i=2 (CCPS) -> right
        offset = (i - 1) * width
        rects = ax_finance.bar(site_indices + offset, savings_values, width, 
                               label=method2path[method]['method_name'],
                               color=color_auto[method], edgecolor='black', alpha=0.9)
        
        # Label bars (formatted K/M) or "No Auto" if zero
        for j, rect in enumerate(rects):
            height = rect.get_height()
            x_center = rect.get_x() + rect.get_width()/2.
            
            if height > 0:
                if height >= 1_000_000:
                    label = f'${height/1_000_000:.1f} M'
                elif height >= 1_000:
                    label = f'${height/1_000:.0f} K'
                else:
                    label = f'${height:.0f}'
                
                # Position label inside the bar at the top, vertical orientation
                # Use 0.9 of bar height to position near the top but inside
                ax_finance.text(x_center, height * 0.9,
                                label, ha='center', va='top', fontsize=16, fontweight='bold', 
                                rotation=90, color='black')
            else:
                # Vertical "No Auto" text for zero savings (positioned relative to y_min)
                no_auto_y = y_min * 1.6  # Position slightly above y_min for visibility
                ax_finance.text(x_center, no_auto_y, "No Auto", 
                               ha='center', va='center', style='italic', color='gray', 
                               fontsize=14, rotation=90, fontweight='normal')

    ax_finance.set_title('(J) Financial Impact by Clinical Site Size', fontsize=16, fontweight='bold', pad=15)
    ax_finance.set_ylabel('Est. Annual Savings (Log Scale)', fontsize=14, fontweight='bold')
    ax_finance.set_xticks(site_indices)
    ax_finance.set_xticklabels(sites, fontsize=12, fontweight='bold')
    
    # Custom Y-axis formatter for Log Scale readability (show as 10M, 10K, etc.)
    import matplotlib.ticker as ticker
    def format_y_axis(y, pos):
        if y >= 1_000_000:
            return f'${int(y/1_000_000)}M'
        elif y >= 1_000:
            return f'${int(y/1_000)}K'
        else:
            return f'${int(y)}'
    
    ax_finance.yaxis.set_major_formatter(ticker.FuncFormatter(format_y_axis))
    ax_finance.set_ylim(y_min, y_max)  # Dynamic limits based on data
    ax_finance.grid(axis='y', alpha=0.3, which="both", linestyle='--')
    
    # Legend for C
    ax_finance.legend(loc='upper left', fontsize=11, frameon=True, fancybox=True)

    # Bold prefix
    fig.text(
        0.5, 1,
        f"Impact Dashboard: {dataset_display_name}",
        ha="right",
        va="center",
        fontsize=20,
        fontweight="bold"
    )

    # Normal suffix
    fig.text(
        0.5, 1,
        f" - Observed Performance (N={real_n_samples:,}) vs. Multi-Site Projections",
        ha="left",
        va="center",
        fontsize=20
    )
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(str(save_path).replace('.png', '.pdf'), bbox_inches='tight')
    print(f"✅ Saved: {save_path}")
    plt.show()

# ==============================================================================
# GENERATE
# ==============================================================================
# Pass the full list of scenarios to Panel C
all_scenarios = scenarios 

for config in dashboard_configs:
    dataset_type = config["type"]
    if dataset_type is None: suffix = "merged"
    else: suffix = dataset_type
    
    save_path = GHASSEMI_FIGURES_PATH / f"impact_dashboard_multisite_{suffix}.png"
    create_impact_dashboard(dataset_type, config["name"], all_scenarios, save_path)

In [ ]:
# ==============================================================================
# IMPACT DASHBOARD - Combined Figure (All Datasets Stacked)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np

def create_impact_dashboard_combined(scenarios_list, save_path, font_addition=0):
    """
    Create a stacked impact dashboard with all datasets (Contextual, Synthesis, Inference, Merged)
    Panel H: Clinical Yield (% of Test Set Automated)
    Panel I: Safety Profile (Actual Test Set Errors)
    Panel J: Financial Impact by Clinical Site Size
    """
    
    print("\n" + "="*80)
    print("🎯 CREATING COMBINED IMPACT DASHBOARD (ALL DATASETS STACKED)")
    print("="*80)
    
    dashboard_configs = [
        {"type": "contextual", "name": "Contextual"},
        {"type": "synthesis", "name": "Synthesis"},
        {"type": "clinical_inference", "name": "Inference"},
        {"type": None, "name": "Merged"}
    ]
    
    n_rows = len(dashboard_configs)
    
    # Setup Methods & Colors
    dashboard_methods = METHODS_TO_SHOW
    color_auto = {m: method2path[m]['base_color'] for m in dashboard_methods}
    color_human = '#E0E0E0' 
    color_safe = '#2E7D32'  
    color_risk = '#C62828'  
    
    # Create figure with stacked rows
    fig = plt.figure(figsize=(22, 7 * n_rows))
    gs = fig.add_gridspec(
        n_rows,
        3,
        width_ratios=[1, 1, 1.4],
        hspace=0.10,
        wspace=0.25
    )
    
    # Prepare legend handles once (for Panel J)
    legend_handles_auto = []
    for method in dashboard_methods:
        legend_handles_auto.append(
            mpatches.Patch(facecolor=color_auto[method], label=method2path[method]['method_name'])
        )
    
    for row_idx, config in enumerate(dashboard_configs):
        dataset_type = config["type"]
        dataset_display_name = config["name"]
        
        # -------------------------------------------------------------------------
        # 1. DATA PREP (REAL TEST SET METRICS)
        # -------------------------------------------------------------------------
        
        # Calculate Real N (Average per LLM)
        real_n_samples = 0
        datasets_to_use = dataset_types if dataset_type is None else [dataset_type]
        
        for dt in datasets_to_use:
            for llm in llm_ids:
                if 'CCPS' in llm2method2dataset2results.get(llm, {}):
                    if dt in llm2method2dataset2results[llm]['CCPS']:
                        shared = llm2dataset2sharedRIDacrossMethods[llm].get(dt, set())
                        count = len(shared) if shared else len(llm2method2dataset2results[llm]['CCPS'][dt])
                        real_n_samples += count
        
        real_n_samples = int(real_n_samples / len(llm_ids)) 
        
        print(f"\nProcessing: {dataset_display_name}")
        print(f"ACTUAL Test Set Size (N): {real_n_samples:,}")
        
        # Calculate Aggregate Metrics per Method
        method_metrics = {}
        
        for method in dashboard_methods:
            automations, accuracies = [], []
            
            for dt in datasets_to_use:
                llm_data = threshold_results.get(dt, {})
                for llm_results in llm_data.values():
                    if method in llm_results:
                        res = llm_results[method]
                        if res['meets_criterion']:
                            automations.append(res['pct_automated'])
                            accuracies.append(res['accuracy_at_tau'])
                        else:
                            automations.append(0.0)
            
            avg_pct_auto = np.mean(automations) if automations else 0.0
            avg_acc = np.mean([a for a in accuracies if a > 0]) if accuracies else 0.0
            
            # Real Test Set Counts (For Panels A & B)
            real_auto_count = int(real_n_samples * (avg_pct_auto / 100.0))
            real_correct = int(real_auto_count * avg_acc)
            real_error = real_auto_count - real_correct
            
            method_metrics[method] = {
                'pct_auto': avg_pct_auto,
                'real_auto': real_auto_count,
                'real_correct': real_correct,
                'real_error': real_error,
                'error_rate': (1 - avg_acc)*100 if real_auto_count > 0 else 0.0,
            }
        
        # -------------------------------------------------------------------------
        # 2. PLOTTING
        # -------------------------------------------------------------------------
        
        ax_auto = fig.add_subplot(gs[row_idx, 0])
        ax_safety = fig.add_subplot(gs[row_idx, 1])
        ax_finance = fig.add_subplot(gs[row_idx, 2])
        
        x_pos = np.arange(len(dashboard_methods))
        bar_width = 0.6
        font_size_metrics = 14 + font_addition
        
        # --- PANEL H: AUTOMATION YIELD (%) ---
        # Shows metrics on the REAL test set N
        
        for i, method in enumerate(dashboard_methods):
            d = method_metrics[method]
            human_pct = 100.0 - d['pct_auto']
            human_count = int(real_n_samples * (human_pct / 100.0))
            
            # Stacked Bar
            ax_auto.bar(i, d['pct_auto'], bar_width, color=color_auto[method], edgecolor='black', alpha=0.9)
            ax_auto.bar(i, human_pct, bar_width, bottom=d['pct_auto'], color=color_human, edgecolor='black', hatch='///', alpha=0.6)
            
            # Labels in colored regions (AI automated)
            if d['pct_auto'] > 5:
                ax_auto.text(i, d['pct_auto']/2, f"{d['pct_auto']:.1f}%\n({d['real_auto']:,})", 
                             ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='white')
            
            # Labels in shaded regions (Human review)
            if human_pct > 5:
                ax_auto.text(i, d['pct_auto'] + human_pct/2, f"{human_pct:.1f}%\n({human_count:,})", 
                             ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='black')
            
            ax_auto.text(i, 102, f"{real_n_samples:,}", ha='center', va='bottom', fontsize=12 + font_addition, fontweight='bold', color='black')
        
        # Add column title only for first row
        if row_idx == 0:
            ax_auto.set_title('(H) Clinical Yield (% of Test Set Automated)', fontsize=16 + font_addition, fontweight='bold', pad=15)
        
        ax_auto.set_ylabel('% of Cases', fontsize=14 + font_addition, fontweight='bold')
        ax_auto.set_ylim(0, 115)
        ax_auto.set_xticks(x_pos)
        
        # Only show x-axis labels on bottom row
        if row_idx == n_rows - 1:
            ax_auto.set_xticklabels([method2path[m]['method_name'] for m in dashboard_methods], fontsize=13 + font_addition, fontweight='bold')
        else:
            ax_auto.set_xticklabels([''] * len(dashboard_methods))
        
        # Apply font size to tick labels
        ax_auto.tick_params(labelsize=13 + font_addition)
        
        # Collect legend handles from first row only
        if row_idx == 0:
            leg_human = mpatches.Patch(facecolor=color_human, edgecolor='black', hatch='///', label='Human Review')
            legend_handles_human = [leg_human]
            ax_auto.legend(handles=legend_handles_human, loc='upper center', fontsize=11 + font_addition, frameon=True, fancybox=True)
        
        # Add black border for Merged row
        if dataset_type is None:
            for spine in ax_auto.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(3)
        
        # Add sticky note with dataset name
        from matplotlib.patches import FancyBboxPatch
        sticky_note = FancyBboxPatch(
            (0.02, 0.90),
            0.15,
            0.05,
            boxstyle="round,pad=0.01",
            transform=ax_auto.transAxes,
            facecolor="#E8E8E8",
            edgecolor="#A0A0A0",
            linewidth=2,
            alpha=0.9,
            zorder=1000
        )
        ax_auto.add_patch(sticky_note)
        
        ax_auto.text(
            0.095, 0.925,
            dataset_display_name,
            transform=ax_auto.transAxes,
            fontsize=13 + font_addition,
            weight="bold",
            ha="center",
            va="center",
            zorder=1001
        )
        
        # --- PANEL I: SAFETY (REAL COUNTS) ---
        # Shows Correct vs Error on REAL test set
        
        max_real_cases = max([d['real_auto'] for d in method_metrics.values()]) if method_metrics else 100
        
        for i, method in enumerate(dashboard_methods):
            d = method_metrics[method]
            
            ax_safety.bar(i, d['real_correct'], bar_width, color=color_auto[method], edgecolor='black', alpha=0.9)
            ax_safety.bar(i, d['real_error'], bar_width, bottom=d['real_correct'], color='#000000', edgecolor='black', alpha=0.9)
            
            # Calculate percentages relative to total automated cases
            total_auto = d['real_auto']
            if total_auto > 0:
                correct_pct = (d['real_correct'] / total_auto) * 100.0
                error_pct = (d['real_error'] / total_auto) * 100.0
            else:
                correct_pct = 0.0
                error_pct = 0.0
            
            # Method Color Label (Correct) - inside the bar
            if d['real_correct'] > 0:
                ax_safety.text(i, d['real_correct']/2, f"{correct_pct:.1f}%\n({d['real_correct']:,})", 
                               ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='white')
            
            # Black Label (Error) - above the bar in black
            if d['real_error'] > 0:
                y_txt = d['real_correct'] + d['real_error'] + (max_real_cases * 0.02)
                ax_safety.text(i, y_txt, f"{error_pct:.1f}%\n({d['real_error']:,})", 
                               ha='center', va='bottom', fontweight='bold', fontsize=font_size_metrics, color='#000000')
            elif d['pct_auto'] == 0:
                 ax_safety.text(i, 5, "No Auto", ha='center', va='bottom', style='italic', color='gray', fontsize=font_size_metrics)

        # Add column title only for first row
        if row_idx == 0:
            ax_safety.set_title('(I) Safety Profile (Actual Test Set Errors)', fontsize=16 + font_addition, fontweight='bold', pad=15)
        
        ax_safety.set_ylabel('Automated Cases (Count)', fontsize=14 + font_addition, fontweight='bold')
        ax_safety.set_ylim(0, max_real_cases * 1.15)
        ax_safety.set_xticks(x_pos)
        
        # Only show x-axis labels on bottom row
        if row_idx == n_rows - 1:
            ax_safety.set_xticklabels([method2path[m]['method_name'] for m in dashboard_methods], fontsize=13 + font_addition, fontweight='bold')
        else:
            ax_safety.set_xticklabels([''] * len(dashboard_methods))
        
        # Apply font size to tick labels
        ax_safety.tick_params(labelsize=13 + font_addition)
        
        # Collect legend handles from first row only
        if row_idx == 0:
            leg_b = [mpatches.Patch(facecolor='#000000', label='Error')]
            legend_handles_error = leg_b
            ax_safety.legend(handles=legend_handles_error, loc='upper left', frameon=True, fontsize=11 + font_addition)
        
        # Add black border for Merged row
        if dataset_type is None:
            for spine in ax_safety.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(3)
        
        # Add sticky note with dataset name
        sticky_note = FancyBboxPatch(
            (0.02, 0.90),
            0.15,
            0.05,
            boxstyle="round,pad=0.01",
            transform=ax_safety.transAxes,
            facecolor="#E8E8E8",
            edgecolor="#A0A0A0",
            linewidth=2,
            alpha=0.9,
            zorder=1000
        )
        ax_safety.add_patch(sticky_note)
        
        ax_safety.text(
            0.095, 0.925,
            dataset_display_name,
            transform=ax_safety.transAxes,
            fontsize=13 + font_addition,
            weight="bold",
            ha="center",
            va="center",
            zorder=1001
        )

        # --- PANEL J: FINANCIAL (MULTI-SCENARIO GROUPED BAR) ---
        # X-Axis: Sites (Small, Medium, Large)
        # Groups: Methods
        
        sites = ["Small", "Medium", "Large"]
        site_indices = np.arange(len(sites))
        width = 0.25  # Width of individual bars
        
        # We use a LOG scale because the values range from $10k to $3M
        ax_finance.set_yscale('log')
        
        # Calculate all savings values first to determine max for dynamic ylim
        all_savings_values = []
        all_method_savings = {}  # Store savings per method for later use
        
        for i, method in enumerate(dashboard_methods):
            d = method_metrics[method]
            pct_auto = d['pct_auto']
            savings_values = []
            
            for scen in scenarios_list:
                # Calculate savings for this specific scenario
                # Cost per case = (Wage / 60) * 2 mins
                cost_per_case = (scen["Wage_Basis"] / 60.0) * 2.0
                savings = scen["Annual_Volume"] * (pct_auto / 100.0) * cost_per_case
                savings_values.append(savings)
                if savings > 0:
                    all_savings_values.append(savings)
            
            all_method_savings[method] = savings_values
        
        # Calculate dynamic y-axis limits based on max savings
        max_savings = max(all_savings_values) if all_savings_values else 1_000_000
        min_savings = min(all_savings_values) if all_savings_values else 1000
        
        # Set dynamic ylim: lower bound at least 1000, upper bound 1.5x max (similar to Panel B's 1.15)
        y_min = max(1000, min_savings * 0.5)  # At least 1000, or half of min if larger
        y_max = max_savings * 1.5  # 1.5x the maximum bar value
        
        # Now create bars and labels
        for i, method in enumerate(dashboard_methods):
            savings_values = all_method_savings[method]
            
            # Position bars: Center is 'site_indices'. Offset by 'i'
            # i=0 (PTrue) -> left, i=1 (PIK) -> center, i=2 (CCPS) -> right
            offset = (i - 1) * width
            rects = ax_finance.bar(site_indices + offset, savings_values, width, 
                                   label=method2path[method]['method_name'],
                                   color=color_auto[method], edgecolor='black', alpha=0.9)
            
            # Label bars (formatted K/M) or "No Auto" if zero
            for j, rect in enumerate(rects):
                height = rect.get_height()
                x_center = rect.get_x() + rect.get_width()/2.
                
                if height > 0:
                    if height >= 1_000_000:
                        label = f'${height/1_000_000:.1f} M'
                    elif height >= 1_000:
                        label = f'${height/1_000:.0f} K'
                    else:
                        label = f'${height:.0f}'
                    
                    # Position label inside the bar at the top, vertical orientation
                    # Use 0.9 of bar height to position near the top but inside
                    ax_finance.text(x_center, height * 0.9,
                                    label, ha='center', va='top', fontsize=16 + font_addition, fontweight='bold', 
                                    rotation=90, color='black')
                else:
                    # Vertical "No Auto" text for zero savings (positioned relative to y_min)
                    no_auto_y = y_min * 1.6  # Position slightly above y_min for visibility
                    ax_finance.text(x_center, no_auto_y, "No Auto", 
                                   ha='center', va='center', style='italic', color='gray', 
                                   fontsize=14 + font_addition, rotation=90, fontweight='normal')

        # Add column title only for first row
        if row_idx == 0:
            ax_finance.set_title('(J) Financial Impact by Clinical Site Size', fontsize=16 + font_addition, fontweight='bold', pad=15)
        
        ax_finance.set_ylabel('Est. Annual Savings (Log Scale)', fontsize=14 + font_addition, fontweight='bold')
        ax_finance.set_xticks(site_indices)
        
        # Only show x-axis labels on bottom row
        if row_idx == n_rows - 1:
            ax_finance.set_xticklabels(sites, fontsize=12 + font_addition, fontweight='bold')
        else:
            ax_finance.set_xticklabels([''] * len(sites))
        
        # Apply font size to tick labels
        ax_finance.tick_params(labelsize=13 + font_addition)
        
        # Custom Y-axis formatter for Log Scale readability (show as 10M, 10K, etc.)
        def format_y_axis(y, pos):
            if y >= 1_000_000:
                return f'${int(y/1_000_000)}M'
            elif y >= 1_000:
                return f'${int(y/1_000)}K'
            else:
                return f'${int(y)}'
        
        ax_finance.yaxis.set_major_formatter(ticker.FuncFormatter(format_y_axis))
        ax_finance.set_ylim(y_min, y_max)  # Dynamic limits based on data
        ax_finance.grid(axis='y', alpha=0.3, which="both", linestyle='--')
        
        # Legend for Panel J - only on first row
        if row_idx == 0:
            ax_finance.legend(handles=legend_handles_auto, loc='upper left', fontsize=11 + font_addition, frameon=True, fancybox=True)
        
        # Add black border for Merged row
        if dataset_type is None:
            for spine in ax_finance.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(3)
        
        # Add sticky note with dataset name
        sticky_note = FancyBboxPatch(
            (0.02, 0.90),
            0.15,
            0.05,
            boxstyle="round,pad=0.01",
            transform=ax_finance.transAxes,
            facecolor="#E8E8E8",
            edgecolor="#A0A0A0",
            linewidth=2,
            alpha=0.9,
            zorder=1000
        )
        ax_finance.add_patch(sticky_note)
        
        ax_finance.text(
            0.095, 0.925,
            dataset_display_name,
            transform=ax_finance.transAxes,
            fontsize=13 + font_addition,
            weight="bold",
            ha="center",
            va="center",
            zorder=1001
        )
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(str(save_path).replace('.png', '.pdf'), bbox_inches='tight')
    print(f"✅ Saved: {save_path}")
    plt.show()
    
    return fig


# Generate combined impact dashboard
impact_dashboard_save_path = OVERALL_RESULTS_PATH / "impact_dashboard_multisite_merged.png"
fig_impact = create_impact_dashboard_combined(
    scenarios_list=scenarios,
    save_path=impact_dashboard_save_path,
    font_addition=0
)


In [ ]:
# ==============================================================================
# IMPACT DASHBOARD - Table Format (All Datasets)
# ==============================================================================

import pandas as pd
import numpy as np

def create_impact_dashboard_table(scenarios_list, save_path):
    """
    Create a comprehensive table with all impact dashboard metrics
    Panel H: Clinical Yield (% of Test Set Automated)
    Panel I: Safety Profile (Actual Test Set Errors)
    Panel J: Financial Impact by Clinical Site Size
    """
    
    print("\n" + "="*80)
    print("🎯 CREATING IMPACT DASHBOARD TABLE (ALL DATASETS)")
    print("="*80)
    
    # Reset index if df_metrics has MultiIndex
    df_metrics_reset = df_metrics.reset_index() if isinstance(df_metrics.index, pd.MultiIndex) else df_metrics
    
    dashboard_configs = [
        {"type": "contextual", "name": "Contextual"},
        {"type": "synthesis", "name": "Synthesis"},
        {"type": "clinical_inference", "name": "Inference"},
        {"type": None, "name": "Merged"}
    ]
    
    # Setup Methods
    dashboard_methods = METHODS_TO_SHOW
    
    # Prepare table rows
    table_rows = []
    
    for config in dashboard_configs:
        dataset_type = config["type"]
        dataset_display_name = config["name"]
        
        # -------------------------------------------------------------------------
        # 1. DATA PREP (REAL TEST SET METRICS)
        # -------------------------------------------------------------------------
        
        # Calculate Real N (Average per LLM)
        real_n_samples = 0
        datasets_to_use = dataset_types if dataset_type is None else [dataset_type]
        
        for dt in datasets_to_use:
            for llm in llm_ids:
                if 'CCPS' in llm2method2dataset2results.get(llm, {}):
                    if dt in llm2method2dataset2results[llm]['CCPS']:
                        shared = llm2dataset2sharedRIDacrossMethods[llm].get(dt, set())
                        count = len(shared) if shared else len(llm2method2dataset2results[llm]['CCPS'][dt])
                        real_n_samples += count
        
        real_n_samples = int(real_n_samples / len(llm_ids)) 
        
        print(f"\nProcessing: {dataset_display_name}")
        print(f"ACTUAL Test Set Size (N): {real_n_samples:,}")
        
        # Calculate Aggregate Metrics per Method
        method_metrics = {}
        
        for method in dashboard_methods:
            automations, accuracies, aurocs = [], [], []
            
            for dt in datasets_to_use:
                llm_data = threshold_results.get(dt, {})
                for llm_results in llm_data.values():
                    if method in llm_results:
                        res = llm_results[method]
                        if res['meets_criterion']:
                            automations.append(res['pct_automated'])
                            accuracies.append(res['accuracy_at_tau'])
                        else:
                            automations.append(0.0)
            
            # Get AUROC from df_metrics
            for dt in datasets_to_use:
                if 'method' in df_metrics_reset.columns and 'dataset_type' in df_metrics_reset.columns:
                    method_df = df_metrics_reset[(df_metrics_reset['method'] == method) & (df_metrics_reset['dataset_type'] == dt)]
                    if len(method_df) > 0 and 'auroc' in method_df.columns:
                        aurocs.extend(method_df['auroc'].values.tolist())
            
            avg_pct_auto = np.mean(automations) if automations else 0.0
            avg_acc = np.mean([a for a in accuracies if a > 0]) if accuracies else 0.0
            avg_auroc = np.mean(aurocs) if aurocs else 0.0
            
            # Real Test Set Counts
            real_auto_count = int(real_n_samples * (avg_pct_auto / 100.0))
            real_correct = int(real_auto_count * avg_acc)
            real_error = real_auto_count - real_correct
            
            # Calculate financial savings for each site
            savings_small = 0.0
            savings_medium = 0.0
            savings_large = 0.0
            
            if len(scenarios_list) >= 3:
                for i, scen in enumerate(scenarios_list):
                    cost_per_case = (scen["Wage_Basis"] / 60.0) * 2.0
                    savings = scen["Annual_Volume"] * (avg_pct_auto / 100.0) * cost_per_case
                    if i == 0:
                        savings_small = savings
                    elif i == 1:
                        savings_medium = savings
                    elif i == 2:
                        savings_large = savings
            
            method_metrics[method] = {
                'pct_auto': avg_pct_auto,
                'real_auto': real_auto_count,
                'real_correct': real_correct,
                'real_error': real_error,
                'auroc': avg_auroc,
                'error_pct': (real_error / real_auto_count * 100.0) if real_auto_count > 0 else 0.0,
                'savings_small': savings_small,
                'savings_medium': savings_medium,
                'savings_large': savings_large,
            }
        
        # Find best values per dataset (for marking with asterisk)
        best_pct_auto = max([method_metrics[m]['pct_auto'] for m in dashboard_methods])
        
        # For safety metrics, only consider methods with automated cases
        methods_with_auto = [m for m in dashboard_methods if method_metrics[m]['real_auto'] > 0]
        if methods_with_auto:
            best_auroc = max([method_metrics[m]['auroc'] for m in methods_with_auto])
            best_error_pct = min([method_metrics[m]['error_pct'] for m in methods_with_auto])
        else:
            best_auroc = 0.0
            best_error_pct = 100.0
        
        best_savings_small = max([method_metrics[m]['savings_small'] for m in dashboard_methods])
        best_savings_medium = max([method_metrics[m]['savings_medium'] for m in dashboard_methods])
        best_savings_large = max([method_metrics[m]['savings_large'] for m in dashboard_methods])
        
        # Create rows for this dataset
        for method in dashboard_methods:
            d = method_metrics[method]
            method_display_name = method2path[method]['method_name']
            
            # Format financial values
            def format_currency(val, is_best=False):
                if val >= 1_000_000:
                    formatted = f"${val/1_000_000:.2f}M"
                elif val >= 1_000:
                    formatted = f"${val/1_000:.1f}K"
                else:
                    formatted = f"${val:.0f}"
                return f"{formatted}*" if is_best else formatted
            
            # Determine which values are best (using small epsilon for floating point comparison)
            epsilon = 0.01
            is_best_pct_auto = abs(d['pct_auto'] - best_pct_auto) < epsilon
            is_best_auroc = (d['real_auto'] > 0 and abs(d['auroc'] - best_auroc) < epsilon) if methods_with_auto else False
            is_best_error_pct = (d['real_auto'] > 0 and abs(d['error_pct'] - best_error_pct) < epsilon) if methods_with_auto else False
            is_best_savings_small = abs(d['savings_small'] - best_savings_small) < epsilon
            is_best_savings_medium = abs(d['savings_medium'] - best_savings_medium) < epsilon
            is_best_savings_large = abs(d['savings_large'] - best_savings_large) < epsilon
            
            row = {
                'Dataset': dataset_display_name,
                'Method': method_display_name,
                'Total N': f"{real_n_samples:,}",
                # Panel H: Clinical Yield
                '% Auto': f"{d['pct_auto']:.1f}{'*' if is_best_pct_auto else ''}",
                # Combined count: Automated (Correct - Error)
                'Count Auto': f"{d['real_auto']:,} ({d['real_correct']:,} - {d['real_error']:,})",
                # Panel I: Safety Profile
                'AUROC': f"{d['auroc']:.3f}{'*' if is_best_auroc else ''}",
                'Error %': f"{d['error_pct']:.1f}{'*' if is_best_error_pct else ''}",
                # Panel J: Financial Impact
                'Small': format_currency(d['savings_small'], is_best_savings_small),
                'Medium': format_currency(d['savings_medium'], is_best_savings_medium),
                'Large': format_currency(d['savings_large'], is_best_savings_large),
            }
            
            table_rows.append(row)
    
    # Create DataFrame
    df_table = pd.DataFrame(table_rows)
    
    # Reorder columns for better readability
    column_order = [
        'Dataset',
        'Total N',
        'Method',
        '% Auto',
        'Count Auto',
        'AUROC',
        'Error %',
        'Small',
        'Medium',
        'Large',
    ]
    
    df_table = df_table[column_order]
    
    # Print formatted table
    print("\n" + "="*120)
    print("IMPACT DASHBOARD TABLE")
    print("="*120)
    print("\n")
    
    # Print table by dataset
    for dataset_name in df_table['Dataset'].unique():
        print(f"\n{'='*120}")
        print(f"DATASET: {dataset_name}")
        print('='*120)
        df_subset = df_table[df_table['Dataset'] == dataset_name].drop('Dataset', axis=1)
        
        # Print header
        header = " | ".join([f"{col:<20}" for col in df_subset.columns])
        print(header)
        print("-" * len(header))
        
        # Print rows
        for idx, row in df_subset.iterrows():
            row_str = " | ".join([f"{str(val):<20}" for val in row.values])
            print(row_str)
    
    print("\n" + "="*120)
    
    # Save to CSV
    df_table.to_csv(save_path, index=False)
    print(f"\n✅ Saved table to {save_path}")
    
    # Also save a more detailed version with numeric values for calculations
    save_path_detailed = str(save_path).replace('.csv', '_detailed.csv')
    
    # Create detailed version with numeric values
    table_rows_detailed = []
    for config in dashboard_configs:
        dataset_type = config["type"]
        dataset_display_name = config["name"]
        
        real_n_samples = 0
        datasets_to_use = dataset_types if dataset_type is None else [dataset_type]
        
        for dt in datasets_to_use:
            for llm in llm_ids:
                if 'CCPS' in llm2method2dataset2results.get(llm, {}):
                    if dt in llm2method2dataset2results[llm]['CCPS']:
                        shared = llm2dataset2sharedRIDacrossMethods[llm].get(dt, set())
                        count = len(shared) if shared else len(llm2method2dataset2results[llm]['CCPS'][dt])
                        real_n_samples += count
        
        real_n_samples = int(real_n_samples / len(llm_ids))
        
        for method in dashboard_methods:
            automations, accuracies = [], []
            
            for dt in datasets_to_use:
                llm_data = threshold_results.get(dt, {})
                for llm_results in llm_data.values():
                    if method in llm_results:
                        res = llm_results[method]
                        if res['meets_criterion']:
                            automations.append(res['pct_automated'])
                            accuracies.append(res['accuracy_at_tau'])
                        else:
                            automations.append(0.0)
            
            avg_pct_auto = np.mean(automations) if automations else 0.0
            avg_acc = np.mean([a for a in accuracies if a > 0]) if accuracies else 0.0
            
            real_auto_count = int(real_n_samples * (avg_pct_auto / 100.0))
            real_correct = int(real_auto_count * avg_acc)
            real_error = real_auto_count - real_correct
            human_count = real_n_samples - real_auto_count
            
            savings_small = 0.0
            savings_medium = 0.0
            savings_large = 0.0
            
            if len(scenarios_list) >= 3:
                for i, scen in enumerate(scenarios_list):
                    cost_per_case = (scen["Wage_Basis"] / 60.0) * 2.0
                    savings = scen["Annual_Volume"] * (avg_pct_auto / 100.0) * cost_per_case
                    if i == 0:
                        savings_small = savings
                    elif i == 1:
                        savings_medium = savings
                    elif i == 2:
                        savings_large = savings
            
            # Get AUROC for detailed table
            aurocs_detailed = []
            for dt in datasets_to_use:
                if 'method' in df_metrics_reset.columns and 'dataset_type' in df_metrics_reset.columns:
                    method_df = df_metrics_reset[(df_metrics_reset['method'] == method) & (df_metrics_reset['dataset_type'] == dt)]
                    if len(method_df) > 0 and 'auroc' in method_df.columns:
                        aurocs_detailed.extend(method_df['auroc'].values.tolist())
            avg_auroc_detailed = np.mean(aurocs_detailed) if aurocs_detailed else 0.0
            
            row_detailed = {
                'Dataset': dataset_display_name,
                'Method': method2path[method]['method_name'],
                'Total_N': real_n_samples,
                'Pct_Automated': avg_pct_auto,
                'Count_Automated': real_auto_count,
                'Correct_Count': real_correct,
                'Error_Count': real_error,
                'AUROC': avg_auroc_detailed,
                'Error_Pct': (real_error / real_auto_count * 100.0) if real_auto_count > 0 else 0.0,
                'Savings_Small_Site': savings_small,
                'Savings_Medium_Site': savings_medium,
                'Savings_Large_Site': savings_large,
            }
            
            table_rows_detailed.append(row_detailed)
    
    df_detailed = pd.DataFrame(table_rows_detailed)
    df_detailed.to_csv(save_path_detailed, index=False)
    print(f"✅ Saved detailed table (numeric values) to {save_path_detailed}")
    
    return df_table


# Generate impact dashboard table
impact_table_save_path = OVERALL_RESULTS_PATH / "table_impact_dashboard.csv"
df_impact_table = create_impact_dashboard_table(
    scenarios_list=scenarios,
    save_path=impact_table_save_path
)

In [ ]:
# ==============================================================================
# THREE-PANEL IMPACT DASHBOARD (REAL DATA + MULTI-SCENARIO FINANCIALS)
# 6-ROW VERSION: 2x, 1x, 0.5x Human Performance @ 90% and 80% Accuracy
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("\n" + "="*80)
print("🎯 CREATING IMPACT DASHBOARD (REAL METRICS + MULTI-SITE FINANCIALS)")
print("="*80)

# ==============================================================================
# HELPER FUNCTION: Calculate thresholds for arbitrary error rate constraints
# ==============================================================================

def calculate_threshold_for_max_error(
    results, 
    confidence_scores, 
    max_error_rate=5.0,  # Maximum allowed error rate (%)
    thresholds=np.arange(0.0, 1.0, 0.01)
):
    """
    Find optimal confidence threshold where AI Error Rate <= max_error_rate.
    
    Args:
        results: List of result dicts with 'ground_truth_correctness'
        confidence_scores: List of confidence scores (same order as results)
        max_error_rate: Maximum allowed error rate in percentage (e.g., 5.0 for 5%)
        thresholds: Array of confidence thresholds to test
    
    Returns:
        Dictionary with:
            - tau: Optimal threshold (None if criterion not met)
            - pct_automated: % of cases with confidence >= tau
            - accuracy_at_tau: Accuracy at this threshold
            - target_accuracy: The minimum accuracy we aimed for
            - meets_criterion: Whether criterion was met
    """
    
    target_accuracy = 1.0 - (max_error_rate / 100.0)
    
    data = list(zip(results, confidence_scores))
    total_samples = len(data)
    
    if total_samples == 0:
        return {
            'tau': None, 'pct_automated': 0.0, 'accuracy_at_tau': 0.0,
            'target_accuracy': target_accuracy, 'meets_criterion': False
        }

    optimal_tau = None
    optimal_pct_automated = 0.0
    optimal_accuracy = 0.0
    
    for tau in sorted(thresholds): 
        high_conf_subset = [(r, c) for r, c in data if c >= tau]
        
        if len(high_conf_subset) < 10:
            continue
            
        high_conf_gt = np.array([r['ground_truth_correctness'] for r, c in high_conf_subset])
        subset_accuracy = np.mean(high_conf_gt)
        
        if subset_accuracy >= target_accuracy:
            optimal_tau = tau
            optimal_pct_automated = (len(high_conf_subset) / total_samples) * 100
            optimal_accuracy = subset_accuracy
            break
            
    meets_criterion = optimal_tau is not None
    
    return {
        'tau': optimal_tau,
        'pct_automated': optimal_pct_automated,
        'accuracy_at_tau': optimal_accuracy,
        'target_accuracy': target_accuracy,
        'meets_criterion': meets_criterion
    }

# ==============================================================================
# MAIN DASHBOARD FUNCTION
# ==============================================================================

def create_impact_dashboard(dataset_type, dataset_display_name, scenarios_list, save_path):
    """
    Panel A: Actual Test Set Yield (Real N)
    Panel B: Actual Test Set Safety (Real N)
    Panel C: Projected Savings across 3 Different Clinical Sites (Grouped Bar)
    
    Now with 6 rows:
    - Row 1: 2x better than 90% human (≤2.5% error)
    - Row 2: As good as 90% human (≤5% error)
    - Row 3: Half as good as 90% human (≤10% error)
    - Row 4: 2x better than 80% human (≤10% error)
    - Row 5: As good as 80% human (≤20% error)
    - Row 6: Half as good as 80% human (≤40% error)
    """
    
    # -------------------------------------------------------------------------
    # 1. DATA PREP (REAL TEST SET METRICS)
    # -------------------------------------------------------------------------
    
    # Calculate Real N (Average per LLM)
    real_n_samples = 0
    datasets_to_use = dataset_types if dataset_type is None else [dataset_type]
    
    for dt in datasets_to_use:
        for llm in llm_ids:
            if 'CCPS' in llm2method2dataset2results.get(llm, {}):
                if dt in llm2method2dataset2results[llm]['CCPS']:
                    shared = llm2dataset2sharedRIDacrossMethods[llm].get(dt, set())
                    count = len(shared) if shared else len(llm2method2dataset2results[llm]['CCPS'][dt])
                    real_n_samples += count
    
    real_n_samples = int(real_n_samples / len(llm_ids)) 
    
    print(f"\nProcessing: {dataset_display_name}")
    print(f"ACTUAL Test Set Size (N): {real_n_samples:,}")
    
    # Setup Methods & Colors
    dashboard_methods = METHODS_TO_SHOW
    color_auto = {m: method2path[m]['base_color'] for m in dashboard_methods}
    color_human = '#E0E0E0' 
    
    # Define 6 scenarios
    scenarios_config = [
        {'label': '2x Better (90%)', 'human_acc': 0.90, 'multiplier': 2.0, 'max_error': 2.5},
        {'label': 'As Good (90%)', 'human_acc': 0.90, 'multiplier': 1.0, 'max_error': 5.0},
        {'label': '0.5x (90%)', 'human_acc': 0.90, 'multiplier': 0.5, 'max_error': 10.0},
        {'label': '2x Better (80%)', 'human_acc': 0.80, 'multiplier': 2.0, 'max_error': 10.0},
        {'label': 'As Good (80%)', 'human_acc': 0.80, 'multiplier': 1.0, 'max_error': 20.0},
        {'label': '0.5x (80%)', 'human_acc': 0.80, 'multiplier': 0.5, 'max_error': 40.0},
    ]
    
    # -------------------------------------------------------------------------
    # RECALCULATE THRESHOLDS FOR EACH SCENARIO
    # -------------------------------------------------------------------------
    print(f"\n📊 Recalculating thresholds for all 6 scenarios...")
    
    all_scenario_metrics = []
    
    for scenario_idx, scenario_cfg in enumerate(scenarios_config):
        print(f"\n  Scenario {scenario_idx + 1}: {scenario_cfg['label']} (max error: {scenario_cfg['max_error']}%)")
        method_metrics = {}
        
        for method in dashboard_methods:
            automations, accuracies = [], []
            
            for dt in datasets_to_use:
                for llm_id in llm_ids:
                    if method not in llm2method2dataset2results.get(llm_id, {}):
                        continue
                    if dt not in llm2method2dataset2results[llm_id].get(method, {}):
                        continue
                    
                    results = llm2method2dataset2results[llm_id][method][dt]
                    shared_ids = llm2dataset2sharedRIDacrossMethods.get(llm_id, {}).get(dt, set())
                    
                    # Filter to shared IDs
                    if shared_ids:
                        filtered_results = [r for r in results if r["record_id"] in shared_ids]
                    else:
                        filtered_results = results
                    
                    if not filtered_results:
                        continue
                    
                    confidence_scores = [r["confidence_score"] for r in filtered_results]
                    
                    # RECALCULATE threshold for THIS scenario's max error
                    threshold_result = calculate_threshold_for_max_error(
                        filtered_results,
                        confidence_scores,
                        max_error_rate=scenario_cfg['max_error']
                    )
                    
                    if threshold_result['meets_criterion']:
                        automations.append(threshold_result['pct_automated'])
                        accuracies.append(threshold_result['accuracy_at_tau'])
                    else:
                        automations.append(0.0)
            
            avg_pct_auto = np.mean(automations) if automations else 0.0
            avg_acc = np.mean([a for a in accuracies if a > 0]) if accuracies else 0.0
            
            # Real Test Set Counts
            real_auto_count = int(real_n_samples * (avg_pct_auto / 100.0))
            real_correct = int(real_auto_count * avg_acc)
            real_error = real_auto_count - real_correct
            
            method_metrics[method] = {
                'pct_auto': avg_pct_auto,
                'real_auto': real_auto_count,
                'real_correct': real_correct,
                'real_error': real_error,
                'error_rate': (1 - avg_acc)*100 if real_auto_count > 0 else 0.0,
            }
            
            print(f"    {method}: {avg_pct_auto:.1f}% automation, {avg_acc*100:.1f}% accuracy")
        
        all_scenario_metrics.append(method_metrics)
    
    # -------------------------------------------------------------------------
    # 2. PLOTTING (6 ROWS x 3 COLUMNS)
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=(22, 35))
    gs = fig.add_gridspec(6, 3, width_ratios=[1, 1, 1.4], hspace=0.15, wspace=0.25)
    
    x_pos = np.arange(len(dashboard_methods))
    bar_width = 0.6
    font_size_metrics = 14
    
    sites = ["Small", "Medium", "Large"]
    site_indices = np.arange(len(sites))
    width = 0.25
    
    # Iterate through 6 rows
    for row_idx, (scenario_cfg, method_metrics) in enumerate(zip(scenarios_config, all_scenario_metrics)):
        
        ax_auto = fig.add_subplot(gs[row_idx, 0])
        ax_safety = fig.add_subplot(gs[row_idx, 1])
        ax_finance = fig.add_subplot(gs[row_idx, 2])
        
        # --- PANEL A: AUTOMATION YIELD (%) ---
        for i, method in enumerate(dashboard_methods):
            d = method_metrics[method]
            human_pct = 100.0 - d['pct_auto']
            human_count = int(real_n_samples * (human_pct / 100.0))
            
            ax_auto.bar(i, d['pct_auto'], bar_width, color=color_auto[method], edgecolor='black', alpha=0.9)
            ax_auto.bar(i, human_pct, bar_width, bottom=d['pct_auto'], color=color_human, edgecolor='black', hatch='///', alpha=0.6)
            
            if d['pct_auto'] > 5:
                ax_auto.text(i, d['pct_auto']/2, f"{d['pct_auto']:.1f}%\n({d['real_auto']:,})", 
                             ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='white')
            
            if human_pct > 5:
                ax_auto.text(i, d['pct_auto'] + human_pct/2, f"{human_pct:.1f}%\n({human_count:,})", 
                             ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='black')
            
            ax_auto.text(i, 102, f"{real_n_samples:,}", ha='center', va='bottom', fontsize=12, fontweight='bold', color='black')
        
        # Remove y-axis label and add centered text above the row
        ax_auto.set_ylabel('')
        
        # Add centered label above the row using axes bbox for accurate positioning
        bbox = ax_auto.get_position()
        fig.text(0.5, bbox.y1 + 0.005, scenario_cfg['label'], 
                ha='center', va='bottom', fontsize=14, fontweight='bold', transform=fig.transFigure)
        
        if row_idx == 0:
            ax_auto.set_title('(H) Clinical Yield (% of Test Set Automated)', fontsize=16, fontweight='bold', pad=15, y=1.07)
        
        ax_auto.set_ylim(0, 115)
        ax_auto.set_xticks(x_pos)
        
        if row_idx == 5:
            ax_auto.set_xticklabels([method2path[m]['method_name'] for m in dashboard_methods], fontsize=13, fontweight='bold')
        else:
            ax_auto.set_xticklabels([])
        
        if row_idx == 0:
            leg_human = mpatches.Patch(facecolor=color_human, edgecolor='black', hatch='///', label='Human Review')
            ax_auto.legend(handles=[leg_human], loc='upper center', fontsize=11, frameon=True, fancybox=True)
        
        # --- PANEL B: SAFETY (REAL COUNTS) ---
        max_real_cases = max([d['real_auto'] for d in method_metrics.values()]) if method_metrics else 100
        
        for i, method in enumerate(dashboard_methods):
            d = method_metrics[method]
            
            ax_safety.bar(i, d['real_correct'], bar_width, color=color_auto[method], edgecolor='black', alpha=0.9)
            ax_safety.bar(i, d['real_error'], bar_width, bottom=d['real_correct'], color='#000000', edgecolor='black', alpha=0.9)
            
            total_auto = d['real_auto']
            if total_auto > 0:
                correct_pct = (d['real_correct'] / total_auto) * 100.0
                error_pct = (d['real_error'] / total_auto) * 100.0
            else:
                correct_pct = 0.0
                error_pct = 0.0
            
            if d['real_correct'] > 0:
                ax_safety.text(i, d['real_correct']/2, f"{correct_pct:.1f}%\n({d['real_correct']:,})", 
                               ha='center', va='center', fontweight='bold', fontsize=font_size_metrics, color='white')
            
            if d['real_error'] > 0:
                y_txt = d['real_correct'] + d['real_error'] + (max_real_cases * 0.02)
                ax_safety.text(i, y_txt, f"{error_pct:.1f}%\n({d['real_error']:,})", 
                               ha='center', va='bottom', fontweight='bold', fontsize=font_size_metrics, color='#000000')
            elif d['pct_auto'] == 0:
                ax_safety.text(i, 5, "No Auto", ha='center', va='bottom', style='italic', color='gray')
        
        if row_idx == 0:
            ax_safety.set_title('(I) Safety Profile (Actual Test Set Errors)', fontsize=16, fontweight='bold', pad=15, y=1.07)

        ax_safety.set_ylim(0, max_real_cases * 1.15)
        ax_safety.set_xticks(x_pos)
        
        if row_idx == 5:
            ax_safety.set_xticklabels([method2path[m]['method_name'] for m in dashboard_methods], fontsize=13, fontweight='bold')
        else:
            ax_safety.set_xticklabels([])
        
        if row_idx == 0:
            leg_b = [mpatches.Patch(facecolor='#000000', label='Error')]
            ax_safety.legend(handles=leg_b, loc='upper left', frameon=True)
        
        # --- PANEL C: FINANCIAL (MULTI-SCENARIO GROUPED BAR) ---
        ax_finance.set_yscale('log')
        
        all_savings_values = []
        all_method_savings = {}
        
        for i, method in enumerate(dashboard_methods):
            d = method_metrics[method]
            pct_auto = d['pct_auto']
            savings_values = []
            
            for scen in scenarios_list:
                cost_per_case = (scen["Wage_Basis"] / 60.0) * 2.0
                savings = scen["Annual_Volume"] * (pct_auto / 100.0) * cost_per_case
                savings_values.append(savings)
                if savings > 0:
                    all_savings_values.append(savings)
            
            all_method_savings[method] = savings_values
        
        max_savings = max(all_savings_values) if all_savings_values else 1_000_000
        min_savings = min(all_savings_values) if all_savings_values else 1000
        
        y_min = max(1000, min_savings * 0.5)
        y_max = max_savings * 1.5
        
        for i, method in enumerate(dashboard_methods):
            savings_values = all_method_savings[method]
            offset = (i - 1) * width
            rects = ax_finance.bar(site_indices + offset, savings_values, width, 
                                   label=method2path[method]['method_name'],
                                   color=color_auto[method], edgecolor='black', alpha=0.9)
            
            for j, rect in enumerate(rects):
                height = rect.get_height()
                x_center = rect.get_x() + rect.get_width()/2.
                
                if height > 0:
                    if height >= 1_000_000:
                        label = f'{height/1_000_000:.1f} M'
                    elif height >= 1_000:
                        label = f'{height/1_000:.0f} K'
                    else:
                        label = f'{height:.0f}'
                    
                    ax_finance.text(x_center, height * 1.05,
                                    label, ha='center', va='bottom', fontsize=13, fontweight='bold', 
                                    color='black')
                else:
                    no_auto_y = y_min * 1.6
                    ax_finance.text(x_center, no_auto_y, "No Auto", 
                                   ha='center', va='center', style='italic', color='gray', 
                                   fontsize=14, rotation=90, fontweight='normal')
        
        if row_idx == 0:
            ax_finance.set_title('(J) Financial Impact by Clinical Site Size', fontsize=16, fontweight='bold', pad=15, y=1.07)
        
        import matplotlib.ticker as ticker
        def format_y_axis(y, pos):
            if y >= 1_000_000:
                return f'{int(y/1_000_000)} M'
            elif y >= 1_000:
                return f'{int(y/1_000)} K'
            else:
                return f'{int(y)}'
        
        ax_finance.yaxis.set_major_formatter(ticker.FuncFormatter(format_y_axis))
        ax_finance.set_ylim(y_min, y_max)
        ax_finance.grid(axis='y', alpha=0.3, which="both", linestyle='--')
        ax_finance.set_xticks(site_indices)
        
        if row_idx == 5:
            ax_finance.set_xticklabels(sites, fontsize=12, fontweight='bold')
        else:
            ax_finance.set_xticklabels([])
        
        if row_idx == 0:
            ax_finance.legend(loc='upper left', fontsize=11, frameon=True, fancybox=True)
    
    # Overall title
    fig.text(
        0.5, 0.91,
        f"Impact Dashboard: {dataset_display_name}",
        ha="right",
        va="top",
        fontsize=20,
        fontweight="bold"
    )
    
    fig.text(
        0.5, 0.91,
        f" - Observed Performance (N={real_n_samples:,}) vs. Multi-Site Projections",
        ha="left",
        va="top",
        fontsize=20
    )
    
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(str(save_path).replace('.png', '.pdf'), bbox_inches='tight')
    print(f"✅ Saved: {save_path}")
    plt.show()

# ==============================================================================
# GENERATE
# ==============================================================================
all_scenarios = scenarios 

dashboard_configs = [
    {"type": "contextual", "name": "CORAL-MCQA-Contextual"},
    {"type": "synthesis", "name": "CORAL-MCQA-Synthesis"},
    {"type": "clinical_inference", "name": "CORAL-MCQA-Clinical-Inference"},
    {"type": None, "name": "All Datasets (Merged)"}
]

for config in dashboard_configs:
    dataset_type = config["type"]
    if dataset_type is None: suffix = "merged"
    else: suffix = dataset_type
    
    save_path = GHASSEMI_FIGURES_PATH / f"impact_dashboard_multisite_{suffix}.png"
    create_impact_dashboard(dataset_type, config["name"], all_scenarios, save_path)

# Clinical Utility & Financial Impact Analysis

### 1. Executive Summary

This analysis quantifies the **operational impact** of deploying the CCPS confidence estimation method compared to standard baselines (P(True) and P(IK)). We effectively translate "model accuracy" into "hospital ROI" by modeling three realistic deployment scenarios based on **2025 CMS Physician Fee Schedule** data.

**The "Nail in the Coffin" Finding:**
On the *Clinical Inference* dataset, CCPS safely offloads **50.1%** of the workload—more than double the standard baseline—while maintaining an error rate of **~2.1%** (far below the 5% safety limit). At a national network scale, this efficiency gain translates to **$2.7 Million in annual gross savings** (equivalent to ~11,000 hours of specialist capacity).

---

### 2. Methodology & Assumptions (The "Fine Print")

*Anticipating questions on how values were derived:*

#### A. Financial Basis (CMS 2025)

All cost projections are grounded in the **2025 CMS Physician Fee Schedule** for CPT Code **99205** (*High Complexity Medical Decision Making*).

* **Hourly Wage:** Derived from the CMS National Payment Amount, standardized to a 60-minute encounter.
* **Private Practice:** $236.81/hr (Non-Facility Price).
* **Hospital/Academic:** $160.32/hr (Facility Price).


* **Review Speed:** We assume a conservative **2.0 minutes per case** for a human oncologist to review/verify a record.
* **Unit Cost:**
* *Private:* **$7.89/case**.
* *Hospital:* **$5.34/case**.



#### B. Safety Thresholding ()

We do not optimize for raw accuracy. Instead, we solve for the specific confidence threshold () where the model guarantees safety **2x better than the human baseline**.

* **Human Benchmark:** 90% Accuracy (10% Error Rate).
* **AI Safety Constraint:** Error Rate  5%.
* **Constraint Logic:** If a model cannot meet >95% precision on *any* subset of data, its automation rate is clamped to 0%.

---

### 3. Impact Dashboard: Figure Explanation

*Refer to the "Impact Dashboard" plots generated in the notebook.*

#### Panel A: Clinical Yield (Real Test Data)

**"How much work can the AI actually handle?"**

* **Data Source:** Actual Test Set (N = Real Sample Count).
* **Metrics:** The percentage of test cases where model confidence .
* **Interpretation:** The **green bar** represents the volume of "easy" cases the AI successfully identified and automated. The **gray hatched area** represents cases sent to human review because the AI was uncertain.
* *Key Result:* On Clinical Inference, CCPS automates **50.1%** (188/377 cases), whereas P(True) only automates 23.5%.



#### Panel B: Safety Profile (Real Test Errors)

**"Did we hurt anyone in the process?"**

* **Data Source:** Actual Test Set (Subset of Automations).
* **Metrics:**
* **Green (Correct):** The AI was confident *and* right.
* **Red (Error):** The AI was confident *but* wrong.


* **Interpretation:** This visually proves safety. The red sliver represents the **Clinical Risk**.
* *Key Result:* CCPS committed only **4 errors** out of 188 automated decisions (2.1% Error Rate), strictly adhering to the <5% safety limit.



#### Panel C: Financial Projection (Scaled Impact)

**"What is this worth to the hospital?"**

* **Data Source:** Extrapolation of Panel A's yield to annual volumes.
* **Calculation:** `(Annual Volume × % Automated) × Unit Cost`.
* **Scenarios:**
1. **Private Specialist (Small):** 10k cases/yr ($7.89/case).
2. **Academic Center (Medium):** 50k cases/yr ($5.34/case).
3. **Integrated Network (National):** 1M cases/yr ($5.34/case).


* **Interpretation:** The logarithmic scale highlights that CCPS provides superior savings at every level of scale. The **$2.7M** figure for National Networks represents "found capacity"—eliminating the need for human review on 500,000+ routine cases.

---

### 4. Critical Nuance: "Hard" Tasks

The analysis of **Contextual Reasoning** and **Synthesis** datasets reveals a critical failure mode for standard methods:

* **The Failure:** The baseline **P(True)** method failed to automate a single case (0.0% yield) on these harder datasets because it could not distinguish safe from unsafe predictions effectively.
* **The CCPS Advantage:** CCPS successfully recovered **14.5% – 35.7%** of these complex cases for safe automation. This proves CCPS is not just optimizing thresholds, but fundamentally better at measuring uncertainty in complex reasoning tasks.

### 5. Limitations

* **Gross vs. Net:** Financial figures represent **gross operational savings** (labor hours repurpose). They do not subtract the downstream cost of reworking the <5% of cases where the AI might err.
* **Review Time:** Savings are linearly dependent on the "2-minute" human review assumption. If the human task is faster (e.g., 30s), savings decrease proportionally; if slower (e.g., 5m), savings increase.

In [ ]:
import numpy as np
import pandas as pd

print("\n" + "="*80)
print("📊 BUILDING FINAL IMPACT TABLE (MATCHING DASHBOARD - FIXED)")
print("="*80)

HUMAN_MINUTES_PER_CASE = 2.0

largest_scenario = scenarios[2]
print(f"Using scenario: {largest_scenario['Setting']} ({largest_scenario['Annual_Volume']:,} cases/year)")

datasets_configs = [
    ("contextual", "Contextual"),
    ("synthesis", "Synthesis"),
    ("clinical_inference", "Clinical Inference"),
]

methods = list(method2path.keys())
rows = []

def mean_std(arr):
    if len(arr) == 0:
        return 0, 0
    return np.mean(arr), np.std(arr)

for dataset_type, dataset_name in datasets_configs:

    llm_n_samples = {}
    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        if 'CCPS' in llm2method2dataset2results.get(llm_id, {}):
            if dataset_type in llm2method2dataset2results[llm_id]['CCPS']:
                shared = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
                results = llm2method2dataset2results[llm_id]['CCPS'][dataset_type]
                count = len(shared) if shared else len(results)
                llm_n_samples[llm_display] = count

    mean_n, std_n = mean_std(list(llm_n_samples.values()))

    for method in methods:

        automations = []
        auto_counts = []
        correct_counts = []
        error_counts = []
        err_rates = []

        llm_data = threshold_results.get(dataset_type, {})

        for llm_display, llm_results in llm_data.items():
            if method in llm_results:
                res = llm_results[method]
                llm_n = llm_n_samples.get(llm_display, 0)

                if res['meets_criterion'] and llm_n > 0:
                    pct_auto = res['pct_automated']
                    acc = res['accuracy_at_tau']

                    automations.append(pct_auto)

                    auto = llm_n * pct_auto / 100
                    correct = auto * acc
                    error = auto - correct

                    auto_counts.append(auto)
                    correct_counts.append(correct)
                    error_counts.append(error)
                    err_rates.append((1 - acc) * 100)
                else:
                    automations.append(0.0)
                    auto_counts.append(0)
                    correct_counts.append(0)
                    error_counts.append(0)

        med_auto, std_auto = mean_std(automations)
        med_auto_count, std_auto_count = mean_std(auto_counts)
        med_correct, std_correct = mean_std(correct_counts)
        med_error, std_error = mean_std(error_counts)
        med_err_rate, std_err_rate = mean_std(err_rates)

        cost_per_case_scenario = (largest_scenario["Wage_Basis"] / 60.0) * HUMAN_MINUTES_PER_CASE
        projected_auto_cases = largest_scenario["Annual_Volume"] * (med_auto / 100.0)
        projected_savings = projected_auto_cases * cost_per_case_scenario
        projected_hours = (projected_auto_cases * HUMAN_MINUTES_PER_CASE) / 60.0

        rows.append({
            "Dataset": dataset_name,
            "Method": method2path[method]['method_name'],
            "Test Set Size": f"{mean_n:.0f} ± {std_n:.0f}",
            "% Automated": f"{med_auto:.0f} ± {std_auto:.0f}",
            "Automated Cases (Test)": f"{med_auto_count:.0f} ± {std_auto_count:.0f}",
            "Correct (Test)": f"{med_correct:.0f} ± {std_correct:.0f}",
            "Errors (Test)": f"{med_error:.0f} ± {std_error:.0f}",
            "Error Rate (%)": f"{med_err_rate:.0f} ± {std_err_rate:.0f}",
            "Hours Saved": f"{projected_hours/1000:.1f}K" if projected_hours > 0 else "0",
            "Dollars Saved": f"${projected_savings/1_000_000:.2f}M" if projected_savings > 0 else "$0"
        })


# ---------------- MERGED ----------------
for method in methods:
    all_automations = []
    all_auto_counts = []
    all_correct_counts = []
    all_error_counts = []
    all_err_rates = []
    all_n_values = []

    for dataset_type in ["contextual", "synthesis", "clinical_inference"]:

        llm_n_samples = {}
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            if 'CCPS' in llm2method2dataset2results.get(llm_id, {}):
                if dataset_type in llm2method2dataset2results[llm_id]['CCPS']:
                    shared = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
                    results = llm2method2dataset2results[llm_id]['CCPS'][dataset_type]
                    count = len(shared) if shared else len(results)
                    llm_n_samples[llm_display] = count

        all_n_values.extend(llm_n_samples.values())

        llm_data = threshold_results.get(dataset_type, {})

        for llm_display, llm_results in llm_data.items():
            if method in llm_results:
                res = llm_results[method]
                llm_n = llm_n_samples.get(llm_display, 0)

                if res['meets_criterion'] and llm_n > 0:
                    pct_auto = res['pct_automated']
                    acc = res['accuracy_at_tau']

                    all_automations.append(pct_auto)

                    auto = llm_n * pct_auto / 100
                    correct = auto * acc
                    error = auto - correct

                    all_auto_counts.append(auto)
                    all_correct_counts.append(correct)
                    all_error_counts.append(error)
                    all_err_rates.append((1 - acc) * 100)
                else:
                    all_automations.append(0.0)
                    all_auto_counts.append(0)
                    all_correct_counts.append(0)
                    all_error_counts.append(0)

    mean_n, std_n = mean_std(all_n_values)

    med_auto, std_auto = mean_std(all_automations)
    med_auto_count, std_auto_count = mean_std(all_auto_counts)
    med_correct, std_correct = mean_std(all_correct_counts)
    med_error, std_error = mean_std(all_error_counts)
    med_err_rate, std_err_rate = mean_std(all_err_rates)

    cost_per_case_scenario = (largest_scenario["Wage_Basis"] / 60.0) * HUMAN_MINUTES_PER_CASE
    projected_auto_cases = largest_scenario["Annual_Volume"] * (med_auto / 100.0)
    projected_savings = projected_auto_cases * cost_per_case_scenario
    projected_hours = (projected_auto_cases * HUMAN_MINUTES_PER_CASE) / 60.0

    rows.append({
        "Dataset": "All Datasets (Merged)",
        "Method": method2path[method]['method_name'],
        "Test Set Size": f"{mean_n:.0f} ± {std_n:.0f}",
        "% Automated": f"{med_auto:.0f} ± {std_auto:.0f}",
        "Automated Cases (Test)": f"{med_auto_count:.0f} ± {std_auto_count:.0f}",
        "Correct (Test)": f"{med_correct:.0f} ± {std_correct:.0f}",
        "Errors (Test)": f"{med_error:.0f} ± {std_error:.0f}",
        "Error Rate (%)": f"{med_err_rate:.0f} ± {std_err_rate:.0f}",
        "Hours Saved": f"{projected_hours/1000:.1f}K" if projected_hours > 0 else "0",
        "Dollars Saved": f"${projected_savings/1_000_000:.2f}M" if projected_savings > 0 else "$0"
    })

df_impact = pd.DataFrame(rows)

print("\nFinal Impact Table:\n")
print(df_impact.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

print("\n" + "="*80)
print("📊 BUILDING FINAL IMPACT TABLE (MATCHING DASHBOARD - FIXED)")
print("="*80)

# Financial constants (CMS 2025)
HUMAN_MINUTES_PER_CASE = 2.0

# Use the largest scenario (already defined earlier in the script)
# scenarios[2] is "Integrated Health Network" with 1M annual volume
largest_scenario = scenarios[2]
print(f"Using scenario: {largest_scenario['Setting']} ({largest_scenario['Annual_Volume']:,} cases/year)")

datasets_configs = [
    ("contextual", "Contextual"),
    ("synthesis", "Synthesis"),
    ("clinical_inference", "Clinical Inference"),
]

methods = list(method2path.keys())

rows = []

for dataset_type, dataset_name in datasets_configs:

    # ------------------------------------------------------------------
    # Get N per LLM (for test set size reporting only)
    # ------------------------------------------------------------------
    llm_n_samples = {}
    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        if 'CCPS' in llm2method2dataset2results.get(llm_id, {}):
            if dataset_type in llm2method2dataset2results[llm_id]['CCPS']:
                shared = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
                results = llm2method2dataset2results[llm_id]['CCPS'][dataset_type]
                count = len(shared) if shared else len(results)
                llm_n_samples[llm_display] = count
    
    # Median N for display
    median_n = int(np.median(list(llm_n_samples.values()))) if llm_n_samples else 0

    # ------------------------------------------------------------------
    # Method metrics - Calculate automation rates
    # ------------------------------------------------------------------
    for method in methods:

        automations = []
        auto_counts = []
        correct_counts = []
        error_counts = []
        err_rates = []

        llm_data = threshold_results.get(dataset_type, {})
        
        for llm_display, llm_results in llm_data.items():
            if method in llm_results:
                res = llm_results[method]
                
                # Get THIS LLM's actual N
                llm_n = llm_n_samples.get(llm_display, 0)
                
                if res['meets_criterion'] and llm_n > 0:
                    pct_auto = res['pct_automated']
                    acc = res['accuracy_at_tau']
                    
                    automations.append(pct_auto)
                    
                    # Calculate counts using THIS LLM's actual N (for test set metrics)
                    auto = llm_n * pct_auto / 100
                    correct = auto * acc
                    error = auto - correct
                    
                    auto_counts.append(auto)
                    correct_counts.append(correct)
                    error_counts.append(error)
                    err_rates.append((1 - acc) * 100)
                else:
                    automations.append(0.0)
                    auto_counts.append(0)
                    correct_counts.append(0)
                    error_counts.append(0)

        # Compute medians and IQR
        if automations:
            med_auto = np.median(automations)
            low_auto = np.percentile(automations, 25)
            high_auto = np.percentile(automations, 75)
            
            med_auto_count = np.median(auto_counts)
            low_auto_count = np.percentile(auto_counts, 25)
            high_auto_count = np.percentile(auto_counts, 75)
            
            med_correct = np.median(correct_counts)
            low_correct = np.percentile(correct_counts, 25)
            high_correct = np.percentile(correct_counts, 75)
            
            med_error = np.median(error_counts)
            low_error = np.percentile(error_counts, 25)
            high_error = np.percentile(error_counts, 75)
            
            # For error rate, only include cases with automation
            if err_rates:
                med_err_rate = np.median(err_rates)
                low_err_rate = np.percentile(err_rates, 25)
                high_err_rate = np.percentile(err_rates, 75)
            else:
                med_err_rate = low_err_rate = high_err_rate = 0
            
            # ------------------------------------------------------------------
            # FINANCIAL PROJECTIONS FOR LARGEST SCENARIO (1M cases)
            # ------------------------------------------------------------------
            cost_per_case_scenario = (largest_scenario["Wage_Basis"] / 60.0) * HUMAN_MINUTES_PER_CASE
            projected_auto_cases = largest_scenario["Annual_Volume"] * (med_auto / 100.0)
            projected_savings = projected_auto_cases * cost_per_case_scenario
            projected_hours = (projected_auto_cases * HUMAN_MINUTES_PER_CASE) / 60.0
            
        else:
            med_auto = low_auto = high_auto = 0
            med_auto_count = low_auto_count = high_auto_count = 0
            med_correct = low_correct = high_correct = 0
            med_error = low_error = high_error = 0
            med_err_rate = low_err_rate = high_err_rate = 0
            projected_hours = 0
            projected_savings = 0

        rows.append({
            "Dataset": dataset_name,
            "Method": method2path[method]['method_name'],
            "Test Set Size": median_n,
            "% Automated": f"{med_auto:.0f} / {low_auto:.0f}-{high_auto:.0f}",
            "Automated Cases (Test)": f"{med_auto_count:.0f} / {low_auto_count:.0f}-{high_auto_count:.0f}",
            "Correct (Test)": f"{med_correct:.0f} / {low_correct:.0f}-{high_correct:.0f}",
            "Errors (Test)": f"{med_error:.0f} / {low_error:.0f}-{high_error:.0f}",
            "Error Rate (%)": f"{med_err_rate:.0f} / {low_err_rate:.0f}-{high_err_rate:.0f}",
            "Hours Saved": f"{projected_hours/1000:.1f}K" if projected_hours > 0 else "0",
            "Dollars Saved": f"${projected_savings/1_000_000:.2f}M" if projected_savings > 0 else "$0"
        })

# ------------------------------------------------------------------
# MERGED ROW - aggregate across all three datasets
# ------------------------------------------------------------------
for method in methods:
    all_automations = []
    all_auto_counts = []
    all_correct_counts = []
    all_error_counts = []
    all_err_rates = []
    
    # Track total N across datasets
    all_n_values = []
    
    for dataset_type in ["contextual", "synthesis", "clinical_inference"]:
        # Get N per LLM for this dataset
        llm_n_samples = {}
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            if 'CCPS' in llm2method2dataset2results.get(llm_id, {}):
                if dataset_type in llm2method2dataset2results[llm_id]['CCPS']:
                    shared = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
                    results = llm2method2dataset2results[llm_id]['CCPS'][dataset_type]
                    count = len(shared) if shared else len(results)
                    llm_n_samples[llm_display] = count
        
        if llm_n_samples:
            all_n_values.extend(llm_n_samples.values())
        
        # Get results for this dataset
        llm_data = threshold_results.get(dataset_type, {})
        
        for llm_display, llm_results in llm_data.items():
            if method in llm_results:
                res = llm_results[method]
                
                # Get THIS LLM's actual N for THIS dataset
                llm_n = llm_n_samples.get(llm_display, 0)
                
                if res['meets_criterion'] and llm_n > 0:
                    pct_auto = res['pct_automated']
                    acc = res['accuracy_at_tau']
                    
                    all_automations.append(pct_auto)
                    
                    auto = llm_n * pct_auto / 100
                    correct = auto * acc
                    error = auto - correct
                    
                    all_auto_counts.append(auto)
                    all_correct_counts.append(correct)
                    all_error_counts.append(error)
                    all_err_rates.append((1 - acc) * 100)
                else:
                    all_automations.append(0.0)
                    all_auto_counts.append(0)
                    all_correct_counts.append(0)
                    all_error_counts.append(0)
    
    # Total median N across all datasets
    total_median_n = int(np.median(all_n_values)) if all_n_values else 0
    
    # Compute medians
    if all_automations:
        med_auto = np.median(all_automations)
        low_auto = np.percentile(all_automations, 25)
        high_auto = np.percentile(all_automations, 75)
        
        med_auto_count = np.median(all_auto_counts)
        low_auto_count = np.percentile(all_auto_counts, 25)
        high_auto_count = np.percentile(all_auto_counts, 75)
        
        med_correct = np.median(all_correct_counts)
        low_correct = np.percentile(all_correct_counts, 25)
        high_correct = np.percentile(all_correct_counts, 75)
        
        med_error = np.median(all_error_counts)
        low_error = np.percentile(all_error_counts, 25)
        high_error = np.percentile(all_error_counts, 75)
        
        if all_err_rates:
            med_err_rate = np.median(all_err_rates)
            low_err_rate = np.percentile(all_err_rates, 25)
            high_err_rate = np.percentile(all_err_rates, 75)
        else:
            med_err_rate = low_err_rate = high_err_rate = 0
        
        # Financial projections for merged
        cost_per_case_scenario = (largest_scenario["Wage_Basis"] / 60.0) * HUMAN_MINUTES_PER_CASE
        projected_auto_cases = largest_scenario["Annual_Volume"] * (med_auto / 100.0)
        projected_savings = projected_auto_cases * cost_per_case_scenario
        projected_hours = (projected_auto_cases * HUMAN_MINUTES_PER_CASE) / 60.0
    else:
        med_auto = low_auto = high_auto = 0
        med_auto_count = low_auto_count = high_auto_count = 0
        med_correct = low_correct = high_correct = 0
        med_error = low_error = high_error = 0
        med_err_rate = low_err_rate = high_err_rate = 0
        projected_hours = 0
        projected_savings = 0
    
    rows.append({
        "Dataset": "All Datasets (Merged)",
        "Method": method2path[method]['method_name'],
        "Test Set Size": total_median_n,
        "% Automated": f"{med_auto:.0f} / {low_auto:.0f}-{high_auto:.0f}",
        "Automated Cases (Test)": f"{med_auto_count:.0f} / {low_auto_count:.0f}-{high_auto_count:.0f}",
        "Correct (Test)": f"{med_correct:.0f} / {low_correct:.0f}-{high_correct:.0f}",
        "Errors (Test)": f"{med_error:.0f} / {low_error:.0f}-{high_error:.0f}",
        "Error Rate (%)": f"{med_err_rate:.0f} / {low_err_rate:.0f}-{high_err_rate:.0f}",
        "Hours Saved": f"{projected_hours/1000:.1f}K" if projected_hours > 0 else "0",
        "Dollars Saved": f"${projected_savings/1_000_000:.2f}M" if projected_savings > 0 else "$0"
    })

df_impact = pd.DataFrame(rows)

print("\nFinal Impact Table (ASTRO 2026 Format - Matches Dashboard):\n")
print(df_impact.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

print("\n" + "="*80)
print("📊 BUILDING FINAL IMPACT TABLE (PER LLM)")
print("="*80)

HUMAN_MINUTES_PER_CASE = 2.0
largest_scenario = scenarios[2]

datasets_configs = [
    ("contextual", "Contextual"),
    ("synthesis", "Synthesis"),
    ("clinical_inference", "Clinical Inference"),
]

methods = list(method2path.keys())
rows = []

# ---------------------------------------------------------
# PER DATASET, PER LLM
# ---------------------------------------------------------
for dataset_type, dataset_name in datasets_configs:

    llm_data = threshold_results.get(dataset_type, {})

    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)

        if llm_display not in llm_data:
            continue

        # Compute test set size for this LLM
        llm_n = 0
        if 'CCPS' in llm2method2dataset2results.get(llm_id, {}):
            if dataset_type in llm2method2dataset2results[llm_id]['CCPS']:
                shared = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
                results = llm2method2dataset2results[llm_id]['CCPS'][dataset_type]
                llm_n = len(shared) if shared else len(results)

        for method in methods:

            llm_results = llm_data[llm_display]
            if method not in llm_results:
                continue

            res = llm_results[method]

            if not res['meets_criterion'] or llm_n == 0:
                pct_auto = 0
                auto = correct = error = err_rate = 0
            else:
                pct_auto = res['pct_automated']
                acc = res['accuracy_at_tau']

                auto = llm_n * pct_auto / 100
                correct = auto * acc
                error = auto - correct
                err_rate = (1 - acc) * 100

            cost_per_case = (largest_scenario["Wage_Basis"] / 60.0) * HUMAN_MINUTES_PER_CASE
            projected_auto_cases = largest_scenario["Annual_Volume"] * (pct_auto / 100.0)
            projected_savings = projected_auto_cases * cost_per_case
            projected_hours = (projected_auto_cases * HUMAN_MINUTES_PER_CASE) / 60.0

            rows.append({
                "Dataset": dataset_name,
                "LLM": llm_display,
                "Method": method2path[method]['method_name'],
                "Test Set Size": int(llm_n),
                "% Automated": f"{pct_auto:.0f}",
                "Automated Cases (Test)": f"{auto:.0f}",
                "Correct (Test)": f"{correct:.0f}",
                "Errors (Test)": f"{error:.0f}",
                "Error Rate (%)": f"{err_rate:.0f}",
                "Hours Saved": f"{projected_hours/1000:.1f}K" if projected_hours > 0 else "0",
                "Dollars Saved": f"${projected_savings/1_000_000:.2f}M" if projected_savings > 0 else "$0"
            })


# ---------------------------------------------------------
# MERGED DATASETS PER LLM
# ---------------------------------------------------------
for llm_id in llm_ids:
    llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)

    for method in methods:
        total_auto = total_correct = total_error = total_n = 0
        automations = []
        err_rates = [] 

        for dataset_type in ["contextual", "synthesis", "clinical_inference"]:
            llm_data = threshold_results.get(dataset_type, {})

            if llm_display not in llm_data:
                continue
            if method not in llm_data[llm_display]:
                continue

            res = llm_data[llm_display][method]

            llm_n = 0
            if 'CCPS' in llm2method2dataset2results.get(llm_id, {}):
                if dataset_type in llm2method2dataset2results[llm_id]['CCPS']:
                    shared = llm2dataset2sharedRIDacrossMethods[llm_id].get(dataset_type, set())
                    results = llm2method2dataset2results[llm_id]['CCPS'][dataset_type]
                    llm_n = len(shared) if shared else len(results)

            if res['meets_criterion'] and llm_n > 0:
                pct_auto = res['pct_automated']
                acc = res['accuracy_at_tau']

                auto = llm_n * pct_auto / 100
                correct = auto * acc
                error = auto - correct

                total_auto += auto
                total_correct += correct
                total_error += error
                total_n += llm_n

                automations.append(pct_auto)
                err_rates.append((1 - acc) * 100)

        if total_n == 0:
            continue

        mean_auto = np.mean(automations) if automations else 0
        mean_err_rate = np.mean(err_rates) if err_rates else 0

        cost_per_case = (largest_scenario["Wage_Basis"] / 60.0) * HUMAN_MINUTES_PER_CASE
        projected_auto_cases = largest_scenario["Annual_Volume"] * (mean_auto / 100.0)
        projected_savings = projected_auto_cases * cost_per_case
        projected_hours = (projected_auto_cases * HUMAN_MINUTES_PER_CASE) / 60.0

        rows.append({
            "Dataset": "All Datasets (Merged)",
            "LLM": llm_display,
            "Method": method2path[method]['method_name'],
            "Test Set Size": int(total_n),
            "% Automated": f"{mean_auto:.0f}",
            "Automated Cases (Test)": f"{total_auto:.0f}",
            "Correct (Test)": f"{total_correct:.0f}",
            "Errors (Test)": f"{total_error:.0f}",
            "Error Rate (%)": f"{mean_err_rate:.0f}",
            "Hours Saved": f"{projected_hours/1000:.1f}K" if projected_hours > 0 else "0",
            "Dollars Saved": f"${projected_savings/1_000_000:.2f}M" if projected_savings > 0 else "$0"
        })

df_impact = pd.DataFrame(rows)

print("\nFinal Impact Table (Per LLM):\n")
print(df_impact.to_string(index=False))
df_impact


In [ ]:
# ==============================================================================
# ADDITIONAL ANALYSIS: CCPS HEAD-TO-HEAD vs ALL METHODS + EXCLUSIVE UNLOCKS
# ==============================================================================

import numpy as np
import pandas as pd

TARGET_METHOD = 'CCPS'
BASELINE_METHODS = ['PTRUE', 'PIK', 'SAPLMA-F']

# ==============================================================================
# HEAD-TO-HEAD PER BASELINE
# ==============================================================================

all_h2h_rows = []
summary_by_baseline = {}

for COMPARISON_METHOD in BASELINE_METHODS:
    print(f"\n{'='*80}")
    print(f"CCPS vs {method2path[COMPARISON_METHOD]['method_name']} HEAD-TO-HEAD")
    print(f"{'='*80}")

    wins = losses = ties = exclusive_unlocks = baseline_exclusive = both_zero = 0
    rows = []

    for dataset_type in dataset_types:
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})

            ccps_res = llm_data.get(TARGET_METHOD, {})
            comp_res = llm_data.get(COMPARISON_METHOD, {})

            ccps_auto = ccps_res.get('pct_automated', 0) if ccps_res.get('meets_criterion') else 0
            comp_auto = comp_res.get('pct_automated', 0) if comp_res.get('meets_criterion') else 0

            diff = ccps_auto - comp_auto

            if ccps_auto > 0 and comp_auto == 0:
                exclusive_unlocks += 1
                outcome = 'CCPS_EXCLUSIVE'
            elif comp_auto > 0 and ccps_auto == 0:
                baseline_exclusive += 1
                outcome = f'{COMPARISON_METHOD}_EXCLUSIVE'
            elif ccps_auto == 0 and comp_auto == 0:
                both_zero += 1
                outcome = 'BOTH_ZERO'
            elif ccps_auto > comp_auto:
                wins += 1
                outcome = 'CCPS_WINS'
            elif comp_auto > ccps_auto:
                losses += 1
                outcome = 'BASELINE_WINS'
            else:
                ties += 1
                outcome = 'TIE'

            rows.append({
                'Dataset': dataset_display_names[dataset_type],
                'LLM': llm_display,
                f'CCPS %Auto': f"{ccps_auto:.1f}%",
                f'{COMPARISON_METHOD} %Auto': f"{comp_auto:.1f}%",
                'CCPS Advantage': f"{diff:+.1f}%",
                'Outcome': outcome
            })
            all_h2h_rows.append({
                'Baseline': method2path[COMPARISON_METHOD]['method_name'],
                'Dataset': dataset_display_names[dataset_type],
                'LLM': llm_display,
                'CCPS %Auto': ccps_auto,
                f'Baseline %Auto': comp_auto,
                'CCPS Advantage': diff,
                'Outcome': outcome
            })

    df_h2h = pd.DataFrame(rows)
    print(df_h2h.to_string(index=False))

    total_cells = len(rows)
    contested = wins + losses + ties
    print(f"\n  CCPS wins (both nonzero, CCPS higher):        {wins}")
    print(f"  Baseline wins (both nonzero, baseline higher): {losses}")
    print(f"  Ties (both nonzero, equal):                   {ties}")
    print(f"  CCPS exclusive unlock (baseline=0, CCPS>0):   {exclusive_unlocks}")
    print(f"  Baseline exclusive (CCPS=0, baseline>0):      {baseline_exclusive}")
    print(f"  Both zero:                                    {both_zero}")
    if contested > 0:
        print(f"\n  CCPS win rate among contested pairs:         {wins}/{contested} = {wins/contested*100:.1f}%")
    print(f"  CCPS overall advantage cells:                {wins + exclusive_unlocks}/{total_cells} = {(wins+exclusive_unlocks)/total_cells*100:.1f}%")

    summary_by_baseline[COMPARISON_METHOD] = {
        'wins': wins, 'losses': losses, 'ties': ties,
        'exclusive_unlocks': exclusive_unlocks,
        'baseline_exclusive': baseline_exclusive,
        'both_zero': both_zero,
        'total': total_cells,
        'contested': contested
    }

# ==============================================================================
# AGGREGATE SUMMARY TABLE ACROSS ALL BASELINES
# ==============================================================================

print(f"\n{'='*80}")
print("AGGREGATE SUMMARY: CCPS vs ALL BASELINES")
print(f"{'='*80}")

summary_rows = []
for method, s in summary_by_baseline.items():
    win_rate = f"{s['wins']}/{s['contested']} ({s['wins']/s['contested']*100:.0f}%)" if s['contested'] > 0 else "N/A"
    overall = f"{s['wins']+s['exclusive_unlocks']}/{s['total']} ({(s['wins']+s['exclusive_unlocks'])/s['total']*100:.0f}%)"
    summary_rows.append({
        'Baseline': method2path[method]['method_name'],
        'CCPS Wins': s['wins'],
        'Baseline Wins': s['losses'],
        'Ties': s['ties'],
        'CCPS Exclusive Unlocks': s['exclusive_unlocks'],
        'Baseline Exclusive': s['baseline_exclusive'],
        'Both Zero': s['both_zero'],
        'Win Rate (contested)': win_rate,
        'CCPS Overall Advantage': overall
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

# ==============================================================================
# ECE AT FIXED 30% HUMAN REVIEW RATE
# ==============================================================================

print(f"\n{'='*80}")
print("ECE AT FIXED 30% HUMAN REVIEW RATE")
print(f"{'='*80}")

TARGET_WORKLOAD = 30

ece_at_30 = {}
for method_name in METHODS_TO_SHOW:
    ece_vals_all = []
    for dataset_type in dataset_types:
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            ece_data_lookup = {
                'contextual': ece_coverage_data_contextual,
                'synthesis': ece_coverage_data,
                'clinical_inference': ece_coverage_data_clinical
            }
            ece_data = ece_data_lookup[dataset_type]
            if llm_display in ece_data and method_name in ece_data[llm_display]:
                workloads = ece_data[llm_display][method_name]['human_workloads']
                ece_vals = ece_data[llm_display][method_name]['ece_values']
                idx = min(range(len(workloads)), key=lambda i: abs(workloads[i] - TARGET_WORKLOAD))
                ece_vals_all.append(ece_vals[idx])

    if ece_vals_all:
        ece_at_30[method_name] = {
            'mean': np.mean(ece_vals_all),
            'std': np.std(ece_vals_all),
            'n': len(ece_vals_all)
        }

print(f"\nECE at ~{TARGET_WORKLOAD}% human review rate (lower is better):")
for method_name, stats in ece_at_30.items():
    display = method2path[method_name]['method_name']
    print(f"  {display}: ECE = {stats['mean']:.4f} +/- {stats['std']:.4f}  (n={stats['n']})")

print()
ccps_ece = ece_at_30.get('CCPS', {}).get('mean', None)
for method_name in ['PTRUE', 'PIK', 'SAPLMA-F']:
    if method_name in ece_at_30 and ccps_ece is not None:
        baseline_ece = ece_at_30[method_name]['mean']
        reduction = baseline_ece - ccps_ece
        rel = reduction / baseline_ece * 100 if baseline_ece > 0 else 0
        print(f"  CCPS ECE reduction vs {method2path[method_name]['method_name']}: {reduction:.4f} ({rel:.1f}% relative)")

# ==============================================================================
# COMPLEXITY GRADIENT: Does CCPS advantage grow with task difficulty?
# ==============================================================================

print(f"\n{'='*80}")
print("CCPS ADVANTAGE BY TASK COMPLEXITY TIER (vs ALL BASELINES)")
print(f"{'='*80}")

tier_order = ['contextual', 'synthesis', 'clinical_inference']
tier_labels = ['Contextual (Easy)', 'Synthesis (Medium)', 'Clinical Inference (Hard)']

for dataset_type, label in zip(tier_order, tier_labels):
    print(f"\n  {label}:")
    ccps_autos = []
    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
        ccps_res = llm_data.get('CCPS', {})
        ccps_autos.append(ccps_res.get('pct_automated', 0) if ccps_res.get('meets_criterion') else 0)
    mean_ccps = np.mean(ccps_autos)
    print(f"    CCPS mean safe yield: {mean_ccps:.1f}%")

    for COMPARISON_METHOD in BASELINE_METHODS:
        comp_autos = []
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
            comp_res = llm_data.get(COMPARISON_METHOD, {})
            comp_autos.append(comp_res.get('pct_automated', 0) if comp_res.get('meets_criterion') else 0)
        mean_comp = np.mean(comp_autos)
        diff = mean_ccps - mean_comp
        print(f"    vs {method2path[COMPARISON_METHOD]['method_name']}: baseline={mean_comp:.1f}%  advantage={diff:+.1f}%")

In [ ]:
import numpy as np

print("=" * 80)
print("ROW 3: ROBUSTNESS - AUROC AND CALIBRATION QUALITY ACROSS WORKLOAD BINS")
print("Workload bins: 0-25%, 25-50%, 50-75%, 75-100% human review rate")
print("=" * 80)

BINS = [(0, 25), (25, 50), (50, 75), (75, 100)]
METHODS_ROW3 = ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']

# Store for merged summary
bin_auroc_all = {m: {b: [] for b in BINS} for m in METHODS_ROW3}
bin_cal_all   = {m: {b: [] for b in BINS} for m in METHODS_ROW3}

ece_lookup = {
    'contextual':         ece_coverage_data_contextual,
    'synthesis':          ece_coverage_data,
    'clinical_inference': ece_coverage_data_clinical
}

# Use risk_coverage_data which contains system_auroc_scores
auroc_lookup = {
    'contextual':         risk_coverage_data_contextual,
    'synthesis':          risk_coverage_data,
    'clinical_inference': risk_coverage_data_clinical
}

for dataset_type in dataset_types:
    print(f"\n{'=' * 60}")
    print(f"Dataset: {dataset_display_names[dataset_type]}")
    print(f"{'=' * 60}")

    ece_data  = ece_lookup[dataset_type]
    auroc_data = auroc_lookup[dataset_type]

    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        print(f"\n  [{llm_display}]")

        for method in METHODS_ROW3:
            print(f"    {method}")

            # AUROC across coverage (use system_auroc_scores key)
            if (llm_display in auroc_data and method in auroc_data[llm_display]):
                workloads_a = np.array(auroc_data[llm_display][method]['human_workloads'])
                auroc_vals  = np.array(auroc_data[llm_display][method]['system_auroc_scores'])
            else:
                workloads_a, auroc_vals = np.array([]), np.array([])

            # ECE across coverage
            if (llm_display in ece_data and method in ece_data[llm_display]):
                workloads_e = np.array(ece_data[llm_display][method]['human_workloads'])
                ece_vals    = np.array(ece_data[llm_display][method]['ece_values'])
            else:
                workloads_e, ece_vals = np.array([]), np.array([])

            for (lo, hi) in BINS:
                # AUROC bin
                if len(workloads_a) > 0:
                    mask_a = (workloads_a >= lo) & (workloads_a < hi)
                    bin_auroc = auroc_vals[mask_a]
                else:
                    bin_auroc = np.array([])

                # ECE bin -> calibration quality = 1 - ECE
                if len(workloads_e) > 0:
                    mask_e = (workloads_e >= lo) & (workloads_e < hi)
                    bin_cal = 1.0 - ece_vals[mask_e]
                else:
                    bin_cal = np.array([])

                auroc_str = (f"median={np.median(bin_auroc):.4f} "
                             f"IQR=[{np.percentile(bin_auroc,25):.4f},{np.percentile(bin_auroc,75):.4f}]"
                             ) if len(bin_auroc) > 0 else "N/A"

                cal_str = (f"median={np.median(bin_cal):.4f} "
                           f"IQR=[{np.percentile(bin_cal,25):.4f},{np.percentile(bin_cal,75):.4f}]"
                           ) if len(bin_cal) > 0 else "N/A"

                print(f"      [{lo}-{hi}%] AUROC: {auroc_str} | CalQual: {cal_str}")

                # Accumulate for merged
                if len(bin_auroc) > 0:
                    bin_auroc_all[method][(lo,hi)].extend(bin_auroc.tolist())
                if len(bin_cal) > 0:
                    bin_cal_all[method][(lo,hi)].extend(bin_cal.tolist())

print(f"\n{'=' * 80}")
print("MERGED (All Datasets, All LLMs) - AUROC and Calibration Quality by Workload Bin")
print("=" * 80)

for method in METHODS_ROW3:
    print(f"\n  {method}")
    for (lo, hi) in BINS:
        a_vals = np.array(bin_auroc_all[method][(lo,hi)])
        c_vals = np.array(bin_cal_all[method][(lo,hi)])

        auroc_str = (f"median={np.median(a_vals):.4f} "
                     f"IQR=[{np.percentile(a_vals,25):.4f},{np.percentile(a_vals,75):.4f}]"
                     ) if len(a_vals) > 0 else "N/A"

        cal_str = (f"median={np.median(c_vals):.4f} "
                   f"IQR=[{np.percentile(c_vals,25):.4f},{np.percentile(c_vals,75):.4f}]"
                   ) if len(c_vals) > 0 else "N/A"

        print(f"    [{lo}-{hi}%] AUROC: {auroc_str} | CalQual: {cal_str}")

print("\n✅ Row 3 robustness verification complete.")

In [ ]:
import numpy as np

print("=" * 80)
print("TABLE 1: CALIBRATION METRICS (ECE, AUROC, Brier) - Per Dataset Per Method")
print("Averaged across LLMs")
print("=" * 80)

df_reset = df_metrics.reset_index()

for dataset_type in dataset_types:
    print(f"\n--- {dataset_display_names[dataset_type]} ---")
    subset = df_reset[df_reset['dataset_type'] == dataset_type]
    method_agg = subset.groupby('method').agg(
        ECE_mean=('ece', 'mean'),
        ECE_std=('ece', 'std'),
        AUROC_mean=('auroc', 'mean'),
        AUROC_std=('auroc', 'std'),
        Brier_mean=('brier', 'mean'),
        Brier_std=('brier', 'std'),
    ).reset_index()
    for _, row in method_agg.iterrows():
        print(f"  {row['method']:15s} | ECE: {row['ECE_mean']:.4f} ± {row['ECE_std']:.4f} | "
              f"AUROC: {row['AUROC_mean']:.4f} ± {row['AUROC_std']:.4f} | "
              f"Brier: {row['Brier_mean']:.4f} ± {row['Brier_std']:.4f}")

# MERGED: average across all dataset types
print(f"\n--- MERGED (All Datasets) ---")
method_agg_merged = df_reset.groupby('method').agg(
    ECE_mean=('ece', 'mean'),
    ECE_std=('ece', 'std'),
    AUROC_mean=('auroc', 'mean'),
    AUROC_std=('auroc', 'std'),
    Brier_mean=('brier', 'mean'),
    Brier_std=('brier', 'std'),
).reset_index()
for _, row in method_agg_merged.iterrows():
    print(f"  {row['method']:15s} | ECE: {row['ECE_mean']:.4f} ± {row['ECE_std']:.4f} | "
          f"AUROC: {row['AUROC_mean']:.4f} ± {row['AUROC_std']:.4f} | "
          f"Brier: {row['Brier_mean']:.4f} ± {row['Brier_std']:.4f}")


print("\n")
print("=" * 80)
print("TABLE 2: SAFE YIELD - Per Dataset Per Method Per LLM")
print("Threshold = tau where accuracy >= 95% (5% error constraint)")
print("=" * 80)

for dataset_type in dataset_types:
    print(f"\n--- {dataset_display_names[dataset_type]} ---")
    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
        print(f"  [{llm_display}]")
        for method in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']:
            res = llm_data.get(method, {})
            if res.get('meets_criterion'):
                print(f"    {method:12s} | tau={res['tau']:.2f} | "
                      f"yield={res['pct_automated']:.1f}% | "
                      f"acc={res['accuracy_at_tau']:.3f}")
            else:
                print(f"    {method:12s} | FAILS 95% threshold")


print("\n")
print("=" * 80)
print("TABLE 2 SUMMARY: SAFE YIELD averaged across LLMs - Per Dataset")
print("=" * 80)

# Store for merged computation
method_yields_all_datasets = {m: [] for m in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']}

for dataset_type in dataset_types:
    print(f"\n--- {dataset_display_names[dataset_type]} ---")
    for method in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']:
        yields = []
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            res = threshold_results.get(dataset_type, {}).get(llm_display, {}).get(method, {})
            val = res.get('pct_automated', 0) if res.get('meets_criterion') else 0.0
            yields.append(val)
            method_yields_all_datasets[method].append(val)
        print(f"  {method:12s} | mean: {np.mean(yields):.1f}% | "
              f"std: {np.std(yields):.1f}% | "
              f"range: {np.min(yields):.1f}-{np.max(yields):.1f}%")

print(f"\n--- MERGED (All Datasets, All LLMs) ---")
for method in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']:
    vals = method_yields_all_datasets[method]
    print(f"  {method:12s} | mean: {np.mean(vals):.1f}% | "
          f"std: {np.std(vals):.1f}% | "
          f"range: {np.min(vals):.1f}-{np.max(vals):.1f}%")


print("\n")
print("=" * 80)
print("CCPS vs PIK ADVANTAGE BY TIER + MERGED (complexity gradient)")
print("=" * 80)

ccps_all, pik_all = [], []

for dataset_type in dataset_types:
    ccps_yields, pik_yields = [], []
    for llm_id in llm_ids:
        llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
        llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
        ccps = llm_data.get('CCPS', {})
        pik = llm_data.get('PIK', {})
        c = ccps.get('pct_automated', 0) if ccps.get('meets_criterion') else 0
        p = pik.get('pct_automated', 0) if pik.get('meets_criterion') else 0
        ccps_yields.append(c)
        pik_yields.append(p)
        ccps_all.append(c)
        pik_all.append(p)
    diff = np.mean(ccps_yields) - np.mean(pik_yields)
    print(f"  {dataset_display_names[dataset_type]}")
    print(f"    CCPS: {np.mean(ccps_yields):.1f}% | PIK: {np.mean(pik_yields):.1f}% | "
          f"advantage: {diff:+.1f}pp")

diff_merged = np.mean(ccps_all) - np.mean(pik_all)
print(f"  MERGED (All Datasets)")
print(f"    CCPS: {np.mean(ccps_all):.1f}% | PIK: {np.mean(pik_all):.1f}% | "
      f"advantage: {diff_merged:+.1f}pp")


print("\n")
print("=" * 80)
print("ECE AT 30% HUMAN REVIEW RATE - Per Dataset + Merged")
print("=" * 80)

TARGET_WORKLOAD = 30
ece_lookup = {
    'contextual': ece_coverage_data_contextual,
    'synthesis': ece_coverage_data,
    'clinical_inference': ece_coverage_data_clinical
}

method_ece_all = {m: [] for m in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']}

for dataset_type in dataset_types:
    print(f"\n--- {dataset_display_names[dataset_type]} ---")
    ece_data = ece_lookup[dataset_type]
    for method_name in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']:
        ece_vals = []
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            if llm_display in ece_data and method_name in ece_data[llm_display]:
                workloads = ece_data[llm_display][method_name]['human_workloads']
                vals = ece_data[llm_display][method_name]['ece_values']
                idx = min(range(len(workloads)), key=lambda i: abs(workloads[i] - TARGET_WORKLOAD))
                ece_vals.append(vals[idx])
                method_ece_all[method_name].append(vals[idx])
        display = method2path[method_name]['method_name']
        if ece_vals:
            print(f"  {display:15s} | ECE={np.mean(ece_vals):.4f} ± {np.std(ece_vals):.4f}")

print(f"\n--- MERGED (All Datasets) ---")
for method_name in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']:
    vals = method_ece_all[method_name]
    display = method2path[method_name]['method_name']
    if vals:
        print(f"  {display:15s} | ECE={np.mean(vals):.4f} ± {np.std(vals):.4f}")


print("\n")
print("=" * 80)
print("FINANCIAL IMPACT - Per Dataset + Merged (Integrated Health Network, 1M cases/year)")
print("=" * 80)

largest = scenarios[2]
cost_per_case = (largest['Wage_Basis'] / 60.0) * 2.0

method_savings_all = {m: [] for m in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']}

for dataset_type in dataset_types:
    print(f"\n--- {dataset_display_names[dataset_type]} ---")
    for method in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']:
        yields = []
        for llm_id in llm_ids:
            llm_display = LLM_DISPLAY_NAMES.get(llm_id, llm_id)
            res = threshold_results.get(dataset_type, {}).get(llm_display, {}).get(method, {})
            yields.append(res.get('pct_automated', 0) if res.get('meets_criterion') else 0)
        mean_yield = np.mean(yields)
        savings = largest['Annual_Volume'] * (mean_yield / 100.0) * cost_per_case
        method_savings_all[method].append(savings)
        print(f"  {method:12s} | yield={mean_yield:.1f}% | savings=${savings/1e6:.2f}M/yr")

print(f"\n--- MERGED (Average across datasets) ---")
for method in ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']:
    mean_savings = np.mean(method_savings_all[method])
    total_savings = np.sum(method_savings_all[method])
    print(f"  {method:12s} | mean per-dataset savings=${mean_savings/1e6:.2f}M/yr | "
          f"total across datasets=${total_savings/1e6:.2f}M/yr")

print("\n✅ Verification complete. Cross-check these numbers before writing.")

In [ ]:
# ==============================================================================
# TABLE 2: SAFE YIELD - Per Dataset Per Method Per LLM
# ==============================================================================

import numpy as np
import pandas as pd

print("\n" + "=" * 120)
print("TABLE 2: SAFE YIELD - Per Dataset Per Method Per LLM")
print("Threshold = tau where accuracy >= 95% (5% error constraint)")
print("=" * 120 + "\n")

# Define methods
methods = ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']
method_labels = {
    'PTRUE': 'P(True)',
    'PIK': 'P(IK)',
    'SAPLMA-F': 'SAPLMA-F',
    'CCPS': 'CCPS (Ours)'
}

# Define datasets with their display names and sample sizes
dataset_configs = [
    {'type': 'contextual', 'display': 'Contextual', 'n': 1458},
    {'type': 'synthesis', 'display': 'Synthesis', 'n': 764},
    {'type': 'clinical_inference', 'display': 'Clinical Inference', 'n': 612},
    {'type': 'merged', 'display': 'Merged (All Tiers)', 'n': None}
]

# LLM order
llm_order = ['Qwen-0.5B', 'Llama-1B', 'Qwen-1.5B', 'Llama-3B', 'Qwen-3B']

def print_table_header():
    """Print clean table header matching the image format."""
    # Header row 1: Method names
    header1 = f"{'LLM':<20}"
    for method in methods:
        method_name = method_labels[method]
        header1 += f"{method_name:^25}"
    print(header1)
    
    # Header row 2: Subcolumns
    header2 = f"{'':20}"
    for _ in methods:
        header2 += f"{'tau / yield / acc':^25}"
    print(header2)
    print("─" * 120)

# Process each dataset
for dataset_idx, dataset_config in enumerate(dataset_configs):
    dataset_type = dataset_config['type']
    dataset_display = dataset_config['display']
    dataset_n = dataset_config['n']
    
    # Print dataset section header
    if dataset_n:
        print(f"\n{dataset_display} (n={dataset_n:,})")
    else:
        print(f"\n{dataset_display}")
    
    print_table_header()
    
    # Process individual LLMs or merged data
    if dataset_type != 'merged':
        # Print individual LLM rows
        for llm_display in llm_order:
            row = f"{llm_display:<20}"
            llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
            
            for method in methods:
                res = llm_data.get(method, {})
                if res.get('meets_criterion'):
                    tau_val = f"{res['tau']:.2f}"
                    yield_val = f"{res['pct_automated']:.1f}%"
                    acc_val = f"{res['accuracy_at_tau']:.3f}"
                    cell_content = f"{tau_val} / {yield_val} / {acc_val}"
                else:
                    cell_content = "FAILS"
                
                row += f"{cell_content:^25}"
            
            print(row)
        
        # Print mean (SD) row - showing yield mean (std)
        row_mean = f"{'Mean (SD)':<20}"
        
        for method in methods:
            yields = []
            
            for llm_display in llm_order:
                llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
                res = llm_data.get(method, {})
                if res.get('meets_criterion'):
                    yields.append(res['pct_automated'])
            
            if yields:
                yield_mean = np.mean(yields)
                yield_std = np.std(yields)
                cell_content = f"{yield_mean:.1f}% ({yield_std:.1f}%)"
            else:
                cell_content = "0.0% (0.0%)"
            
            row_mean += f"{cell_content:^25}"
        
        print(row_mean)
    
    else:
        # Merged: calculate across all datasets and LLMs - showing yield mean (std)
        row_merged = f"{'Mean (SD)':<20}"
        
        for method in methods:
            yields = []
            
            for dt in ['contextual', 'synthesis', 'clinical_inference']:
                for llm_display in llm_order:
                    llm_data = threshold_results.get(dt, {}).get(llm_display, {})
                    res = llm_data.get(method, {})
                    if res.get('meets_criterion'):
                        yields.append(res['pct_automated'])
            
            if yields:
                yield_mean = np.mean(yields)
                yield_std = np.std(yields)
                cell_content = f"{yield_mean:.1f}% ({yield_std:.1f}%)"
            else:
                cell_content = "0.0% (0.0%)"
            
            row_merged += f"{cell_content:^25}"
        
        print(row_merged)

print("\n" + "=" * 120)

# Save to CSV
# Create DataFrame for CSV export
csv_rows = []

for dataset_config in dataset_configs:
    dataset_type = dataset_config['type']
    dataset_display = dataset_config['display']
    dataset_n = dataset_config['n']
    
    # Add dataset header row
    if dataset_n:
        dataset_header = f"{dataset_display} (n={dataset_n:,})"
    else:
        dataset_header = dataset_display
    
    csv_rows.append({'LLM': dataset_header, 'P(True)': '', 'P(IK)': '', 'SAPLMA-F': '', 'CCPS (Ours)': ''})
    
    if dataset_type != 'merged':
        # Individual LLM rows
        for llm_display in llm_order:
            row_data = {'LLM': llm_display}
            llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
            
            for method in methods:
                res = llm_data.get(method, {})
                if res.get('meets_criterion'):
                    tau_val = f"{res['tau']:.2f}"
                    yield_val = f"{res['pct_automated']:.1f}%"
                    acc_val = f"{res['accuracy_at_tau']:.3f}"
                    row_data[method_labels[method]] = f"{tau_val} / {yield_val} / {acc_val}"
                else:
                    row_data[method_labels[method]] = "FAILS"
            
            csv_rows.append(row_data)
        
        # Mean (SD) row
        row_mean = {'LLM': 'Mean (SD)'}
        
        for method in methods:
            yields = []
            
            for llm_display in llm_order:
                llm_data = threshold_results.get(dataset_type, {}).get(llm_display, {})
                res = llm_data.get(method, {})
                if res.get('meets_criterion'):
                    yields.append(res['pct_automated'])
            
            if yields:
                yield_mean = np.mean(yields)
                yield_std = np.std(yields)
                row_mean[method_labels[method]] = f"{yield_mean:.1f}% ({yield_std:.1f}%)"
            else:
                row_mean[method_labels[method]] = "0.0% (0.0%)"
        
        csv_rows.append(row_mean)
    
    else:
        # Merged mean row
        row_merged = {'LLM': 'Mean (SD)'}
        
        for method in methods:
            yields = []
            
            for dt in ['contextual', 'synthesis', 'clinical_inference']:
                for llm_display in llm_order:
                    llm_data = threshold_results.get(dt, {}).get(llm_display, {})
                    res = llm_data.get(method, {})
                    if res.get('meets_criterion'):
                        yields.append(res['pct_automated'])
            
            if yields:
                yield_mean = np.mean(yields)
                yield_std = np.std(yields)
                row_merged[method_labels[method]] = f"{yield_mean:.1f}% ({yield_std:.1f}%)"
            else:
                row_merged[method_labels[method]] = "0.0% (0.0%)"
        
        csv_rows.append(row_merged)

# Create and save DataFrame
df_table = pd.DataFrame(csv_rows)
csv_path = OVERALL_RESULTS_PATH / "table2_safe_yield.csv"
df_table.to_csv(csv_path, index=False)
print(f"✅ Saved table to {csv_path}")

print("✅ Table 2 generated successfully")
print("=" * 120)


In [ ]:
# ==============================================================================
# TABLE 2 (FINAL): SAFE YIELD - Flat CSV Export
# Threshold = tau where accuracy >= 95% (5% error constraint)
# Only includes capable models (excludes Qwen-0.5B and Llama-1B which fail
# to meet the safety criterion on any method/dataset combination).
# Means computed over capable models only (n=3 per dataset).
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

methods = ['PTRUE', 'PIK', 'SAPLMA-F', 'CCPS']
method_labels = {
    'PTRUE':    'P(True)',
    'PIK':      'P(IK)',
    'SAPLMA-F': 'SAPLMA-F',
    'CCPS':     'CCPS (Ours)',
}

dataset_configs = [
    {'type': 'contextual',         'display': 'Contextual',         'n': 1458},
    {'type': 'synthesis',          'display': 'Synthesis',          'n': 764},
    {'type': 'clinical_inference', 'display': 'Clinical Inference', 'n': 612},
]

# Exclude models that fail to meet the safety criterion on any
# method/dataset combination -- they fall below the minimum competence
# floor required for safe automation regardless of confidence method.
CAPABLE_LLMS = ['Qwen-1.5B', 'Llama-3B', 'Qwen-3B']

def calculate_auroc(dataset_type, llm_display, method_name):
    try:
        if 'LLM_DISPLAY_NAMES' not in globals() or 'llm2method2dataset2results' not in globals():
            return None
        llm_id_map = {v: k for k, v in LLM_DISPLAY_NAMES.items()}
        llm_id = llm_id_map.get(llm_display, llm_display)
        results = llm2method2dataset2results.get(llm_id, {}).get(method_name, {}).get(dataset_type, [])
        if not results:
            return None
        if 'llm2dataset2sharedRIDacrossMethods' in globals():
            shared = llm2dataset2sharedRIDacrossMethods.get(llm_id, {}).get(dataset_type, set())
            if shared:
                results = [r for r in results if r.get('record_id') in shared]
        if not results:
            return None
        y_true = np.array([r.get('ground_truth_correctness', 0) for r in results])
        y_conf = np.array([r.get('confidence_score', 0.0) for r in results])
        if len(np.unique(y_true)) < 2:
            return 0.5
        return round(float(roc_auc_score(y_true, y_conf)), 3)
    except Exception:
        return None

# --- Per-LLM rows (capable models only) ---
rows = []
for dc in dataset_configs:
    dt, display, n = dc['type'], dc['display'], dc['n']
    for llm in CAPABLE_LLMS:
        llm_data = threshold_results.get(dt, {}).get(llm, {})
        for method in methods:
            res = llm_data.get(method, {})
            meets = res.get('meets_criterion', False)
            auroc = calculate_auroc(dt, llm, method)
            if auroc is not None:
                auroc = round(auroc, 3)
            rows.append({
                'Dataset':    display,
                'N':          n,
                'LLM':        llm,
                'Method':     method_labels[method],
                'Meets_5pct': meets,
                'Tau':        round(res['tau'], 3) if meets else None,
                'Yield_pct':  round(res['pct_automated'], 1) if meets else 0.0,
                'Accuracy':   round(res['accuracy_at_tau'], 3) if meets else None,
                'AUROC':      auroc,
            })

df_flat = pd.DataFrame(rows)

# --- Summary rows ---
summary_rows = []
for dc in dataset_configs:
    display, n = dc['display'], dc['n']
    for method in methods:
        label = method_labels[method]
        subset = df_flat[(df_flat['Dataset'] == display) & (df_flat['Method'] == label)]
        yields = subset['Yield_pct'].values
        aurocs = subset['AUROC'].dropna().values
        summary_rows.append({
            'Dataset':    display,
            'N':          n,
            'LLM':        f'Mean (SD), n={len(CAPABLE_LLMS)}',
            'Method':     label,
            'Meets_5pct': None,
            'Tau':        None,
            'Yield_pct':  f"{round(float(np.mean(yields)), 1)} ± {round(float(np.std(yields)), 1)}",
            'Accuracy':   None,
            'AUROC':      round(float(np.mean(aurocs)), 3) if len(aurocs) > 0 else None,
        })

df_summary = pd.DataFrame(summary_rows)

# Sort df_flat (per-model data)
dataset_order = {'Contextual': 0, 'Synthesis': 1, 'Clinical Inference': 2}
llm_order_map = {llm: i for i, llm in enumerate(CAPABLE_LLMS)}

df_flat['_d'] = df_flat['Dataset'].map(dataset_order)
df_flat['_l'] = df_flat['LLM'].map(llm_order_map)
df_flat = df_flat.sort_values(['_d', '_l', 'Method']).drop(columns=['_d', '_l']).reset_index(drop=True)
df_flat = df_flat.fillna('-')

# Sort df_summary (means only)
df_summary['_d'] = df_summary['Dataset'].map(dataset_order)
df_summary = df_summary.sort_values(['_d', 'Method']).drop(columns=['_d']).reset_index(drop=True)
df_summary = df_summary.fillna('-')

# Save per-model table
save_path_per_model = OVERALL_RESULTS_PATH / "table2_safe_yield_per_model.csv"
df_flat.to_csv(save_path_per_model, index=False)

# Save means table
save_path_means = OVERALL_RESULTS_PATH / "table2_safe_yield_means.csv"
df_summary.to_csv(save_path_means, index=False)

print("=" * 100)
print("TABLE 2A: SAFE YIELD - Per Model (capable models only: Qwen-1.5B, Llama-3B, Qwen-3B)")
print("Threshold = tau where accuracy >= 95% (5% error constraint)")
print(f"Saved to: {save_path_per_model}  |  Shape: {df_flat.shape}")
print("=" * 100)
print(df_flat.to_string(index=False))

print("\n" + "=" * 100)
print("TABLE 2B: SAFE YIELD - Means Only (capable models only: Qwen-1.5B, Llama-3B, Qwen-3B)")
print("Threshold = tau where accuracy >= 95% (5% error constraint)")
print(f"Saved to: {save_path_means}  |  Shape: {df_summary.shape}")
print("=" * 100)
print(df_summary.to_string(index=False))


In [ ]:
import pandas as pd
import numpy as np

print("=" * 120)
print("TABLE 1a: CALIBRATION METRICS - Mean Values")
print("=" * 120)

# Prepare data
df_reset = df_metrics.reset_index()

# Define method order
method_order = ['PTRUE', 'SAPLMA-M', 'SAPLMA-UM', 'SAPLMA-F', 'PIK', 'CCPS']

# Define dataset order
dataset_order = ['contextual', 'synthesis', 'clinical_inference', 'merged']

# Create multi-index columns
columns = pd.MultiIndex.from_product(
    [dataset_order, ['ECE', 'Brier', 'AUROC']],
    names=['Dataset', 'Metric']
)

# Initialize dataframe for MEAN VALUES
table_data_mean = []

for method in method_order:
    row_data = []
    
    for dataset in dataset_order:
        if dataset == 'merged':
            # Merged: average across all dataset types
            subset = df_reset[df_reset['method'] == method]
        else:
            # Specific dataset
            subset = df_reset[(df_reset['method'] == method) & (df_reset['dataset_type'] == dataset)]
        
        if len(subset) > 0:
            ece_mean = subset['ece'].mean()
            auroc_mean = subset['auroc'].mean()
            brier_mean = subset['brier'].mean()
            
            row_data.extend([
                f"{ece_mean:.3f}",
                f"{brier_mean:.3f}",
                f"{auroc_mean:.3f}"
            ])
        else:
            row_data.extend(['N/A', 'N/A', 'N/A'])
    
    table_data_mean.append(row_data)

# Create DataFrame for mean
df_table_mean = pd.DataFrame(table_data_mean, index=method_order, columns=columns)

# Display method names from method2path
df_table_mean.index = [method2path.get(m, {}).get('method_name', m) for m in method_order]

# Rename columns to display names
display_names = {
    'contextual': 'Contextual',
    'synthesis': 'Synthesis',
    'clinical_inference': 'Inference',
    'merged': 'All Datasets'
}

df_table_mean.columns = pd.MultiIndex.from_product(
    [[display_names[d] for d in dataset_order], ['ECE', 'Brier', 'AUROC']],
    names=['Dataset', 'Metric']
)

print(df_table_mean.to_string())

# Save to CSV
csv_path_mean = OVERALL_RESULTS_PATH / "table1a_calibration_metrics_mean.csv"
df_table_mean.to_csv(csv_path_mean)
print(f"\n✅ Saved to {csv_path_mean}")


print("\n\n")
print("=" * 120)
print("TABLE 1b: CALIBRATION METRICS - Mean ± Standard Deviation")
print("=" * 120)

# Initialize dataframe for MEAN ± STD
table_data_std = []

for method in method_order:
    row_data = []
    
    for dataset in dataset_order:
        if dataset == 'merged':
            # Merged: average across all dataset types
            subset = df_reset[df_reset['method'] == method]
        else:
            # Specific dataset
            subset = df_reset[(df_reset['method'] == method) & (df_reset['dataset_type'] == dataset)]
        
        if len(subset) > 0:
            ece_mean = subset['ece'].mean()
            auroc_mean = subset['auroc'].mean()
            brier_mean = subset['brier'].mean()
            ece_std = subset['ece'].std()
            auroc_std = subset['auroc'].std()
            brier_std = subset['brier'].std()
            
            row_data.extend([
                f"{ece_mean:.3f} ± {ece_std:.3f}",
                f"{brier_mean:.3f} ± {brier_std:.3f}",
                f"{auroc_mean:.3f} ± {auroc_std:.3f}"
            ])
        else:
            row_data.extend(['N/A', 'N/A', 'N/A'])
    
    table_data_std.append(row_data)

# Create DataFrame for std
columns_std = pd.MultiIndex.from_product(
    [dataset_order, ['ECE', 'Brier', 'AUROC']],
    names=['Dataset', 'Metric']
)

df_table_std = pd.DataFrame(table_data_std, index=method_order, columns=columns_std)

# Display method names from method2path
df_table_std.index = [method2path.get(m, {}).get('method_name', m) for m in method_order]

# Rename columns to display names
df_table_std.columns = pd.MultiIndex.from_product(
    [[display_names[d] for d in dataset_order], ['ECE', 'Brier', 'AUROC']],
    names=['Dataset', 'Metric']
)

# Transpose the table (flip rows and columns)
df_table_std_transposed = df_table_std.T

print(df_table_std_transposed.to_string())

# Save to CSV
csv_path_std = OVERALL_RESULTS_PATH / "table1b_calibration_metrics_mean_std.csv"
df_table_std_transposed.to_csv(csv_path_std)
print(f"\n✅ Saved to {csv_path_std}")

print("\n✅ Tables 1a and 1b generated successfully")
